<a href="https://colab.research.google.com/github/lipikamallikarjuna/MissingRfidDetection-ANSC6060/blob/main/MissingRFIDcode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install ydata-profiling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.5/682.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 3.7 MB/s eta 0:00:00


In [3]:
!pip install dtaidistance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 16.4 MB/s eta 0:00:00


In [4]:
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from ydata_profiling import ProfileReport
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors
from dtaidistance import dtw
from scipy.optimize import curve_fit

/tmp/ipykernel_541/1101268528.py:5: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [5]:
import numpy as np
import pandas as pd
import numpy as np
import time
import csv
import subprocess

#Data-Loading

In [ ]:
ProjectRoot      = Path("/content/drive/MyDrive/MiniProject")
DataFilePath    = ProjectRoot / "Data_set_prep_assignment_1 .csv"  # ← Removed leading slash, kept space
OutputFilePath  = ProjectRoot / "MiniProjectRuleBook&MetaData.xlsx"
CleanedFilePath = ProjectRoot / "AnimalIDRecovered.csv"          # ← all writes go here

df = pd.read_csv(DataFilePath, low_memory=False, dtype={"AnimalId": "string"})
df = df.rename(columns={
    "Avgmilkflow": "AverageMilkFlowKgPerMin",
    "Flow30_60Session": "MilkFlow30To60SecondsKgPerMin",
    "YieldFirst2Min_Session": "MilkYieldFirst2MinutesKg",
    "YieldSession": "TotalMilkYieldSessionKg",
    "DurationSession_sec": "MilkingDurationSeconds",
    "milking": "MilkingSession"
})
df["EventDate"] = pd.to_datetime(df["EventDate"], errors="coerce")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
display(df.head())

Rows: 8,495,421
Columns: 11


,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,AverageMilkFlowKgPerMin,MilkFlow30To60SecondsKgPerMin,MilkYieldFirst2MinutesKg,TotalMilkYieldSessionKg,MilkingDurationSeconds,MilkingSession
0,-8839528597343470980,1.0,365.0,Pregnant,2019-12-09,2.902991,0.898113,4.975908,11.158372,226,1
1,-5365332022005618386,1.0,66.0,Bred,2021-07-12,3.311224,1.401600,5.347854,15.059267,268,2
2,<NA>,NaN,NaN,NaN,2020-08-24,3.220506,3.501733,7.547777,14.560315,270,1
3,7750423760892252046,1.0,244.0,Pregnant,2019-12-04,3.900894,4.100475,8.400531,15.331422,231,3
4,<NA>,NaN,NaN,NaN,2021-03-21,4.218409,5.098378,9.198853,13.970645,198,2


#Data Understanding

##Profile Report

In [ ]:

# Generate the profile report
#print("Generating profile report")
#profile = ProfileReport(df, title="Dairy Farm Data Profile Report")

# Define the output path for the HTML report
#output_report_path = project_root / "DairyFarmData_ProfileReport.html"

# Save the report as an HTML file
#profile.to_file(output_report_path)

#print(f"Profile report saved to: {output_report_path}")

#Data-Preprocessing

##Data Quality Audit

In [ ]:
# Data Quality Audit
NumericColumns = [
    "LactationNumber", "DaysInMilk",
    "AverageMilkFlowKgPerMin", "MilkFlow30To60SecondsKgPerMin",
    "MilkYieldFirst2MinutesKg", "TotalMilkYieldSessionKg",
    "MilkingDurationSeconds"
]

DataQuality = pd.DataFrame({
    "DataType":          df.dtypes.astype(str),
    "MissingCount":      df.isna().sum(),
    "MissingPercent":    (df.isna().mean() * 100).round(2),
    "UniqueValues":      df.nunique(dropna=True),
    "NegativeCount":     df[NumericColumns].lt(0).sum().reindex(df.columns, fill_value=0),
    "ZeroCount":         df[NumericColumns].eq(0).sum().reindex(df.columns, fill_value=0),
})
DataQuality.loc["EventDate", "InvalidDateCount"]       = df["EventDate"].isna().sum()
DataQuality.loc["MilkingSession", "OutsideRangeCount"] = (~df["MilkingSession"].isin([1,2,3])).sum()

print("Data Quality Audit")
display(DataQuality.sort_values("MissingPercent", ascending=False))

print("\nDataset Summary")
display(pd.DataFrame({
    "Metric": ["Total records", "Total columns", "Date range start",
               "Date range end", "Unique AnimalIds"],
    "Value":  [len(df), df.shape[1], df["EventDate"].min().date(),
               df["EventDate"].max().date(), df["AnimalId"].nunique(dropna=True)]
}))

# Animal Record & Date Profile
AnimalRecords = (
    df[df["AnimalId"].notna()]
    .groupby("AnimalId")
    .agg(
        FirstDate   = ("EventDate", "min"),
        LastDate    = ("EventDate", "max"),
        UniqueDates = ("EventDate", "nunique"),
        TotalRecords= ("EventDate", "size")
    )
)

print("Per-Animal Record Summary")
display(AnimalRecords.describe())

Thresholds = pd.DataFrame({
    "Threshold":   ["< 10 records", "10–100 records", "> 100 records", "> 500 records"],
    "AnimalCount": [
        (AnimalRecords["TotalRecords"] < 10).sum(),
        ((AnimalRecords["TotalRecords"] >= 10) & (AnimalRecords["TotalRecords"] <= 100)).sum(),
        (AnimalRecords["TotalRecords"] > 100).sum(),
        (AnimalRecords["TotalRecords"] > 500).sum(),
    ]
})
print("\nRecord-Count Thresholds")
display(Thresholds)

print("\nSession Distribution")
display(df.groupby("MilkingSession").size().rename("RecordCount").to_frame())

Data Quality Audit


,DataType,MissingCount,MissingPercent,UniqueValues,NegativeCount,ZeroCount,InvalidDateCount,OutsideRangeCount
AnimalId,string,1701003,20.02,9087,0,0,NaN,NaN
LactationNumber,float64,1701003,20.02,12,0,0,NaN,NaN
DaysInMilk,float64,1701007,20.02,870,0,0,NaN,NaN
ReproductionStatus,object,1701003,20.02,4,0,0,NaN,NaN
EventDate,datetime64[ns],0,0.00,830,0,0,0.0,NaN
AverageMilkFlowKgPerMin,float64,172,0.00,219,0,215,NaN,NaN
MilkFlow30To60SecondsKgPerMin,float64,0,0.00,102,0,3155,NaN,NaN
MilkYieldFirst2MinutesKg,float64,0,0.00,1675,0,0,NaN,NaN
TotalMilkYieldSessionKg,float64,0,0.00,1010,0,0,NaN,NaN
MilkingDurationSeconds,int64,0,0.00,604,0,0,NaN,NaN



Dataset Summary


,Metric,Value
0,Total records,8495421
1,Total columns,11
2,Date range start,2019-06-14
3,Date range end,2021-10-24
4,Unique AnimalIds,9087


Per-Animal Record Summary


,FirstDate,LastDate,UniqueDates,TotalRecords
count,9087,9087,9087.000000,9087.000000
mean,2020-01-14 06:52:58.078574080,2021-04-02 00:12:02.614724352,296.540993,747.707494
min,2019-06-14 00:00:00,2019-06-15 00:00:00,1.000000,1.000000
25%,2019-06-15 00:00:00,2020-09-11 00:00:00,123.000000,313.000000
50%,2019-07-25 00:00:00,2021-09-27 00:00:00,279.000000,704.000000
75%,2020-07-30 00:00:00,2021-10-24 00:00:00,471.000000,1186.000000
max,2021-10-24 00:00:00,2021-10-24 00:00:00,701.000000,1763.000000
std,NaN,NaN,195.889120,492.065506



Record-Count Thresholds


,Threshold,AnimalCount
0,< 10 records,123
1,10–100 records,918
2,> 100 records,8046
3,> 500 records,5775



Session Distribution


,RecordCount
MilkingSession,
1,2831807
2,2831807
3,2831807


##Missingness Pattern

In [ ]:
# Missingness Pattern
IdColumns = ["AnimalId", "LactationNumber", "DaysInMilk", "ReproductionStatus"]

MissingPattern = (
    df[IdColumns]
    .isna().astype(int)
    .value_counts()
    .reset_index(name="RecordCount")
)
print("Missingness Combinations (0=present, 1=missing)")
display(MissingPattern)

Partial = df[IdColumns].isna().sum(axis=1)
PartialCounts = Partial.value_counts().sort_index()
print("\nIdentifier Columns Missing Per Row")
display(PartialCounts.rename("RecordCount").to_frame())

NPatterns = MissingPattern.shape[0]
if NPatterns <= 2:
    print("\n✓ Clean block-missingness confirmed: all 4 identifiers missing together.")
else:
    print(f"\n⚠ {NPatterns} missingness patterns detected — partial missingness exists.")

# Known vs Missing AnimalId Comparison
DfMaster = df.copy()
DfMaster["AnimalIdMissing"] = DfMaster["AnimalId"].isna()

ComparisonColumns = [
    "AverageMilkFlowKgPerMin", "MilkFlow30To60SecondsKgPerMin",
    "MilkYieldFirst2MinutesKg", "TotalMilkYieldSessionKg",
    "MilkingDurationSeconds", "MilkingSession"
]

KnownStats   = DfMaster.loc[~DfMaster["AnimalIdMissing"], ComparisonColumns].describe().T
MissingStats = DfMaster.loc[ DfMaster["AnimalIdMissing"], ComparisonColumns].describe().T

Comparison = KnownStats[["count","mean","std","min","50%","max"]].join(
    MissingStats[["count","mean","std","min","50%","max"]],
    lsuffix="_KnownId", rsuffix="_MissingId"
)
print("Feature Distribution: Known vs Missing AnimalId ")
display(Comparison)

Missingness Combinations (0=present, 1=missing)


,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,RecordCount
0,0,0,0,0,6794414
1,1,1,1,1,1701003
2,0,0,1,0,4



Identifier Columns Missing Per Row


,RecordCount
0,6794414
1,4
4,1701003



⚠ 3 missingness patterns detected — partial missingness exists.
Feature Distribution: Known vs Missing AnimalId 


,count_KnownId,mean_KnownId,std_KnownId,min_KnownId,50%_KnownId,max_KnownId,count_MissingId,mean_MissingId,std_MissingId,min_MissingId,50%_MissingId,max_MissingId
AverageMilkFlowKgPerMin,6794287.0,3.525899,0.729146,0.000000,3.492661,23.496085,1700962.0,3.526221,0.729128,0.000000,3.492661,23.496085
MilkFlow30To60SecondsKgPerMin,6794418.0,3.751396,1.485795,0.000000,3.900894,10.500663,1701003.0,3.752198,1.485216,0.000000,3.900894,10.101502
MilkYieldFirst2MinutesKg,6794418.0,7.490827,2.014306,2.000342,7.525097,11.997518,1701003.0,7.492219,2.013337,2.000342,7.525097,11.997518
TotalMilkYieldSessionKg,6794418.0,13.928592,3.645881,5.034875,13.743849,54.839318,1701003.0,13.927960,3.646393,5.034875,13.743849,53.977492
MilkingDurationSeconds,6794418.0,236.870908,54.844360,100.000000,231.000000,785.000000,1701003.0,236.835210,54.838978,100.000000,231.000000,773.000000
MilkingSession,6794418.0,2.000101,0.816531,1.000000,2.000000,3.000000,1701003.0,1.999598,0.816361,1.000000,2.000000,3.000000


##Outlier Flagging

In [ ]:
# Outlier Flagging
NumericMilking = [
    "AverageMilkFlowKgPerMin", "MilkFlow30To60SecondsKgPerMin",
    "MilkYieldFirst2MinutesKg", "TotalMilkYieldSessionKg",
    "MilkingDurationSeconds"
]

FlagRows = {}
for Col in NumericMilking:
    Q1, Q3 = DfMaster[Col].quantile(0.25), DfMaster[Col].quantile(0.75)
    IQR    = Q3 - Q1
    Low, High = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    FlagRows[f"{Col}IQRLow"]  = DfMaster[Col] < Low
    FlagRows[f"{Col}IQRHigh"] = DfMaster[Col] > High
    print(f"{Col}: IQR fence [{Low:.3f}, {High:.3f}]")

FlagRows["DaysInMilkOver350"]      = DfMaster["DaysInMilk"] > 350
FlagRows["DurationUnder150s"]      = DfMaster["MilkingDurationSeconds"] < 150
FlagRows["DurationOver600s"]       = DfMaster["MilkingDurationSeconds"] > 600
FlagRows["AvgFlowIsZero"]          = DfMaster["AverageMilkFlowKgPerMin"] == 0
FlagRows["Flow30To60IsZero"]       = DfMaster["MilkFlow30To60SecondsKgPerMin"] == 0
FlagRows["LactationNumberInvalid"] = (
    DfMaster["LactationNumber"].notna() &
    ~DfMaster["LactationNumber"].isin(range(1, 13))
)

OutlierFlags = pd.DataFrame(FlagRows, index=DfMaster.index)

print("\nOutlier / Flag Counts")
display(OutlierFlags.sum().rename("FlaggedRows").to_frame())

AverageMilkFlowKgPerMin: IQR fence [1.497, 5.488]
MilkFlow30To60SecondsKgPerMin: IQR fence [-0.352, 8.049]
MilkYieldFirst2MinutesKg: IQR fence [1.560, 13.463]
TotalMilkYieldSessionKg: IQR fence [4.082, 23.678]
MilkingDurationSeconds: IQR fence [84.500, 384.500]

Outlier / Flag Counts


,FlaggedRows
AverageMilkFlowKgPerMinIQRLow,3474
AverageMilkFlowKgPerMinIQRHigh,3411
MilkFlow30To60SecondsKgPerMinIQRLow,0
MilkFlow30To60SecondsKgPerMinIQRHigh,1560
MilkYieldFirst2MinutesKgIQRLow,0
MilkYieldFirst2MinutesKgIQRHigh,0
TotalMilkYieldSessionKgIQRLow,0
TotalMilkYieldSessionKgIQRHigh,43802
MilkingDurationSecondsIQRLow,0
MilkingDurationSecondsIQRHigh,50700


## Meta-Data Rulebook creation

In [ ]:
# Rulebook & Metadata
Definitions = {
    "AnimalId":                      "Animal identifier",
    "LactationNumber":               "Lactation number",
    "DaysInMilk":                    "Days since calving",
    "ReproductionStatus":            "Reproductive-status category",
    "EventDate":                     "Date of the milking event",
    "AverageMilkFlowKgPerMin":       "Average milk flow rate",
    "MilkFlow30To60SecondsKgPerMin": "Flow rate from 30-60 seconds into milking",
    "MilkYieldFirst2MinutesKg":      "Milk produced during the first 2 minutes",
    "TotalMilkYieldSessionKg":       "Total milk produced during the session",
    "MilkingDurationSeconds":        "Session duration in seconds",
    "MilkingSession":                "Milking/session indicator"
}
CanonicalUnits = {
    "AnimalId": "Identifier", "LactationNumber": "Count", "DaysInMilk": "Days",
    "ReproductionStatus": "Category", "EventDate": "Date",
    "AverageMilkFlowKgPerMin": "kg/min", "MilkFlow30To60SecondsKgPerMin": "kg/min",
    "MilkYieldFirst2MinutesKg": "kg", "TotalMilkYieldSessionKg": "kg",
    "MilkingDurationSeconds": "seconds", "MilkingSession": "Category"
}
RulebookDomain = {
    "AnimalId":                      (None, None, "none",   False, "BlockWithAnimalId", True,  "Hashed int IDs; negatives valid; TARGET column"),
    "LactationNumber":               (1,    12,   "domain", False, "BlockWithAnimalId", True,  ">12 biologically impossible"),
    "DaysInMilk":                    (1,    870,  "IQR",    False, "BlockWithAnimalId", True,  ">350 soft flag — past typical dry-off"),
    "ReproductionStatus":            (None, None, "none",   False, "BlockWithAnimalId", True,  "4 valid categories: Pregnant, Bred, Fresh, Open"),
    "EventDate":                     (None, None, "domain", False, "None",              True,  "No missing; 2019-06-14 to 2021-10-24"),
    "AverageMilkFlowKgPerMin":       (0,    23.5, "IQR",    False, "Independent (172)", True,  "Zero = likely sensor error"),
    "MilkFlow30To60SecondsKgPerMin": (0,    10.5, "IQR",    True,  "None",              True,  "3155 zeros — slow let-down or sensor"),
    "MilkYieldFirst2MinutesKg":      (2.0,  12.0, "IQR",    False, "None",              True,  "Min 2.0003 suggests sensor floor"),
    "TotalMilkYieldSessionKg":       (5.0,  54.8, "IQR",    False, "None",              True,  "High corr with Duration; max 54.8 extreme"),
    "MilkingDurationSeconds":        (100,  785,  "IQR",    False, "None",              True,  "<150s and >600s are soft flags"),
    "MilkingSession":                (1,    3,    "domain", False, "None",              True,  "Uniform 1/2/3; only 3 valid values"),
}

MetadataRows = []
for Col in df.columns:
    Series = df[Col]
    if pd.api.types.is_numeric_dtype(Series):
        Min, Max = Series.min(), Series.max()
        Range = Max - Min
    elif pd.api.types.is_datetime64_any_dtype(Series):
        Min = Series.min().date().isoformat()
        Max = Series.max().date().isoformat()
        Range = (Series.max() - Series.min()).days
    else:
        Min = Max = Range = "N/A"

    VMin, VMax, Method, ZeroOk, MissPat, UseModel, Notes = RulebookDomain[Col]
    MetadataRows.append({
        "Column":           Col,
        "Definition":       Definitions[Col],
        "Unit":             CanonicalUnits[Col],
        "DataType":         str(Series.dtype),
        "TotalCount":       len(Series),
        "NonMissingCount":  Series.notna().sum(),
        "MissingCount":     Series.isna().sum(),
        "MissingPercent":   round(Series.isna().mean() * 100, 2),
        "UniqueValues":     Series.nunique(dropna=True),
        "Minimum":          Min,
        "Maximum":          Max,
        "Range":            Range,
        "ValidMin":         VMin,
        "ValidMax":         VMax,
        "OutlierMethod":    Method,
        "ZeroAllowed":      ZeroOk,
        "MissingPattern":   MissPat,
        "UseInModel":       UseModel,
        "Notes":            Notes,
        "ExampleValues":    ", ".join(Series.dropna().astype(str).unique()[:5])
    })

DfRulebook = pd.DataFrame(MetadataRows)
DfRulebook.to_excel(OutputFilePath, sheet_name="Rulebook&MetaData", index=False)
print(f"Saved: {OutputFilePath}")
display(DfRulebook[["Column","ValidMin","ValidMax","OutlierMethod",
                     "ZeroAllowed","MissingPattern","UseInModel","Notes"]])

Saved: /content/drive/MyDrive/MiniProject/MiniProjectRuleBook&MetaData.xlsx


,Column,ValidMin,ValidMax,OutlierMethod,ZeroAllowed,MissingPattern,UseInModel,Notes
0,AnimalId,NaN,NaN,none,False,BlockWithAnimalId,True,Hashed int IDs; negatives valid; TARGET column
1,LactationNumber,1.0,12.0,domain,False,BlockWithAnimalId,True,>12 biologically impossible
2,DaysInMilk,1.0,870.0,IQR,False,BlockWithAnimalId,True,>350 soft flag — past typical dry-off
3,ReproductionStatus,NaN,NaN,none,False,BlockWithAnimalId,True,"4 valid categories: Pregnant, Bred, Fresh, Open"
4,EventDate,NaN,NaN,domain,False,None,True,No missing; 2019-06-14 to 2021-10-24
5,AverageMilkFlowKgPerMin,0.0,23.5,IQR,False,Independent (172),True,Zero = likely sensor error
6,MilkFlow30To60SecondsKgPerMin,0.0,10.5,IQR,True,None,True,3155 zeros — slow let-down or sensor
7,MilkYieldFirst2MinutesKg,2.0,12.0,IQR,False,None,True,Min 2.0003 suggests sensor floor
8,TotalMilkYieldSessionKg,5.0,54.8,IQR,False,None,True,High corr with Duration; max 54.8 extreme
9,MilkingDurationSeconds,100.0,785.0,IQR,False,None,True,<150s and >600s are soft flags


#Data Prep

##Data Cleaning

In [ ]:
# Data Cleaning
DfMaster = df.copy()
DfMaster["AnimalIdMissing"] = DfMaster["AnimalId"].isna()

# Outlier flags
NumericMilking = [
    "AverageMilkFlowKgPerMin", "MilkFlow30To60SecondsKgPerMin",
    "MilkYieldFirst2MinutesKg", "TotalMilkYieldSessionKg",
    "MilkingDurationSeconds"
]
FlagRows = {}
for Col in NumericMilking:
    Q1, Q3 = DfMaster[Col].quantile(0.25), DfMaster[Col].quantile(0.75)
    IQR = Q3 - Q1
    FlagRows[f"{Col}IQRLow"]  = DfMaster[Col] < Q1 - 1.5 * IQR
    FlagRows[f"{Col}IQRHigh"] = DfMaster[Col] > Q3 + 1.5 * IQR

FlagRows["DaysInMilkOver350"]       = DfMaster["DaysInMilk"] > 350
FlagRows["DurationUnder150s"]       = DfMaster["MilkingDurationSeconds"] < 150
FlagRows["DurationOver600s"]        = DfMaster["MilkingDurationSeconds"] > 600
FlagRows["AvgFlowIsZero"]           = DfMaster["AverageMilkFlowKgPerMin"] == 0
FlagRows["Flow30To60IsZero"]        = DfMaster["MilkFlow30To60SecondsKgPerMin"] == 0
FlagRows["LactationNumberInvalid"]  = (
    DfMaster["LactationNumber"].notna() &
    ~DfMaster["LactationNumber"].isin(range(1, 13))
)
OutlierFlags = pd.DataFrame(FlagRows, index=DfMaster.index)

# Protect missing rows — never drop
DfMissingProtected = DfMaster[DfMaster["AnimalId"].isna()].copy()
DfKnownClean       = DfMaster[DfMaster["AnimalId"].notna()].copy()

# Drop invalid known rows only
KnownBefore = len(DfKnownClean)
DfKnownClean = DfKnownClean[
    ~OutlierFlags.loc[DfKnownClean.index, "AvgFlowIsZero"] &
    ~OutlierFlags.loc[DfKnownClean.index, "DurationOver600s"] &
    DfKnownClean["AverageMilkFlowKgPerMin"].notna()
]

# Deduplicate known rows
DedupCols = ["AnimalId","EventDate","MilkingSession",
             "TotalMilkYieldSessionKg","MilkingDurationSeconds","AverageMilkFlowKgPerMin"]
NDupes    = DfKnownClean.duplicated(subset=DedupCols, keep=False).sum()
DfKnownClean = DfKnownClean.drop_duplicates(subset=DedupCols, keep="first")

print(f"Known rows dropped (invalid) : {KnownBefore - len(DfKnownClean) - NDupes//2:,}")
print(f"Known rows dropped (dupes)   : {NDupes//2:,}")
print(f"Known rows after cleaning    : {len(DfKnownClean):,}")
print(f"Missing rows protected       : {len(DfMissingProtected):,}")

Known rows dropped (invalid) : 5,264
Known rows dropped (dupes)   : 51,790
Known rows after cleaning    : 6,737,364
Missing rows protected       : 1,701,003


#Modelling

##Master File Creation

In [ ]:
# build Master + Save/overwrite
DfMaster  = pd.concat([DfKnownClean, DfMissingProtected], ignore_index=True)
DfKnown   = DfKnownClean.copy()
DfMissing = DfMissingProtected.copy()

print(f"DfKnown   : {len(DfKnown):,} rows")
print(f"DfMissing : {len(DfMissing):,} rows")
print(f"DfMaster  : {len(DfMaster):,} rows total")

DfMaster.to_csv(CleanedFilePath, index=False)
print(f"Saved: {CleanedFilePath}")

DfKnown   : 6,737,364 rows
DfMissing : 1,701,003 rows
DfMaster  : 8,438,367 rows total
Saved: /content/drive/MyDrive/MiniProject/AnimalIDRecovered.csv


##Feature Set Creation

In [ ]:
# Feature Set
ClusterFeatures = [
    "TotalMilkYieldSessionKg",
    "MilkingDurationSeconds",
    "AverageMilkFlowKgPerMin",
    "MilkFlow30To60SecondsKgPerMin",
    "MilkYieldFirst2MinutesKg"
]

DfKnown["EventDateInt"]   = DfKnown["EventDate"].astype("int64") // 10**9 // 86400
DfMissing["EventDateInt"] = DfMissing["EventDate"].astype("int64") // 10**9 // 86400

print(f"Cluster features : {ClusterFeatures}")
print(f"Known rows ready : {DfKnown[ClusterFeatures].notna().all(axis=1).sum():,}")

Cluster features : ['TotalMilkYieldSessionKg', 'MilkingDurationSeconds', 'AverageMilkFlowKgPerMin', 'MilkFlow30To60SecondsKgPerMin', 'MilkYieldFirst2MinutesKg']
Known rows ready : 6,737,364


##Dataset Summary

In [ ]:
print("=== DATASET SUMMARY ===")
print(f"DfKnown   : {len(DfKnown):,} rows | Cleaned records with active AnimalId")
print(f"DfMissing : {len(DfMissing):,} rows | Records missing AnimalId (protected)")
print(f"DfMaster  : {len(DfMaster):,} rows total\n")

print("=== COLUMNS IN THE DATASETS ===")
print(list(DfMaster.columns))

print("\n=== SAMPLE FROM KNOWN DATA (DfKnown) ===")
display(DfKnown.head(3))

print("\n=== SAMPLE FROM MISSING/UNKNOWN DATA (DfMissing) ===")
display(DfMissing.head(3))

=== DATASET SUMMARY ===
DfKnown   : 6,737,364 rows | Cleaned records with active AnimalId
DfMissing : 1,701,003 rows | Records missing AnimalId (protected)
DfMaster  : 8,438,367 rows total

=== COLUMNS IN THE DATASETS ===
['AnimalId', 'LactationNumber', 'DaysInMilk', 'ReproductionStatus', 'EventDate', 'AverageMilkFlowKgPerMin', 'MilkFlow30To60SecondsKgPerMin', 'MilkYieldFirst2MinutesKg', 'TotalMilkYieldSessionKg', 'MilkingDurationSeconds', 'MilkingSession', 'AnimalIdMissing']

=== SAMPLE FROM KNOWN DATA (DfKnown) ===


,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,AverageMilkFlowKgPerMin,MilkFlow30To60SecondsKgPerMin,MilkYieldFirst2MinutesKg,TotalMilkYieldSessionKg,MilkingDurationSeconds,MilkingSession,AnimalIdMissing,EventDateInt
0,-8839528597343470980,1.0,365.0,Pregnant,2019-12-09,2.902991,0.898113,4.975908,11.158372,226,1,False,18239
1,-5365332022005618386,1.0,66.0,Bred,2021-07-12,3.311224,1.401600,5.347854,15.059267,268,2,False,18820
3,7750423760892252046,1.0,244.0,Pregnant,2019-12-04,3.900894,4.100475,8.400531,15.331422,231,3,False,18234



=== SAMPLE FROM MISSING/UNKNOWN DATA (DfMissing) ===


,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,AverageMilkFlowKgPerMin,MilkFlow30To60SecondsKgPerMin,MilkYieldFirst2MinutesKg,TotalMilkYieldSessionKg,MilkingDurationSeconds,MilkingSession,AnimalIdMissing,EventDateInt
2,<NA>,NaN,NaN,NaN,2020-08-24,3.220506,3.501733,7.547777,14.560315,270,1,True,18498
4,<NA>,NaN,NaN,NaN,2021-03-21,4.218409,5.098378,9.198853,13.970645,198,2,True,18707
5,<NA>,NaN,NaN,NaN,2020-04-16,2.313321,2.499294,4.649322,9.661517,245,2,True,18368


##FIngerprint building

In [ ]:
# Build animal absence fingerprints

# Step 1: For each known animal, find their gap periods
DfDates = (DfKnown[["AnimalId","EventDate","LactationNumber","DaysInMilk",
                     "ReproductionStatus","TotalMilkYieldSessionKg",
                     "AverageMilkFlowKgPerMin","MilkingDurationSeconds"]]
           .drop_duplicates(subset=["AnimalId","EventDate"])
           .sort_values(["AnimalId","EventDate"]))

DfDates["NextDate"] = DfDates.groupby("AnimalId")["EventDate"].shift(-1)
DfDates["GapToNext"] = (DfDates["NextDate"] - DfDates["EventDate"]).dt.days

# A gap exists when next appearance is more than 1 day later
Gaps = DfDates[DfDates["GapToNext"] > 1].copy()
Gaps["GapStart"] = Gaps["EventDate"] + pd.Timedelta(days=1)
Gaps["GapEnd"]   = Gaps["NextDate"]  - pd.Timedelta(days=1)
Gaps["GapDays"]  = (Gaps["GapEnd"] - Gaps["GapStart"]).dt.days + 1

# Rename pre-gap values
Gaps = Gaps.rename(columns={
    "EventDate":               "PreDate",
    "LactationNumber":         "PreLac",
    "DaysInMilk":              "PreDIM",
    "ReproductionStatus":      "PreRepro",
    "TotalMilkYieldSessionKg": "PreYield",
    "AverageMilkFlowKgPerMin": "PreFlow",
    "MilkingDurationSeconds":  "PreDuration"
})

# Get post-gap values by joining next known record
PostGap = DfDates[["AnimalId","EventDate","LactationNumber","DaysInMilk",
                    "ReproductionStatus","TotalMilkYieldSessionKg"]].rename(columns={
    "EventDate":               "PostDate",
    "LactationNumber":         "PostLac",
    "DaysInMilk":              "PostDIM",
    "ReproductionStatus":      "PostRepro",
    "TotalMilkYieldSessionKg": "PostYield"
})

DfGaps = Gaps[["AnimalId","PreDate","GapStart","GapEnd","GapDays",
               "PreLac","PreDIM","PreRepro","PreYield","PreFlow","PreDuration"]].copy()
DfGaps["PostDate"] = DfGaps["GapEnd"] + pd.Timedelta(days=1)
DfGaps = DfGaps.merge(PostGap, on=["AnimalId","PostDate"], how="left")

print(f"Total gap periods found    : {len(DfGaps):,}")
print(f"Animals with any gap       : {DfGaps['AnimalId'].nunique():,}")
print(f"Mean gap duration          : {DfGaps['GapDays'].mean():.0f} days")
print(f"Gaps > 30 days             : {(DfGaps['GapDays'] > 30).sum():,}")
print(f"Gaps > 60 days             : {(DfGaps['GapDays'] > 60).sum():,}")
print()
print("Sample fingerprints:")
display(DfGaps[DfGaps["GapDays"] > 30][
    ["AnimalId","GapStart","GapEnd","GapDays",
     "PreLac","PreDIM","PreRepro","PreYield",
     "PostLac","PostDIM","PostRepro","PostYield"]
].head(5))

Total gap periods found    : 420,670
Animals with any gap       : 8,964
Mean gap duration          : 3 days
Gaps > 30 days             : 8,930
Gaps > 60 days             : 4,916

Sample fingerprints:


,AnimalId,GapStart,GapEnd,GapDays,PreLac,PreDIM,PreRepro,PreYield,PostLac,PostDIM,PostRepro,PostYield
26,-1007048835775197250,2020-01-22,2020-03-27,66,1.0,289.0,Pregnant,12.383072,2.0,2.0,Fresh,15.331422
58,-1007048835775197250,2021-03-12,2021-07-08,119,2.0,349.0,Pregnant,5.579186,3.0,1.0,Fresh,13.290256
85,-1007468268464577766,2019-07-26,2019-09-07,44,2.0,219.0,Pregnant,14.061363,2.0,264.0,Pregnant,9.570799
86,-1007468268464577766,2019-09-25,2019-11-21,58,2.0,280.0,Pregnant,9.026488,3.0,7.0,Fresh,6.985322
123,-1007468268464577766,2021-01-05,2021-07-04,181,3.0,416.0,Pregnant,5.715264,4.0,2.0,Fresh,9.344003


In [ ]:
# Save checkpoint
DfGaps.to_csv(ProjectRoot / "DfGaps.csv", index=False)
print("DfGaps saved")

DfGaps saved


In [ ]:
# Focus only on meaningful gaps — filter to >30 days
DfGaps = DfGaps[DfGaps["GapDays"] > 30].copy().reset_index(drop=True)

# Save
DfGaps.to_csv(ProjectRoot / "DfGaps.csv", index=False)

print(f"Meaningful gap periods : {len(DfGaps):,}")
print(f"Animals with gaps >30d : {DfGaps['AnimalId'].nunique():,}")
print(f"Mean gap duration      : {DfGaps['GapDays'].mean():.0f} days")
print(f"Gaps with calving      : {(DfGaps['PostLac'] > DfGaps['PreLac']).sum():,}")
print(f"Continuous lactation   : {(DfGaps['PostLac'] == DfGaps['PreLac']).sum():,}")

Meaningful gap periods : 8,930
Animals with gaps >30d : 5,617
Mean gap duration      : 67 days
Gaps with calving      : 7,961
Continuous lactation   : 969


In [ ]:
# Build complete gap fingerprint
# For each gap period, compute expected DIM on every gap date
# and the calving date if calving occurred

DfGaps["CalvingDate"] = pd.NaT
CalvMask = DfGaps["PostLac"] > DfGaps["PreLac"]
DfGaps.loc[CalvMask, "CalvingDate"] = (
    DfGaps.loc[CalvMask, "PostDate"] -
    pd.to_timedelta(DfGaps.loc[CalvMask, "PostDIM"].fillna(0).astype(int), unit="D")
)

# Verify calving dates fall within gap
InGap = (
    (DfGaps["CalvingDate"] >= DfGaps["GapStart"]) &
    (DfGaps["CalvingDate"] <= DfGaps["GapEnd"])
)
print(f"Gaps with calving              : {CalvMask.sum():,}")
print(f"Calving date within gap        : {(CalvMask & InGap).sum():,}")
print(f"Calving date outside gap       : {(CalvMask & ~InGap).sum():,}")
print(f"Continuous lactation gaps      : {(~CalvMask).sum():,}")
print()

# For each gap, compute expected DIM on every date
# This is the exact constraint — DIM must match or it's not this animal
def ExpectedDIM(Row, Date):
    if pd.isna(Row["CalvingDate"]) or Date < Row["CalvingDate"]:
        return int(Row["PreDIM"]) + (Date - Row["PreDate"]).days
    else:
        return (Date - Row["CalvingDate"]).days + 1

# Test on one animal
SampleRow = DfGaps[DfGaps["GapDays"] > 60].iloc[0]
print(f"Sample animal : {SampleRow['AnimalId']}")
print(f"Gap           : {SampleRow['GapStart'].date()} → {SampleRow['GapEnd'].date()}")
print(f"PreLac={SampleRow['PreLac']:.0f} PreDIM={SampleRow['PreDIM']:.0f} PreYield={SampleRow['PreYield']:.2f}")
print(f"PostLac={SampleRow['PostLac']:.0f} PostDIM={SampleRow['PostDIM']:.0f} PostYield={SampleRow['PostYield']:.2f}")
if pd.notna(SampleRow["CalvingDate"]):
    print(f"Calving date  : {SampleRow['CalvingDate'].date()}")
print()
print("Expected DIM on first 5 gap days:")
for i in range(5):
    D = SampleRow["GapStart"] + pd.Timedelta(days=i)
    print(f"  {D.date()} → DIM {ExpectedDIM(SampleRow, D)}")

DfGaps.to_csv(ProjectRoot / "DfGaps.csv", index=False)
print("\nSaved")

Gaps with calving              : 7,961
Calving date within gap        : 7,960
Calving date outside gap       : 1
Continuous lactation gaps      : 969

Sample animal : -1007048835775197250
Gap           : 2020-01-22 → 2020-03-27
PreLac=1 PreDIM=289 PreYield=12.38
PostLac=2 PostDIM=2 PostYield=15.33
Calving date  : 2020-03-26

Expected DIM on first 5 gap days:
  2020-01-22 → DIM 290
  2020-01-23 → DIM 291
  2020-01-24 → DIM 292
  2020-01-25 → DIM 293
  2020-01-26 → DIM 294

Saved


##Phase 1 Longitudinal Pattern Building

###Known Animal Pattern

In [ ]:
# Build complete biological timeline per animal
# Sort by AnimalId + Date — always
# Project exact Lac and DIM through every gap day
# Find mystery records whose date falls in the gap
# and whose projected Lac+DIM matches only one animal

# Step 1: Build projected timeline for all gap periods
print("Building projected timelines...")

TimelineRows = []

for _, Row in DfGaps[DfGaps["GapDays"] > 30].iterrows():
    GapDates = pd.date_range(Row["GapStart"], Row["GapEnd"], freq="D")

    for Date in GapDates:
        # Project exact Lac and DIM for this animal on this date
        if pd.isna(Row["CalvingDate"]) or Date < pd.Timestamp(Row["CalvingDate"]):
            ProjLac = Row["PreLac"]
            ProjDIM = int(Row["PreDIM"]) + (Date - Row["PreDate"]).days
        else:
            ProjLac = Row["PostLac"]
            ProjDIM = (Date - pd.Timestamp(Row["CalvingDate"])).days + 1

        TimelineRows.append({
            "AnimalId": Row["AnimalId"],
            "Date":     Date,
            "ProjLac":  ProjLac,
            "ProjDIM":  ProjDIM,
            "GapStart": Row["GapStart"],
            "GapEnd":   Row["GapEnd"]
        })

DfTimeline = pd.DataFrame(TimelineRows).sort_values(["Date","AnimalId"]).reset_index(drop=True)
print(f"Timeline rows built : {len(DfTimeline):,}")
print(f"Unique animals      : {DfTimeline['AnimalId'].nunique():,}")
print(f"Unique dates        : {DfTimeline['Date'].nunique():,}")

# Step 2: For each date, how many animals share the same ProjLac + ProjDIM?
# This is the key — if only 1 animal has ProjLac=2 DIM=45 on 2020-03-15
# then every mystery record on that date with that context belongs to that animal

print("\nChecking uniqueness of Lac+DIM per date...")
Counts = DfTimeline.groupby(["Date","ProjLac","ProjDIM"]).size().reset_index(name="NAnimals")
print(f"Unique Lac+DIM+Date combinations : {len(Counts):,}")
print(f"Combinations with exactly 1 animal : {(Counts['NAnimals']==1).sum():,}")
print(f"Combinations with 2 animals        : {(Counts['NAnimals']==2).sum():,}")
print(f"Combinations with 3+ animals       : {(Counts['NAnimals']>=3).sum():,}")
print()
display(Counts[Counts["NAnimals"]==1].head(5))

Building projected timelines...
Timeline rows built : 598,672
Unique animals      : 5,617
Unique dates        : 861

Checking uniqueness of Lac+DIM per date...
Unique Lac+DIM+Date combinations : 401,751
Combinations with exactly 1 animal : 286,112
Combinations with 2 animals        : 69,692
Combinations with 3+ animals       : 45,947



,Date,ProjLac,ProjDIM,NAnimals
0,2019-06-16,2.0,314,1
1,2019-06-17,2.0,315,1
2,2019-06-17,2.0,378,1
3,2019-06-17,3.0,268,1
4,2019-06-17,3.0,310,1


In [ ]:
DfTimeline.to_csv(ProjectRoot / "DfTimeline.csv", index=False)
print("Saved")

Saved


###Unknown Animal Pattern

In [ ]:
# Build mystery animal date patterns from DfMissing
# We don't know their ID but we know their dates
# Group consecutive missing dates into "mystery gap periods"
# Each mystery animal should appear on a consistent set of dates

# Step 1: Find consecutive date blocks in DfMissing
# Each block = one mystery animal's continuous missing period
MissingDates = (DfMissing.groupby("EventDate")
                .size().reset_index(name="NRecords")
                .sort_values("EventDate"))

print(f"Dates with missing records : {len(MissingDates):,}")
print(f"Min date : {MissingDates['EventDate'].min().date()}")
print(f"Max date : {MissingDates['EventDate'].max().date()}")
print(f"Mean records per date : {MissingDates['NRecords'].mean():.0f}")
print()

# Step 2: For each known animal gap period
# check if DfMissing has records on EVERY date of that gap
# The mystery animal should be present on all gap dates — not just some

print("Checking gap date coverage in DfMissing...")
MissingDateSet = set(DfMissing["EventDate"].dt.date)

CoverageResults = []
for _, Row in DfGaps[DfGaps["GapDays"] > 30].head(20).iterrows():
    GapDates    = pd.date_range(Row["GapStart"], Row["GapEnd"], freq="D")
    GapDateSet  = set(GapDates.date)
    CoveredDays = len(GapDateSet & MissingDateSet)
    Coverage    = CoveredDays / len(GapDateSet)

    CoverageResults.append({
        "AnimalId": Row["AnimalId"],
        "GapStart": Row["GapStart"].date(),
        "GapEnd":   Row["GapEnd"].date(),
        "GapDays":  Row["GapDays"],
        "CoveredDays": CoveredDays,
        "Coverage": Coverage
    })

DfCoverage = pd.DataFrame(CoverageResults)
print(f"Mean date coverage in DfMissing : {DfCoverage['Coverage'].mean():.1%}")
print()
display(DfCoverage[["AnimalId","GapStart","GapEnd","GapDays","CoveredDays","Coverage"]])

Dates with missing records : 830
Min date : 2019-06-14
Max date : 2021-10-24
Mean records per date : 2049

Checking gap date coverage in DfMissing...
Mean date coverage in DfMissing : 91.8%



,AnimalId,GapStart,GapEnd,GapDays,CoveredDays,Coverage
0,-1007048835775197250,2020-01-22,2020-03-27,66,66,1.000000
1,-1007048835775197250,2021-03-12,2021-07-08,119,119,1.000000
2,-1007468268464577766,2019-07-26,2019-09-07,44,26,0.590909
3,-1007468268464577766,2019-09-25,2019-11-21,58,57,0.982759
4,-1007468268464577766,2021-01-05,2021-07-04,181,181,1.000000
5,-1024379415194528358,2019-07-20,2019-12-27,161,142,0.881988
6,-1024379415194528358,2020-10-27,2020-12-14,49,49,1.000000
7,-1033053644401603962,2019-07-26,2019-10-11,78,60,0.769231
8,-1033053644401603962,2020-10-21,2021-02-26,129,129,1.000000
9,-1037603786677155953,2019-07-25,2019-10-09,77,59,0.766234


In [ ]:
# Build per-DIM yield profiles
DIMYield = (DfKnown.groupby(["AnimalId","LactationNumber","DaysInMilk"])
            ["TotalMilkYieldSessionKg"].median().reset_index()
            .rename(columns={"TotalMilkYieldSessionKg":"ExpYield"}))
print(f"DIM profiles : {len(DIMYield):,} rows")

DIM profiles : 2,779,386 rows


In [ ]:
# convert DIMYield to dict
DIMYieldDict = {}
for _, R in DIMYield.iterrows():
    DIMYieldDict[(R["AnimalId"], int(R["LactationNumber"]), int(R["DaysInMilk"]))] = R["ExpYield"]
print(f"DIMYield dict built : {len(DIMYieldDict):,} keys")

DIMYield dict built : 2,779,386 keys


In [ ]:
# Fix: use nearest known DIM for yield lookup (DIMNearestDict cell)
# Build per-animal per-lactation DIM→yield as sorted arrays for fast nearest lookup

print("Building nearest-DIM lookup...")
DIMNearestDict = {}
for (Aid, Lac), Group in DIMYield.groupby(["AnimalId","LactationNumber"]):
    Group = Group.sort_values("DaysInMilk")
    DIMNearestDict[(Aid, Lac)] = (
        Group["DaysInMilk"].values,
        Group["ExpYield"].values
    )
print(f"Animals with DIM profiles : {len(DIMNearestDict):,}")

def GetNearestYield(Aid, Lac, DIM):
    Key = (Aid, Lac)
    if Key not in DIMNearestDict:
        return np.nan
    DIMs, Yields = DIMNearestDict[Key]
    Idx = np.searchsorted(DIMs, DIM)
    Idx = min(Idx, len(DIMs)-1)
    return Yields[Idx]

# Test
print(f"Test lookup: {GetNearestYield('-1007048835775197250', 1, 290):.2f} kg")

Building nearest-DIM lookup...
Animals with DIM profiles : 17,116
Test lookup: 12.38 kg


In [ ]:
Start = time.time()
SigRows = []

for _, Row in DfGaps[DfGaps["GapDays"] > 30].head(100).iterrows():
    Aid      = Row["AnimalId"]
    GapDates = pd.date_range(Row["GapStart"], Row["GapEnd"], freq="D")

    for Date in GapDates:
        if pd.isna(Row["CalvingDate"]) or Date < pd.Timestamp(Row["CalvingDate"]):
            ProjLac = int(Row["PreLac"])
            ProjDIM = int(Row["PreDIM"]) + (Date - Row["PreDate"]).days
        else:
            ProjLac = int(Row["PostLac"])
            ProjDIM = (Date - pd.Timestamp(Row["CalvingDate"])).days + 1

        ExpYield = GetNearestYield(Aid, ProjLac, ProjDIM)

        SigRows.append({
            "AnimalId": Aid,
            "Date":     Date,
            "ProjLac":  ProjLac,
            "ProjDIM":  ProjDIM,
            "ExpYield": ExpYield
        })

DfSig = pd.DataFrame(SigRows).sort_values(["Date","AnimalId"]).reset_index(drop=True)
print(f"Rows built          : {len(DfSig):,}")
print(f"With expected yield : {DfSig['ExpYield'].notna().sum():,} ({DfSig['ExpYield'].notna().mean():.1%})")
print(f"Time for 100 animals: {time.time()-Start:.1f}s")
display(DfSig.head(5))

Rows built          : 6,783
With expected yield : 6,783 (100.0%)
Time for 100 animals: 0.3s


,AnimalId,Date,ProjLac,ProjDIM,ExpYield
0,-1071722002866787073,2019-06-25,2,374,8.708974
1,-1071722002866787073,2019-06-26,2,375,8.708974
2,-1071722002866787073,2019-06-27,2,376,8.708974
3,-1071722002866787073,2019-06-28,2,377,8.708974
4,-1071722002866787073,2019-06-29,2,378,8.708974


In [ ]:
Start = time.time()
SigRows = []

for _, Row in DfGaps[DfGaps["GapDays"] > 30].iterrows():
    Aid      = Row["AnimalId"]
    GapDates = pd.date_range(Row["GapStart"], Row["GapEnd"], freq="D")

    for Date in GapDates:
        if pd.isna(Row["CalvingDate"]) or Date < pd.Timestamp(Row["CalvingDate"]):
            ProjLac = int(Row["PreLac"])
            ProjDIM = int(Row["PreDIM"]) + (Date - Row["PreDate"]).days
        else:
            ProjLac = int(Row["PostLac"])
            ProjDIM = (Date - pd.Timestamp(Row["CalvingDate"])).days + 1

        ExpYield = GetNearestYield(Aid, ProjLac, ProjDIM)

        SigRows.append({
            "AnimalId": Aid,
            "Date":     Date,
            "ProjLac":  ProjLac,
            "ProjDIM":  ProjDIM,
            "ExpYield": ExpYield
        })

DfSig = pd.DataFrame(SigRows).sort_values(["Date","AnimalId"]).reset_index(drop=True)
print(f"Rows built          : {len(DfSig):,}")
print(f"With expected yield : {DfSig['ExpYield'].notna().sum():,} ({DfSig['ExpYield'].notna().mean():.1%})")
print(f"Time for all animals: {time.time()-Start:.1f}s")
display(DfSig.head(5))

Rows built          : 598,672
With expected yield : 598,672 (100.0%)
Time for all animals: 22.1s


,AnimalId,Date,ProjLac,ProjDIM,ExpYield
0,4464122382448073080,2019-06-16,2,314,5.261671
1,-4173944550966427194,2019-06-17,3,413,6.078138
2,-5818515762927999868,2019-06-17,3,268,5.216312
3,-8682444725890532196,2019-06-17,4,11,18.461209
4,2044805066252791229,2019-06-17,3,310,10.432625


In [ ]:
DfSig.to_csv(ProjectRoot / "DfSig.csv", index=False)
print("Saved")

Saved


In [ ]:
# Check uniqueness of Lac+DIM+Date combinations across all animals
Counts = (DfSig.groupby(["Date","ProjLac","ProjDIM"])
          ["AnimalId"].nunique().reset_index()
          .rename(columns={"AnimalId":"NAnimals"}))

print(f"Total Date+Lac+DIM combinations : {len(Counts):,}")
print(f"Unique to 1 animal              : {(Counts['NAnimals']==1).sum():,}")
print(f"Shared by 2 animals             : {(Counts['NAnimals']==2).sum():,}")
print(f"Shared by 3+ animals            : {(Counts['NAnimals']>=3).sum():,}")
print()
print(f"% uniquely identifiable by Lac+DIM+Date : {(Counts['NAnimals']==1).mean():.1%}")

Total Date+Lac+DIM combinations : 401,751
Unique to 1 animal              : 286,112
Shared by 2 animals             : 69,692
Shared by 3+ animals            : 45,947

% uniquely identifiable by Lac+DIM+Date : 71.2%


In [ ]:
Counts = (DfSig.groupby(["Date","ProjLac","ProjDIM"])
          ["AnimalId"].nunique().reset_index()
          .rename(columns={"AnimalId":"NAnimals"}))

UniqueSlots = (Counts[Counts["NAnimals"] == 1]
               .merge(DfSig[["Date","ProjLac","ProjDIM","AnimalId","ExpYield"]],
                      on=["Date","ProjLac","ProjDIM"], how="left"))

UniqueSlots.to_csv(ProjectRoot / "UniqueSlots.csv", index=False)
print(f"Counts      : {len(Counts):,}")
print(f"UniqueSlots : {len(UniqueSlots):,}")

Counts      : 401,751
UniqueSlots : 286,112


In [ ]:
UniqueSlots.to_csv(ProjectRoot / "UniqueSlots.csv", index=False)
print("Saved")

Saved


#Reoad after crash

In [6]:
# Reload after crash

ProjectRoot     = Path("/content/drive/MyDrive/MiniProject")
CleanedFilePath = ProjectRoot / "AnimalIDRecovered.csv"

DfKnown   = pd.read_csv(ProjectRoot / "DfKnown_clean.csv",
                         parse_dates=["EventDate"], low_memory=False)
DfMissing = pd.read_csv(ProjectRoot / "DfMissing_clean.csv",
                         parse_dates=["EventDate"], low_memory=False)

DfKnown["AnimalId"]   = DfKnown["AnimalId"].astype(str)
DfMissing["AnimalId"] = DfMissing["AnimalId"].astype(str)

DfKnown["EventDateInt"]   = DfKnown["EventDate"].astype("int64") // 10**9 // 86400
DfMissing["EventDateInt"] = DfMissing["EventDate"].astype("int64") // 10**9 // 86400

ClusterFeatures = [
    "TotalMilkYieldSessionKg",
    "MilkingDurationSeconds",
    "AverageMilkFlowKgPerMin",
    "MilkFlow30To60SecondsKgPerMin",
    "MilkYieldFirst2MinutesKg"
]

# Load any saved checkpoints if they exist
import os
if os.path.exists(ProjectRoot / "DfGaps.csv"):
    DfGaps = pd.read_csv(ProjectRoot / "DfGaps.csv",
                          parse_dates=["PreDate","GapStart","GapEnd","PostDate"])
    print(f"DfGaps loaded : {len(DfGaps):,} gap periods")
if os.path.exists(ProjectRoot / "DfGaps.csv"):
    DfGaps = pd.read_csv(ProjectRoot / "DfGaps.csv",
                          parse_dates=["PreDate","GapStart","GapEnd","PostDate","CalvingDate"])
    print(f"DfGaps loaded : {len(DfGaps):,}")

if os.path.exists(ProjectRoot / "DfSig.csv"):
    DfSig = pd.read_csv(ProjectRoot / "DfSig.csv", parse_dates=["Date"])
    print(f"DfSig loaded  : {len(DfSig):,}")

if os.path.exists(ProjectRoot / "UniqueSlots.csv"):
    UniqueSlots = pd.read_csv(ProjectRoot / "UniqueSlots.csv", parse_dates=["Date"])
    print(f"UniqueSlots loaded : {len(UniqueSlots):,}")

print(f"DfKnown   : {len(DfKnown):,}")
print(f"DfMissing : {len(DfMissing):,}")
print("READY")

DfGaps loaded : 8,930 gap periods
DfGaps loaded : 8,930
DfSig loaded  : 598,672
UniqueSlots loaded : 286,112
DfKnown   : 6,737,364
DfMissing : 1,701,003
READY


In [ ]:
#phase 1 - matching
import csv
Start     = time.time()
MissingByDate = {D: G for D, G in DfMissing.groupby("EventDate")}
YieldStd  = DfKnown.groupby("AnimalId")["TotalMilkYieldSessionKg"].std().to_dict()
Dates     = sorted(UniqueSlots["Date"].unique())
OutPath   = ProjectRoot / "DfResults.csv"

with open(OutPath, "w", newline="") as f:
    Writer = csv.writer(f)
    Writer.writerow(["OrigIdx","AnimalId","Date","ExpYield","ActYield"])

    for i, Date in enumerate(Dates):
        if Date not in MissingByDate:
            continue
        DayMissing = MissingByDate[Date]
        DaySlots   = UniqueSlots[UniqueSlots["Date"] == Date]
        DayYields  = DayMissing["TotalMilkYieldSessionKg"].values
        DayIndices = DayMissing.index.values

        for _, Slot in DaySlots.iterrows():
            Aid  = Slot["AnimalId"]
            ExpY = Slot["ExpYield"]
            Std  = YieldStd.get(Aid, 2.0)
            Mask = (DayYields >= ExpY - 2*Std) & (DayYields <= ExpY + 2*Std)
            for Idx, ActY in zip(DayIndices[Mask], DayYields[Mask]):
                Writer.writerow([Idx, Aid, str(Date)[:10], ExpY, ActY])

        if i % 100 == 0:
            print(f"  {i}/{len(Dates)} · {time.time()-Start:.0f}s")

print("Done — saved directly to Drive")

DfResults = pd.read_csv(OutPath)
print(f"Matched : {len(DfResults):,}")
print(f"Animals : {DfResults['AnimalId'].nunique():,}")

  0/861 · 9s
  100/861 · 194s
  200/861 · 446s
  300/861 · 723s
  400/861 · 981s
  500/861 · 1205s
  600/861 · 1504s
  700/861 · 1818s
  800/861 · 2186s
Done — saved directly to Drive


KeyboardInterrupt: 

In [ ]:
result = subprocess.run(['wc', '-l', str(ProjectRoot / "DfResults.csv")],
                       capture_output=True, text=True)
print(result.stdout)

208284258 /content/drive/MyDrive/MiniProject/DfResults.csv



In [ ]:
DfResultsSample = pd.read_csv(ProjectRoot / "DfResults.csv", nrows=1000)
print(DfResultsSample.head(10))
print()
# How many unique OrigIdx in sample — should be ~3 per animal per date
print(f"Unique OrigIdx : {DfResultsSample['OrigIdx'].nunique()}")
print(f"Unique animals : {DfResultsSample['AnimalId'].nunique()}")
print(f"Unique dates   : {DfResultsSample['Date'].nunique()}")

   OrigIdx             AnimalId        Date  ExpYield  ActYield
0     5394  4464122382448073080  2019-06-16  5.261671  6.577089
1    24917  4464122382448073080  2019-06-16  5.261671  8.890410
2    35143  4464122382448073080  2019-06-16  5.261671  9.162566
3    37543  4464122382448073080  2019-06-16  5.261671  6.168856
4    39026  4464122382448073080  2019-06-16  5.261671  8.708974
5    50543  4464122382448073080  2019-06-16  5.261671  7.847148
6    58606  4464122382448073080  2019-06-16  5.261671  6.985322
7    62389  4464122382448073080  2019-06-16  5.261671  5.125594
8    84092  4464122382448073080  2019-06-16  5.261671  7.438915
9    84523  4464122382448073080  2019-06-16  5.261671  8.981129

Unique OrigIdx : 975
Unique animals : 2
Unique dates   : 2


In [ ]:
# Check on 1 animal, 1 date only
Sample = DfResultsSample[DfResultsSample["AnimalId"] == "4464122382448073080"].head(1)
Aid    = Sample["AnimalId"].iloc[0]
ExpY   = Sample["ExpYield"].iloc[0]
Std    = YieldStd.get(Aid, 2.0)

print(f"Animal  : {Aid}")
print(f"ExpY    : {ExpY:.3f}")
print(f"Std     : {Std:.3f}")
print(f"±2std   : {ExpY-2*Std:.3f} to {ExpY+2*Std:.3f}")
print(f"±0.5kg  : {ExpY-0.5:.3f} to {ExpY+0.5:.3f}")
print(f"±0.1kg  : {ExpY-0.1:.3f} to {ExpY+0.1:.3f}")

IndexError: single positional indexer is out-of-bounds

In [ ]:
Aid  = DfResultsSample["AnimalId"].iloc[0]
ExpY = DfResultsSample["ExpYield"].iloc[0]
Std  = YieldStd.get(str(Aid), 2.0)

print(f"Animal  : {Aid}")
print(f"ExpY    : {ExpY:.3f}")
print(f"Std     : {Std:.3f}")
print(f"±2std   : {ExpY-2*Std:.3f} to {ExpY+2*Std:.3f}")
print(f"±0.5kg  : {ExpY-0.5:.3f} to {ExpY+0.5:.3f}")
print(f"±0.1kg  : {ExpY-0.1:.3f} to {ExpY+0.1:.3f}")

Animal  : 4464122382448073080
ExpY    : 5.262
Std     : 2.482
±2std   : 0.298 to 10.226
±0.5kg  : 4.762 to 5.762
±0.1kg  : 5.162 to 5.362


In [ ]:
# For each gap animal, assign ALL mystery records on their exact gap dates
Results = []
for _, Row in DfGaps[DfGaps["GapDays"] > 30].iterrows():
    Aid      = Row["AnimalId"]
    GapDates = pd.date_range(Row["GapStart"], Row["GapEnd"], freq="D")

    for Date in GapDates:
        if Date in MissingByDate:
            for Idx in MissingByDate[Date].index:
                Results.append((Idx, Aid, Date))

NameError: name 'DfGaps' is not defined

In [ ]:
# if crashed here, run RELOAD → DIMYield → DIMYieldDict → DIMNearestDict → Counts → Phase 1

In [ ]:
DfGaps      = pd.read_csv(ProjectRoot / "DfGaps.csv",
                           parse_dates=["PreDate","GapStart","GapEnd","PostDate","CalvingDate"])
DfSig       = pd.read_csv(ProjectRoot / "DfSig.csv", parse_dates=["Date"])
UniqueSlots = pd.read_csv(ProjectRoot / "UniqueSlots.csv", parse_dates=["Date"])
print(f"DfGaps:{len(DfGaps):,} DfSig:{len(DfSig):,} UniqueSlots:{len(UniqueSlots):,}")

DfGaps:8,930 DfSig:598,672 UniqueSlots:286,112


In [ ]:
MissingByDate = {D: G for D, G in DfMissing.groupby("EventDate")}
print(f"Dates indexed: {len(MissingByDate):,}")

Dates indexed: 830


In [ ]:
Start   = time.time()
OutPath = ProjectRoot / "DfResults.csv"

with open(OutPath, "w", newline="") as f:
    Writer = csv.writer(f)
    Writer.writerow(["OrigIdx","AnimalId","Date"])

    for i, (Date, Group) in enumerate(UniqueSlots.groupby("Date")):
        if Date not in MissingByDate:
            continue
        DayMissing = MissingByDate[Date]
        for _, Slot in Group.iterrows():
            Aid = Slot["AnimalId"]
            for Idx in DayMissing.index:
                Writer.writerow([Idx, Aid, str(Date)[:10]])

        if i % 100 == 0:
            print(f"  {i}/{UniqueSlots['Date'].nunique()} · {time.time()-Start:.0f}s")

print("Done")

  0/861 · 1s


KeyboardInterrupt: 

In [ ]:
AnimalYield = DfKnown[DfKnown["AnimalId"] == str(Aid)]["TotalMilkYieldSessionKg"]
YieldMean   = AnimalYield.mean()
YieldStd    = AnimalYield.std()
print(f"Known yield: {YieldMean:.3f} ± {YieldStd:.3f} kg")
print(f"Records in DfKnown for this animal: {len(AnimalYield):,}")

Known yield: 14.929 ± 2.908 kg
Records in DfKnown for this animal: 1,220


In [ ]:
Row = DfGaps[DfGaps["GapDays"] > 60].iloc[0]
Aid      = str(Row["AnimalId"])
GapStart = Row["GapStart"]
GapEnd   = Row["GapEnd"]
GapDates = pd.date_range(GapStart, GapEnd, freq="D")

GapRecords  = pd.concat([MissingByDate[D] for D in GapDates if D in MissingByDate])
AnimalYield = DfKnown[DfKnown["AnimalId"] == Aid]["TotalMilkYieldSessionKg"]
YieldMean   = AnimalYield.mean()
YieldStd    = AnimalYield.std()

print(f"Animal     : {Aid}")
print(f"Gap        : {GapStart.date()} → {GapEnd.date()} ({Row['GapDays']:.0f} days)")
print(f"Known yield: {YieldMean:.3f} ± {YieldStd:.3f} kg")
print(f"Mystery records in gap: {len(GapRecords):,}")
print()

for Sess in [1, 2, 3]:
    SessRecs = GapRecords[GapRecords["MilkingSession"] == Sess]
    Close    = SessRecs[SessRecs["TotalMilkYieldSessionKg"].between(YieldMean-0.5, YieldMean+0.5)]
    print(f"Session {Sess}: {len(SessRecs):,} records, {len(Close):,} within ±0.5kg of {YieldMean:.2f}")

Animal     : -1007048835775197250
Gap        : 2020-01-22 → 2020-03-27 (66 days)
Known yield: 14.929 ± 2.908 kg
Mystery records in gap: 131,711

Session 1: 44,017 records, 4,414 within ±0.5kg of 14.93
Session 2: 43,769 records, 4,445 within ±0.5kg of 14.93
Session 3: 43,925 records, 4,007 within ±0.5kg of 14.93


In [ ]:
Row = DfGaps[DfGaps["GapDays"] > 60].iloc[0]
Aid = str(Row["AnimalId"])

# Look at yield by lactation and DIM range
print(f"PreYield  : {Row['PreYield']:.2f} kg (Lac={Row['PreLac']:.0f} DIM={Row['PreDIM']:.0f})")
print(f"PostYield : {Row['PostYield']:.2f} kg (Lac={Row['PostLac']:.0f} DIM={Row['PostDIM']:.0f})")
print()

# Yield per lactation
ByLac = DfKnown[DfKnown["AnimalId"]==Aid].groupby("LactationNumber")["TotalMilkYieldSessionKg"].agg(["mean","min","max","count"])
display(ByLac)

PreYield  : 12.38 kg (Lac=1 DIM=289)
PostYield : 15.33 kg (Lac=2 DIM=2)



,mean,min,max,count
LactationNumber,,,,
1.0,14.045098,9.570799,17.417947,396
2.0,14.885549,5.579186,30.345330,711
3.0,18.295427,8.028585,23.858959,113


In [ ]:
print("Mystery record yield distribution during gap:")
print(GapRecords["TotalMilkYieldSessionKg"].describe())
print()
print("PreYield was 12.38 — looking for records around 8-13 kg range")
Close = GapRecords[GapRecords["TotalMilkYieldSessionKg"].between(8, 13)]
print(f"Records between 8-13 kg : {len(Close):,} out of {len(GapRecords):,}")
print(f"Per day per session     : {len(Close)/66/3:.0f}")

Mystery record yield distribution during gap:
count    131711.000000
mean         13.768201
std           3.714581
min           5.034875
25%          11.249091
50%          13.607771
75%          16.147888
max          53.977492
Name: TotalMilkYieldSessionKg, dtype: float64

PreYield was 12.38 — looking for records around 8-13 kg range
Records between 8-13 kg : 49,901 out of 131,711
Per day per session     : 252


In [ ]:
 # Test combined constraints on 1 animal
Row    = DfGaps[DfGaps["GapDays"] > 60].iloc[0]
Aid    = str(Row["AnimalId"])
PreLac = int(Row["PreLac"])
PreDIM = int(Row["PreDIM"])
PreDate= Row["PreDate"]

# Get per-DIM yield from known records for this lactation
KnownLac = DfKnown[
    (DfKnown["AnimalId"] == Aid) &
    (DfKnown["LactationNumber"] == PreLac)
].sort_values("DaysInMilk")

print(f"Animal : {Aid}")
print(f"Lac {PreLac} records in DfKnown : {len(KnownLac)}")
print()

# Test on 3 specific gap dates
TestDates = pd.date_range(Row["GapStart"], periods=3, freq="D")

for Date in TestDates:
    ExpDIM = PreDIM + (Date - PreDate).days

    # Get expected yield at this DIM
    NearestRow = KnownLac.iloc[(KnownLac["DaysInMilk"] - ExpDIM).abs().argmin()]
    ExpYield   = NearestRow["TotalMilkYieldSessionKg"]

    # Get mystery records on this date
    if Date not in MissingByDate:
        continue
    DayRecs = MissingByDate[Date]

    # Apply tight yield filter ±1kg
    Candidates = DayRecs[
        DayRecs["TotalMilkYieldSessionKg"].between(ExpYield-1, ExpYield+1)
    ]

    print(f"Date {Date.date()} | ExpDIM={ExpDIM} | ExpYield={ExpYield:.2f}")
    print(f"  All mystery records : {len(DayRecs):,}")
    print(f"  Within ±1kg        : {len(Candidates):,}")
    for Sess in [1,2,3]:
        N = len(Candidates[Candidates["MilkingSession"]==Sess])
        print(f"    Session {Sess}       : {N} candidates")
    print()

Animal : -1007048835775197250
Lac 1 records in DfKnown : 396

Date 2020-01-22 | ExpDIM=290 | ExpYield=12.38
  All mystery records : 1,626
  Within ±1kg        : 360
    Session 1       : 120 candidates
    Session 2       : 112 candidates
    Session 3       : 128 candidates

Date 2020-01-23 | ExpDIM=291 | ExpYield=12.38
  All mystery records : 1,853
  Within ±1kg        : 414
    Session 1       : 148 candidates
    Session 2       : 123 candidates
    Session 3       : 143 candidates

Date 2020-01-24 | ExpDIM=292 | ExpYield=12.38
  All mystery records : 1,867
  Within ±1kg        : 420
    Session 1       : 133 candidates
    Session 2       : 147 candidates
    Session 3       : 140 candidates



In [ ]:
# Track which mystery records appear consistently across ALL gap dates
# For session 1 only first — test concept

GapDates = pd.date_range(Row["GapStart"], Row["GapEnd"], freq="D")
PreLac   = int(Row["PreLac"])
PreDIM   = int(Row["PreDIM"])
PreDate  = Row["PreDate"]

# For each gap date, find session 1 candidates within ±1kg of expected yield
CandidateSets = []

for Date in GapDates:
    if Date not in MissingByDate:
        continue
    ExpDIM   = PreDIM + (Date - PreDate).days
    NearestRow = KnownLac.iloc[(KnownLac["DaysInMilk"] - ExpDIM).abs().argmin()]
    ExpYield   = NearestRow["TotalMilkYieldSessionKg"]

    DayRecs   = MissingByDate[Date]
    S1Recs    = DayRecs[DayRecs["MilkingSession"] == 1]
    Candidates = S1Recs[
        S1Recs["TotalMilkYieldSessionKg"].between(ExpYield-1, ExpYield+1)
    ]
    CandidateSets.append(set(Candidates.index))

# How many candidates appear on 50%+ of gap dates?
from collections import Counter
AllCandidates = Counter()
for S in CandidateSets:
    for Idx in S:
        AllCandidates[Idx] += 1

Total = len(CandidateSets)
Consistent = {Idx: Count for Idx, Count in AllCandidates.items()
              if Count >= Total * 0.5}

print(f"Gap days checked          : {Total}")
print(f"Candidates appearing 50%+ : {len(Consistent)}")
print(f"Candidates appearing 80%+ : {sum(1 for c in AllCandidates.values() if c >= Total*0.8)}")
print(f"Candidates appearing 90%+ : {sum(1 for c in AllCandidates.values() if c >= Total*0.9)}")
print(f"Candidates appearing 100% : {sum(1 for c in AllCandidates.values() if c == Total)}")

Gap days checked          : 66
Candidates appearing 50%+ : 0
Candidates appearing 80%+ : 0
Candidates appearing 90%+ : 0
Candidates appearing 100% : 0


#MultiModel approach

In [ ]:
# How consistent is flow rate across sessions for the same animal?
Sample = DfKnown[DfKnown["AnimalId"] == Aid]
print("Flow rate by session:")
display(Sample.groupby("MilkingSession")["AverageMilkFlowKgPerMin"].agg(["mean","std"]))
print()
print("Duration by session:")
display(Sample.groupby("MilkingSession")["MilkingDurationSeconds"].agg(["mean","std"]))

Flow rate by session:


,mean,std
MilkingSession,,
1,4.549719,0.549728
2,4.639427,0.514748
3,4.557416,0.505445



Duration by session:


,mean,std
MilkingSession,,
1,190.347932,23.915157
2,196.517157,29.221844
3,190.356608,29.971487


In [ ]:
Date = pd.Timestamp("2020-01-22")
DayRecs = MissingByDate[Date].copy()

FlowMean = Sample["AverageMilkFlowKgPerMin"].mean()
FlowStd  = Sample["AverageMilkFlowKgPerMin"].std()
DurMean  = Sample["MilkingDurationSeconds"].mean()
DurStd   = Sample["MilkingDurationSeconds"].std()

Candidates = DayRecs[
    (DayRecs["AverageMilkFlowKgPerMin"].between(FlowMean-FlowStd, FlowMean+FlowStd)) &
    (DayRecs["MilkingDurationSeconds"].between(DurMean-DurStd, DurMean+DurStd))
]

print(f"Flow    : {FlowMean:.3f} ± {FlowStd:.3f}")
print(f"Duration: {DurMean:.0f} ± {DurStd:.0f}")
print(f"Total mystery records on date  : {len(DayRecs):,}")
print(f"Matching flow+duration         : {len(Candidates):,}")
print(f"Per session                    : {len(Candidates)/3:.0f}")

Flow    : 4.582 ± 0.525
Duration: 192 ± 28
Total mystery records on date  : 1,626
Matching flow+duration         : 172
Per session                    : 57


In [ ]:
YieldMean = Sample["TotalMilkYieldSessionKg"].mean()
YieldStd  = Sample["TotalMilkYieldSessionKg"].std()

Candidates = DayRecs[
    (DayRecs["TotalMilkYieldSessionKg"].between(YieldMean-YieldStd, YieldMean+YieldStd)) &
    (DayRecs["AverageMilkFlowKgPerMin"].between(FlowMean-FlowStd, FlowMean+FlowStd)) &
    (DayRecs["MilkingDurationSeconds"].between(DurMean-DurStd, DurMean+DurStd))
]

print(f"All 3 features combined : {len(Candidates):,}")
print(f"Per session             : {len(Candidates)/3:.0f}")

# Try tighter tolerances
for Tol in [0.5, 0.3, 0.2]:
    C = DayRecs[
        (DayRecs["TotalMilkYieldSessionKg"].between(YieldMean-Tol*YieldStd, YieldMean+Tol*YieldStd)) &
        (DayRecs["AverageMilkFlowKgPerMin"].between(FlowMean-Tol*FlowStd, FlowMean+Tol*FlowStd)) &
        (DayRecs["MilkingDurationSeconds"].between(DurMean-Tol*DurStd, DurMean+Tol*DurStd))
    ]
    print(f"±{Tol} std all features : {len(C):,} ({len(C)/3:.1f} per session)")

All 3 features combined : 163
Per session             : 54
±0.5 std all features : 39 (13.0 per session)
±0.3 std all features : 16 (5.3 per session)
±0.2 std all features : 10 (3.3 per session)


In [ ]:
# Track consistent candidates across all 66 gap days
from collections import Counter

ConsistentBySession = {}

for Sess in [1, 2, 3]:
    CandidateCounter = Counter()

    for Date in GapDates:
        if Date not in MissingByDate:
            continue

        ExpDIM   = PreDIM + (Date - PreDate).days
        NearRow  = KnownLac.iloc[(KnownLac["DaysInMilk"]-ExpDIM).abs().argmin()]
        ExpYield = NearRow["TotalMilkYieldSessionKg"]

        DayRecs  = MissingByDate[Date]
        SessRecs = DayRecs[DayRecs["MilkingSession"] == Sess]

        Cands = SessRecs[
            (SessRecs["TotalMilkYieldSessionKg"].between(ExpYield-0.2*YieldStd, ExpYield+0.2*YieldStd)) &
            (SessRecs["AverageMilkFlowKgPerMin"].between(FlowMean-0.2*FlowStd, FlowMean+0.2*FlowStd)) &
            (SessRecs["MilkingDurationSeconds"].between(DurMean-0.2*DurStd, DurMean+0.2*DurStd))
        ]

        for Idx in Cands.index:
            CandidateCounter[Idx] += 1

    ConsistentBySession[Sess] = CandidateCounter
    Total = len([D for D in GapDates if D in MissingByDate])
    print(f"Session {Sess} — appearing on 50%+ of {Total} gap days: {sum(1 for c in CandidateCounter.values() if c >= Total*0.5)}")
    print(f"Session {Sess} — appearing on 80%+ of gap days         : {sum(1 for c in CandidateCounter.values() if c >= Total*0.8)}")
    print(f"Session {Sess} — appearing on 90%+ of gap days         : {sum(1 for c in CandidateCounter.values() if c >= Total*0.9)}")
    print()

Session 1 — appearing on 50%+ of 66 gap days: 0
Session 1 — appearing on 80%+ of gap days         : 0
Session 1 — appearing on 90%+ of gap days         : 0

Session 2 — appearing on 50%+ of 66 gap days: 0
Session 2 — appearing on 80%+ of gap days         : 0
Session 2 — appearing on 90%+ of gap days         : 0

Session 3 — appearing on 50%+ of 66 gap days: 0
Session 3 — appearing on 80%+ of gap days         : 0
Session 3 — appearing on 90%+ of gap days         : 0



In [ ]:
# Step 1: Select 200 known animals with gaps > 60 days as simulation set
np.random.seed(42)
SimAnimals = (DfGaps[DfGaps["GapDays"] > 60]["AnimalId"]
              .drop_duplicates()
              .sample(200, random_state=42)
              .tolist())

print(f"Simulation animals : {len(SimAnimals)}")

# Step 2: For each sim animal, get their gap period
SimGaps = DfGaps[
    (DfGaps["AnimalId"].isin(SimAnimals)) &
    (DfGaps["GapDays"] > 60)
].copy()

print(f"Gap periods        : {len(SimGaps)}")
print(f"Mean gap days      : {SimGaps['GapDays'].mean():.0f}")

Simulation animals : 200
Gap periods        : 229
Mean gap days      : 80


In [ ]:
SimProfiles = {}

for Aid in SimAnimals:
    AnimalRecs = DfKnown[DfKnown["AnimalId"] == str(Aid)]
    if len(AnimalRecs) == 0:
        continue

    SimProfiles[str(Aid)] = {
        "YieldMean":    AnimalRecs["TotalMilkYieldSessionKg"].mean(),
        "YieldStd":     AnimalRecs["TotalMilkYieldSessionKg"].std(),
        "FlowMean":     AnimalRecs["AverageMilkFlowKgPerMin"].mean(),
        "FlowStd":      AnimalRecs["AverageMilkFlowKgPerMin"].std(),
        "DurMean":      AnimalRecs["MilkingDurationSeconds"].mean(),
        "DurStd":       AnimalRecs["MilkingDurationSeconds"].std(),
        "Yield2Mean":   AnimalRecs["MilkYieldFirst2MinutesKg"].mean(),
        "Flow3060Mean": AnimalRecs["MilkFlow30To60SecondsKgPerMin"].mean(),
        "DIMYield":     (AnimalRecs.groupby(["LactationNumber","DaysInMilk"])
                        ["TotalMilkYieldSessionKg"].median().to_dict()),
        "NRecords":     len(AnimalRecs)
    }

print(f"Profiles built : {len(SimProfiles)}")
print(f"Sample keys    : {list(list(SimProfiles.values())[0].keys())}")

Profiles built : 200
Sample keys    : ['YieldMean', 'YieldStd', 'FlowMean', 'FlowStd', 'DurMean', 'DurStd', 'Yield2Mean', 'Flow3060Mean', 'DIMYield', 'NRecords']


In [ ]:
Features = ["TotalMilkYieldSessionKg","AverageMilkFlowKgPerMin",
            "MilkingDurationSeconds","MilkYieldFirst2MinutesKg",
            "MilkFlow30To60SecondsKgPerMin"]

# For each gap period, get mystery records on those exact dates
# and find the cluster that best matches the known profile

MatchResults = []

for _, GapRow in SimGaps.iterrows():
    Aid      = str(GapRow["AnimalId"])
    GapDates = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")
    Profile  = SimProfiles[Aid]

    # Get mystery records on gap dates
    GapRecs = pd.concat([
        MissingByDate[D] for D in GapDates if D in MissingByDate
    ])

    if len(GapRecs) < 10:
        continue

    # Profile vector for this animal
    ProfileVec = np.array([
        Profile["YieldMean"], Profile["FlowMean"],
        Profile["DurMean"],   Profile["Yield2Mean"],
        Profile["Flow3060Mean"]
    ])

    # Cluster gap records — number of clusters = estimated animals in pool
    # Use small K first — we want the cluster closest to known profile
    NClust = min(20, len(GapRecs) // 10)
    Scaler = StandardScaler()
    X      = Scaler.fit_transform(GapRecs[Features].fillna(0).values)
    Km     = KMeans(n_clusters=NClust, random_state=42, n_init=5)
    Km.fit(X)

    # Find closest cluster to known profile
    ProfScaled  = Scaler.transform(ProfileVec.reshape(1,-1))
    Distances   = np.linalg.norm(Km.cluster_centers_ - ProfScaled, axis=1)
    BestCluster = np.argmin(Distances)
    BestDist    = Distances[BestCluster]
    SecondDist  = np.sort(Distances)[1]

    # Records in best cluster
    BestRecs = GapRecs[Km.labels_ == BestCluster]

    MatchResults.append({
        "AnimalId":    Aid,
        "GapStart":    GapRow["GapStart"],
        "GapEnd":      GapRow["GapEnd"],
        "GapDays":     GapRow["GapDays"],
        "NMysteryRecs":len(GapRecs),
        "NMatchedRecs":len(BestRecs),
        "BestDist":    BestDist,
        "MarginRatio": SecondDist/BestDist if BestDist > 0 else 999,
        "NClust":      NClust
    })

DfMatchResults = pd.DataFrame(MatchResults)
print(f"Gap periods matched : {len(DfMatchResults)}")
print(f"Mean gap days       : {DfMatchResults['GapDays'].mean():.0f}")
print(f"Mean matched records: {DfMatchResults['NMatchedRecs'].mean():.0f}")
print(f"Mean margin ratio   : {DfMatchResults['MarginRatio'].mean():.2f}x")
display(DfMatchResults.head(5))

Gap periods matched : 229
Mean gap days       : 80
Mean matched records: 9920
Mean margin ratio   : 2.10x


,AnimalId,GapStart,GapEnd,GapDays,NMysteryRecs,NMatchedRecs,BestDist,MarginRatio,NClust
0,-1060927943023542974,2020-08-19,2020-10-24,67,104878,5960,0.717090,1.014030,20
1,-1071722002866787073,2019-06-25,2019-09-03,71,97307,6145,0.829637,1.257661,20
2,-1085174482013799453,2020-06-03,2020-08-06,65,134968,9356,0.506582,2.424361,20
3,-1163534269267890577,2019-07-10,2019-09-11,64,88818,6497,0.928607,1.013915,20
4,-1290611087230779174,2021-03-03,2021-05-14,73,154162,11008,0.543972,1.666480,20


In [ ]:
# Validate — does best cluster yield match known animal yield?
Validation = []

for _, Row in DfMatchResults.iterrows():
    Aid     = str(Row["AnimalId"])
    Profile = SimProfiles[Aid]

    # Get gap records again
    GapDates = pd.date_range(Row["GapStart"], Row["GapEnd"], freq="D")
    GapRecs  = pd.concat([MissingByDate[D] for D in GapDates if D in MissingByDate])

    # Recluster with same K
    NClust  = int(Row["NClust"])
    Scaler  = StandardScaler()
    X       = Scaler.fit_transform(GapRecs[Features].fillna(0).values)
    Km      = KMeans(n_clusters=NClust, random_state=42, n_init=5)
    Km.fit(X)

    ProfVec    = np.array([Profile["YieldMean"], Profile["FlowMean"],
                           Profile["DurMean"],   Profile["Yield2Mean"],
                           Profile["Flow3060Mean"]])
    ProfScaled = Scaler.transform(ProfVec.reshape(1,-1))
    BestClust  = np.argmin(np.linalg.norm(Km.cluster_centers_ - ProfScaled, axis=1))
    BestRecs   = GapRecs[Km.labels_ == BestClust]

    ActualYield = BestRecs["TotalMilkYieldSessionKg"].mean()
    YieldDiff   = abs(ActualYield - Profile["YieldMean"])

    Validation.append({
        "AnimalId":    Aid,
        "ExpYield":    Profile["YieldMean"],
        "ActYield":    ActualYield,
        "YieldDiff":   YieldDiff,
        "WithinHalfKg": YieldDiff < 0.5,
        "Within1Kg":   YieldDiff < 1.0
    })

DfVal = pd.DataFrame(Validation)
print(f"Gap periods validated : {len(DfVal)}")
print(f"Within ±0.5 kg        : {DfVal['WithinHalfKg'].sum():,} ({DfVal['WithinHalfKg'].mean():.1%})")
print(f"Within ±1.0 kg        : {DfVal['Within1Kg'].sum():,} ({DfVal['Within1Kg'].mean():.1%})")
print(f"Mean yield diff       : {DfVal['YieldDiff'].mean():.3f} kg")

Gap periods validated : 229
Within ±0.5 kg        : 71 (31.0%)
Within ±1.0 kg        : 139 (60.7%)
Mean yield diff       : 0.880 kg


In [ ]:
GapsPerAnimal = SimGaps.groupby("AnimalId").size()
print(f"Animals with 1 gap  : {(GapsPerAnimal==1).sum()}")
print(f"Animals with 2 gaps : {(GapsPerAnimal==2).sum()}")
print(f"Animals with 3 gaps : {(GapsPerAnimal==3).sum()}")
print()
print("For animals with 2+ gaps — the combined signature is much more unique")
print("Gap 1 dates + Gap 2 dates + biological state at each = near unique fingerprint")

Animals with 1 gap  : 174
Animals with 2 gaps : 24
Animals with 3 gaps : 1

For animals with 2+ gaps — the combined signature is much more unique
Gap 1 dates + Gap 2 dates + biological state at each = near unique fingerprint


In [ ]:
# How unique is the combined gap signature across all gap animals?
# Check: how many animals share the same PreLac + PostLac + GapDays bucket

DfGapsFull = DfGaps[DfGaps["GapDays"] > 60].copy()

# Round GapDays to nearest 10 to allow some flexibility
DfGapsFull["GapBucket"] = (DfGapsFull["GapDays"] / 10).round() * 10
DfGapsFull["PreDIMBucket"] = (DfGapsFull["PreDIM"] / 30).round() * 30
DfGapsFull["YieldBucket"] = (DfGapsFull["PreYield"] / 2).round() * 2

Uniqueness = DfGapsFull.groupby([
    "PreLac","PostLac","GapBucket","PreDIMBucket","YieldBucket"
]).size().reset_index(name="NAnimals")

print(f"Total combinations          : {len(Uniqueness):,}")
print(f"Unique to 1 animal          : {(Uniqueness['NAnimals']==1).sum():,} ({(Uniqueness['NAnimals']==1).mean():.1%})")
print(f"Shared by 2 animals         : {(Uniqueness['NAnimals']==2).sum():,}")
print(f"Shared by 3+ animals        : {(Uniqueness['NAnimals']>=3).sum():,}")

Total combinations          : 1,178
Unique to 1 animal          : 685 (58.1%)
Shared by 2 animals         : 182
Shared by 3+ animals        : 311


In [ ]:
# Build combined multi-gap signature per animal
MultiGapSig = []

for Aid, AnimalGaps in DfGaps[DfGaps["GapDays"] > 60].groupby("AnimalId"):
    AnimalGaps = AnimalGaps.sort_values("GapStart")

    Sig = {
        "AnimalId": Aid,
        "NGaps": len(AnimalGaps),
        # All gap periods combined
        "TotalGapDays": AnimalGaps["GapDays"].sum(),
        # Gap 1
        "G1Start": AnimalGaps.iloc[0]["GapStart"],
        "G1End":   AnimalGaps.iloc[0]["GapEnd"],
        "G1Days":  AnimalGaps.iloc[0]["GapDays"],
        "G1PreLac": AnimalGaps.iloc[0]["PreLac"],
        "G1PreDIM": AnimalGaps.iloc[0]["PreDIM"],
        "G1PostLac": AnimalGaps.iloc[0]["PostLac"],
        "G1PostDIM": AnimalGaps.iloc[0]["PostDIM"],
        "G1PreYield": AnimalGaps.iloc[0]["PreYield"],
        "G1PostYield": AnimalGaps.iloc[0]["PostYield"],
        # Gap 2 if exists
        "G2Start": AnimalGaps.iloc[1]["GapStart"] if len(AnimalGaps) > 1 else None,
        "G2Days":  AnimalGaps.iloc[1]["GapDays"]  if len(AnimalGaps) > 1 else None,
        "G2PreLac": AnimalGaps.iloc[1]["PreLac"]  if len(AnimalGaps) > 1 else None,
        "G2PreDIM": AnimalGaps.iloc[1]["PreDIM"]  if len(AnimalGaps) > 1 else None,
        "G2PreYield": AnimalGaps.iloc[1]["PreYield"] if len(AnimalGaps) > 1 else None,
    }
    MultiGapSig.append(Sig)

DfMultiGap = pd.DataFrame(MultiGapSig)
print(f"Animals with gap signatures  : {len(DfMultiGap):,}")
print(f"Animals with 1 gap           : {(DfMultiGap['NGaps']==1).sum():,}")
print(f"Animals with 2+ gaps         : {(DfMultiGap['NGaps']>=2).sum():,}")
print()

# Check uniqueness of combined signature
# Round to allow biological flexibility
DfMultiGap["G1DaysBucket"]   = (DfMultiGap["G1Days"] / 10).round() * 10
DfMultiGap["G1PreDIMBucket"] = (DfMultiGap["G1PreDIM"] / 30).round() * 30
DfMultiGap["G1YieldBucket"]  = (DfMultiGap["G1PreYield"] / 2).round() * 2
DfMultiGap["G2DaysBucket"]   = (DfMultiGap["G2Days"].fillna(0) / 10).round() * 10

Uniqueness = DfMultiGap.groupby([
    "NGaps","G1PreLac","G1PostLac","G1DaysBucket",
    "G1PreDIMBucket","G1YieldBucket","G2DaysBucket"
]).size().reset_index(name="NAnimals")

print(f"Total combinations           : {len(Uniqueness):,}")
print(f"Unique to 1 animal           : {(Uniqueness['NAnimals']==1).sum():,} ({(Uniqueness['NAnimals']==1).mean():.1%})")
print(f"Shared by 2                  : {(Uniqueness['NAnimals']==2).sum():,}")
print(f"Shared by 3+                 : {(Uniqueness['NAnimals']>=3).sum():,}")

Animals with gap signatures  : 4,167
Animals with 1 gap           : 3,492
Animals with 2+ gaps         : 675

Total combinations           : 1,321
Unique to 1 animal           : 918 (69.5%)
Shared by 2                  : 180
Shared by 3+                 : 223


In [ ]:
# Check how many sim animals are uniquely identifiable
SimMultiGap = DfMultiGap[DfMultiGap["AnimalId"].isin(SimAnimals)].copy()

# Merge with uniqueness
SimUnique = SimMultiGap.merge(
    Uniqueness,
    on=["NGaps","G1PreLac","G1PostLac","G1DaysBucket",
        "G1PreDIMBucket","G1YieldBucket","G2DaysBucket"],
    how="left"
)

print(f"Sim animals total            : {len(SimUnique):,}")
print(f"Uniquely identifiable        : {(SimUnique['NAnimals']==1).sum():,} ({(SimUnique['NAnimals']==1).mean():.1%})")
print(f"Shared by 2                  : {(SimUnique['NAnimals']==2).sum():,}")
print(f"Shared by 3+                 : {(SimUnique['NAnimals']>=3).sum():,}")

Sim animals total            : 200
Uniquely identifiable        : 39 (19.5%)
Shared by 2                  : 15
Shared by 3+                 : 146


In [ ]:
# Tighter buckets + more dimensions
DfMultiGap["G1DaysBucket"]    = (DfMultiGap["G1Days"] / 5).round() * 5
DfMultiGap["G1PreDIMBucket"]  = (DfMultiGap["G1PreDIM"] / 10).round() * 10
DfMultiGap["G1YieldBucket"]   = (DfMultiGap["G1PreYield"] / 1).round() * 1
DfMultiGap["G1PostYieldBucket"] = (DfMultiGap["G1PostYield"] / 1).round() * 1
DfMultiGap["G2DaysBucket"]    = (DfMultiGap["G2Days"].fillna(0) / 5).round() * 5
DfMultiGap["G2PreDIMBucket"]  = (DfMultiGap["G2PreDIM"].fillna(0) / 10).round() * 10

Uniqueness2 = DfMultiGap.groupby([
    "NGaps","G1PreLac","G1PostLac",
    "G1DaysBucket","G1PreDIMBucket",
    "G1YieldBucket","G1PostYieldBucket",
    "G2DaysBucket","G2PreDIMBucket"
]).size().reset_index(name="NAnimals")

print(f"Total combinations  : {len(Uniqueness2):,}")
print(f"Unique to 1 animal  : {(Uniqueness2['NAnimals']==1).sum():,} ({(Uniqueness2['NAnimals']==1).mean():.1%})")
print(f"Shared by 2         : {(Uniqueness2['NAnimals']==2).sum():,}")
print(f"Shared by 3+        : {(Uniqueness2['NAnimals']>=3).sum():,}")

# Check sim animals
SimUnique2 = DfMultiGap[DfMultiGap["AnimalId"].isin(SimAnimals)].merge(
    Uniqueness2,
    on=["NGaps","G1PreLac","G1PostLac",
        "G1DaysBucket","G1PreDIMBucket",
        "G1YieldBucket","G1PostYieldBucket",
        "G2DaysBucket","G2PreDIMBucket"],
    how="left"
)
print(f"\nSim animals uniquely identifiable : {(SimUnique2['NAnimals']==1).sum():,} / {len(SimUnique2):,} ({(SimUnique2['NAnimals']==1).mean():.1%})")

Total combinations  : 3,327
Unique to 1 animal  : 2,903 (87.3%)
Shared by 2         : 231
Shared by 3+        : 193

Sim animals uniquely identifiable : 135 / 200 (67.5%)


In [ ]:
UniqueSimAnimals = [str(x) for x in SimUnique2[SimUnique2["NAnimals"] == 1]["AnimalId"].tolist()]
print(f"Unique sim animals : {len(UniqueSimAnimals)}")
print(f"Sample : {UniqueSimAnimals[:3]}")
print(f"In SimProfiles : {sum(1 for a in UniqueSimAnimals if a in SimProfiles)}")

Unique sim animals : 135
Sample : ['-9014554096887259434', '-9007167765032396740', '-8761879329991508167']
In SimProfiles : 135


In [ ]:
ValidationResults = []

for Aid in UniqueSimAnimals:
    Profile    = SimProfiles[Aid]
    AnimalGaps = SimGaps[SimGaps["AnimalId"].astype(str) == Aid].sort_values("GapStart")

    AllGapRecs = []
    for _, GapRow in AnimalGaps.iterrows():
        GapDates = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")
        for D in GapDates:
            if D in MissingByDate:
                AllGapRecs.append(MissingByDate[D])

    if len(AllGapRecs) == 0:
        continue

    GapRecs  = pd.concat(AllGapRecs)
    FlowMean = Profile["FlowMean"]
    FlowStd  = Profile["FlowStd"]
    DurMean  = Profile["DurMean"]
    DurStd   = Profile["DurStd"]

    Matched  = GapRecs[
        (GapRecs["AverageMilkFlowKgPerMin"].between(
            FlowMean - 0.5*FlowStd, FlowMean + 0.5*FlowStd)) &
        (GapRecs["MilkingDurationSeconds"].between(
            DurMean - 0.5*DurStd, DurMean + 0.5*DurStd))
    ]

    if len(Matched) == 0:
        continue

    ActYield  = Matched["TotalMilkYieldSessionKg"].mean()
    YieldDiff = abs(ActYield - Profile["YieldMean"])

    ValidationResults.append({
        "AnimalId":  Aid,
        "ExpYield":  Profile["YieldMean"],
        "ActYield":  ActYield,
        "YieldDiff": YieldDiff,
        "NMatched":  len(Matched),
        "TotalRecs": len(GapRecs),
        "MatchRate": len(Matched)/len(GapRecs)
    })

DfValidation = pd.DataFrame(ValidationResults)
print(f"Validated          : {len(DfValidation):,}")
print(f"Within ±0.5 kg     : {(DfValidation['YieldDiff']<0.5).sum():,} ({(DfValidation['YieldDiff']<0.5).mean():.1%})")
print(f"Within ±1.0 kg     : {(DfValidation['YieldDiff']<1.0).sum():,} ({(DfValidation['YieldDiff']<1.0).mean():.1%})")
print(f"Mean yield diff    : {DfValidation['YieldDiff'].mean():.3f} kg")
print(f"Mean match rate    : {DfValidation['MatchRate'].mean():.1%}")
print(f"Mean matched recs  : {DfValidation['NMatched'].mean():.0f}")

Validated          : 135
Within ±0.5 kg     : 128 (94.8%)
Within ±1.0 kg     : 133 (98.5%)
Mean yield diff    : 0.184 kg
Mean match rate    : 6.1%
Mean matched recs  : 12626


In [ ]:
# For one validated animal — check how many records per day
Aid     = ValidationResults[0]["AnimalId"]
Profile = SimProfiles[Aid]
AnimalGaps = SimGaps[SimGaps["AnimalId"].astype(str) == Aid].sort_values("GapStart")

GapRecs = pd.concat([
    MissingByDate[D]
    for _, GapRow in AnimalGaps.iterrows()
    for D in pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")
    if D in MissingByDate
])

Matched = GapRecs[
    (GapRecs["AverageMilkFlowKgPerMin"].between(
        Profile["FlowMean"] - 0.5*Profile["FlowStd"],
        Profile["FlowMean"] + 0.5*Profile["FlowStd"])) &
    (GapRecs["MilkingDurationSeconds"].between(
        Profile["DurMean"] - 0.5*Profile["DurStd"],
        Profile["DurMean"] + 0.5*Profile["DurStd"]))
]

PerDay = Matched.groupby(["EventDate","MilkingSession"]).size().reset_index(name="N")
print(f"Animal : {Aid}")
print(f"Total matched : {len(Matched):,}")
print(f"Mean records per day per session : {PerDay['N'].mean():.0f}")
print(f"Min : {PerDay['N'].min()} Max : {PerDay['N'].max()}")
print()
print("We want 1 per day per session — currently getting:")
display(PerDay.head(10))

Animal : -9014554096887259434
Total matched : 6,322
Mean records per day per session : 28
Min : 6 Max : 54

We want 1 per day per session — currently getting:


,EventDate,MilkingSession,N
0,2020-04-07,1,26
1,2020-04-07,2,27
2,2020-04-07,3,24
3,2020-04-08,1,30
4,2020-04-08,2,24
5,2020-04-08,3,26
6,2020-04-09,1,25
7,2020-04-09,2,28
8,2020-04-09,3,35
9,2020-04-10,1,27


In [ ]:
# Full assignment — animal ID level
# Assign AnimalId to ALL mystery records on gap dates
# Only for uniquely identifiable animals with yield validation passing

Assignments = []

for Aid in UniqueSimAnimals:
    Profile    = SimProfiles[Aid]
    AnimalGaps = SimGaps[SimGaps["AnimalId"].astype(str) == Aid].sort_values("GapStart")

    # Validate yield first — only assign if within ±1kg
    AllGapRecs = []
    for _, GapRow in AnimalGaps.iterrows():
        GapDates = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")
        for D in GapDates:
            if D in MissingByDate:
                AllGapRecs.append(MissingByDate[D])

    if len(AllGapRecs) == 0:
        continue

    GapRecs = pd.concat(AllGapRecs)

    # Quick yield check
    Matched = GapRecs[
        (GapRecs["AverageMilkFlowKgPerMin"].between(
            Profile["FlowMean"] - 0.5*Profile["FlowStd"],
            Profile["FlowMean"] + 0.5*Profile["FlowStd"])) &
        (GapRecs["MilkingDurationSeconds"].between(
            Profile["DurMean"] - 0.5*Profile["DurStd"],
            Profile["DurMean"] + 0.5*Profile["DurStd"]))
    ]

    if len(Matched) == 0:
        continue

    ActYield  = Matched["TotalMilkYieldSessionKg"].mean()
    YieldDiff = abs(ActYield - Profile["YieldMean"])

    # Only assign if yield validates
    if YieldDiff > 1.0:
        continue

    # Assign AnimalId to ALL records on gap dates
    for _, GapRow in AnimalGaps.iterrows():
        GapDates = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")
        for D in GapDates:
            if D not in MissingByDate:
                continue
            for Idx in MissingByDate[D].index:
                Assignments.append({
                    "OrigIdx":   Idx,
                    "AnimalId":  Aid,
                    "Date":      D,
                    "YieldDiff": YieldDiff
                })

DfAssign = pd.DataFrame(Assignments)
print(f"Total assignments    : {len(DfAssign):,}")
print(f"Unique animals       : {DfAssign['AnimalId'].nunique():,}")
print(f"Mean yield diff      : {DfAssign['YieldDiff'].mean():.3f} kg")

Total assignments    : 26,466,172
Unique animals       : 133
Mean yield diff      : 0.193 kg


In [ ]:
# Fit Wood's curve for each sim animal
from scipy.optimize import curve_fit

def WoodsCurve(T, A, B, C):
    return A * (T**B) * np.exp(-C * T)

WoodsParams = {}

for Aid in UniqueSimAnimals:
    AnimalRecs = DfKnown[DfKnown["AnimalId"] == Aid].copy()

    # Fit per lactation
    LacParams = {}
    for Lac, LacRecs in AnimalRecs.groupby("LactationNumber"):
        LacRecs = LacRecs[LacRecs["DaysInMilk"] > 0].sort_values("DaysInMilk")
        if len(LacRecs) < 10:
            continue
        T = LacRecs["DaysInMilk"].values.astype(float)
        Y = LacRecs["TotalMilkYieldSessionKg"].values.astype(float)
        try:
            Params, _ = curve_fit(WoodsCurve, T, Y,
                                  p0=[10, 0.1, 0.003],
                                  maxfev=5000,
                                  bounds=([0,0,0],[100,2,0.1]))
            LacParams[int(Lac)] = Params
        except:
            pass

    if LacParams:
        WoodsParams[Aid] = LacParams

print(f"Wood's curve fitted : {len(WoodsParams):,} animals")
print(f"Sample params (A,B,C): {list(WoodsParams.values())[0]}")

Wood's curve fitted : 135 animals
Sample params (A,B,C): {1: array([6.01885164e+00, 2.29037575e-01, 1.93799635e-03]), 2: array([8.40592784e+00, 2.32848769e-01, 2.66840428e-03]), 3: array([1.02460065e+01, 2.39287594e-01, 4.33162866e-03])}


In [ ]:
import time
Start = time.time()

RecordAssignments = []

for Aid in UniqueSimAnimals[:10]:  # test on 10 first
    Profile    = SimProfiles[Aid]
    AnimalGaps = SimGaps[SimGaps["AnimalId"].astype(str) == Aid].sort_values("GapStart")

    if Aid not in WoodsParams:
        continue

    for _, GapRow in AnimalGaps.iterrows():
        GapDates = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")
        PreLac   = int(GapRow["PreLac"])
        PreDIM   = int(GapRow["PreDIM"])
        PreDate  = GapRow["PreDate"]
        CalvDate = GapRow["CalvingDate"]
        PostLac  = int(GapRow["PostLac"])

        for Date in GapDates:
            if Date not in MissingByDate:
                continue

            # Expected DIM and Lac on this date
            if pd.isna(CalvDate) or Date < pd.Timestamp(CalvDate):
                ExpLac = PreLac
                ExpDIM = PreDIM + (Date - pd.Timestamp(PreDate)).days
            else:
                ExpLac = PostLac
                ExpDIM = (Date - pd.Timestamp(CalvDate)).days + 1

            # Expected yield from Wood's curve
            if ExpLac in WoodsParams[Aid]:
                A, B, C    = WoodsParams[Aid][ExpLac]
                ExpYield   = WoodsCurve(max(ExpDIM, 1), A, B, C)
            else:
                ExpYield   = Profile["YieldMean"]

            DayRecs = MissingByDate[Date]

            # Per session — pick closest record to expected yield + flow + duration
            for Sess in [1, 2, 3]:
                SessRecs = DayRecs[DayRecs["MilkingSession"] == Sess].copy()
                if len(SessRecs) == 0:
                    continue

                # Score each record — distance across all features
                SessRecs["Score"] = (
                    abs(SessRecs["TotalMilkYieldSessionKg"] - ExpYield) /
                        max(Profile["YieldStd"], 0.1) +
                    abs(SessRecs["AverageMilkFlowKgPerMin"] - Profile["FlowMean"]) /
                        max(Profile["FlowStd"], 0.1) +
                    abs(SessRecs["MilkingDurationSeconds"] - Profile["DurMean"]) /
                        max(Profile["DurStd"], 1)
                )

                BestIdx   = SessRecs["Score"].idxmin()
                BestScore = SessRecs["Score"].min()
                SecondScore = SessRecs["Score"].nsmallest(2).iloc[-1] \
                              if len(SessRecs) > 1 else 999

                RecordAssignments.append({
                    "OrigIdx":     BestIdx,
                    "AnimalId":    Aid,
                    "Date":        Date,
                    "Session":     Sess,
                    "ExpYield":    ExpYield,
                    "ActYield":    SessRecs.loc[BestIdx,"TotalMilkYieldSessionKg"],
                    "BestScore":   BestScore,
                    "MarginRatio": SecondScore/BestScore if BestScore > 0 else 999,
                    "ExpDIM":      ExpDIM,
                    "ExpLac":      ExpLac
                })

DfRecAssign = pd.DataFrame(RecordAssignments)
print(f"Records assigned    : {len(DfRecAssign):,}")
print(f"Animals             : {DfRecAssign['AnimalId'].nunique():,}")
print(f"Mean yield diff     : {(DfRecAssign['ActYield']-DfRecAssign['ExpYield']).abs().mean():.3f} kg")
print(f"Mean margin ratio   : {DfRecAssign['MarginRatio'].mean():.2f}x")
print(f"Time for 10 animals : {time.time()-Start:.1f}s")

Records assigned    : 2,889
Animals             : 10
Mean yield diff     : 4.960 kg
Mean margin ratio   : 1.03x
Time for 10 animals : 25.3s


In [ ]:
# Check Wood's curve predictions vs actual for one animal
Aid    = UniqueSimAnimals[0]
GapRow = SimGaps[SimGaps["AnimalId"].astype(str) == Aid].iloc[0]
PreLac = int(GapRow["PreLac"])
PreDIM = int(GapRow["PreDIM"])

A, B, C = WoodsParams[Aid][PreLac]

print(f"Animal : {Aid}")
print(f"PreLac : {PreLac}, PreDIM : {PreDIM}")
print(f"Wood's params : A={A:.3f} B={B:.3f} C={C:.6f}")
print()
print("Expected yield at key DIM values:")
for DIM in [PreDIM, PreDIM+10, PreDIM+20, PreDIM+30]:
    print(f"  DIM {DIM} → {WoodsCurve(DIM, A, B, C):.2f} kg")

print()
print("Actual known yield near this DIM:")
ActRecs = DfKnown[
    (DfKnown["AnimalId"] == Aid) &
    (DfKnown["LactationNumber"] == PreLac) &
    (DfKnown["DaysInMilk"].between(PreDIM-10, PreDIM+10))
]
print(f"  Mean: {ActRecs['TotalMilkYieldSessionKg'].mean():.2f} kg")
print(f"  Std : {ActRecs['TotalMilkYieldSessionKg'].std():.2f} kg")

Animal : -9014554096887259434
PreLac : 1, PreDIM : 267
Wood's params : A=6.019 B=0.229 C=0.001938

Expected yield at key DIM values:
  DIM 267 → 12.90 kg
  DIM 277 → 12.76 kg
  DIM 287 → 12.62 kg
  DIM 297 → 12.47 kg

Actual known yield near this DIM:
  Mean: 13.04 kg
  Std : 0.87 kg


In [ ]:
# Check score components for one date
Aid    = UniqueSimAnimals[0]
GapRow = SimGaps[SimGaps["AnimalId"].astype(str) == Aid].iloc[0]
Date   = GapRow["GapStart"]
Profile = SimProfiles[Aid]

PreLac  = int(GapRow["PreLac"])
PreDIM  = int(GapRow["PreDIM"])
PreDate = GapRow["PreDate"]
A, B, C = WoodsParams[Aid][PreLac]
ExpDIM  = PreDIM + (Date - pd.Timestamp(PreDate)).days
ExpYield = WoodsCurve(max(ExpDIM,1), A, B, C)

DayRecs = MissingByDate[Date].copy()
S1 = DayRecs[DayRecs["MilkingSession"]==1].copy()

S1["YieldTerm"] = abs(S1["TotalMilkYieldSessionKg"] - ExpYield) / max(Profile["YieldStd"],0.1)
S1["FlowTerm"]  = abs(S1["AverageMilkFlowKgPerMin"] - Profile["FlowMean"]) / max(Profile["FlowStd"],0.1)
S1["DurTerm"]   = abs(S1["MilkingDurationSeconds"] - Profile["DurMean"]) / max(Profile["DurStd"],1)
S1["Score"]     = S1["YieldTerm"] + S1["FlowTerm"] + S1["DurTerm"]

Best = S1.nsmallest(3,"Score")
print(f"ExpYield : {ExpYield:.2f}  ExpFlow : {Profile['FlowMean']:.2f}  ExpDur : {Profile['DurMean']:.0f}")
print()
display(Best[["TotalMilkYieldSessionKg","AverageMilkFlowKgPerMin",
              "MilkingDurationSeconds","YieldTerm","FlowTerm","DurTerm","Score"]])

ExpYield : 12.88  ExpFlow : 3.77  ExpDur : 256



,TotalMilkYieldSessionKg,AverageMilkFlowKgPerMin,MilkingDurationSeconds,YieldTerm,FlowTerm,DurTerm,Score
628290,15.830374,3.719457,255,0.942713,0.120111,0.030450,1.093274
811283,15.830374,3.719457,252,0.942713,0.120111,0.114577,1.177401
1664958,15.694296,3.719457,250,0.899160,0.120111,0.170661,1.189932


In [ ]:
print(f"Records within ±0.5kg of {ExpYield:.2f}:")
print(len(S1[S1["TotalMilkYieldSessionKg"].between(ExpYield-0.5, ExpYield+0.5)]))
print(f"Records within ±1kg:")
print(len(S1[S1["TotalMilkYieldSessionKg"].between(ExpYield-1, ExpYield+1)]))
print(f"Records within ±2kg:")
print(len(S1[S1["TotalMilkYieldSessionKg"].between(ExpYield-2, ExpYield+2)]))

Records within ±0.5kg of 12.88:
81
Records within ±1kg:
152
Records within ±2kg:
285


In [ ]:
# Yield-first approach — filter by yield first, then pick best flow+duration
S1Filtered = S1[S1["TotalMilkYieldSessionKg"].between(ExpYield-0.5, ExpYield+0.5)].copy()

S1Filtered["Score"] = (
    abs(S1Filtered["AverageMilkFlowKgPerMin"] - Profile["FlowMean"]) / max(Profile["FlowStd"], 0.1) +
    abs(S1Filtered["MilkingDurationSeconds"] - Profile["DurMean"]) / max(Profile["DurStd"], 1)
)

Best = S1Filtered.nsmallest(3, "Score")
print(f"ExpYield : {ExpYield:.2f}  ExpFlow : {Profile['FlowMean']:.2f}  ExpDur : {Profile['DurMean']:.0f}")
print(f"Candidates within ±0.5kg : {len(S1Filtered)}")
display(Best[["TotalMilkYieldSessionKg","AverageMilkFlowKgPerMin","MilkingDurationSeconds","Score"]])

ExpYield : 12.88  ExpFlow : 3.77  ExpDur : 256
Candidates within ±0.5kg : 81


,TotalMilkYieldSessionKg,AverageMilkFlowKgPerMin,MilkingDurationSeconds,Score
444380,13.154179,3.719457,211,1.384422
1343067,12.882023,3.719457,208,1.468549
1166391,13.018101,3.719457,206,1.524634


In [ ]:
Start = time.time()
RecordAssignments = []

for Aid in UniqueSimAnimals:
    Profile    = SimProfiles[Aid]
    AnimalGaps = SimGaps[SimGaps["AnimalId"].astype(str) == Aid].sort_values("GapStart")
    if Aid not in WoodsParams:
        continue

    for _, GapRow in AnimalGaps.iterrows():
        GapDates = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")
        PreLac   = int(GapRow["PreLac"])
        PreDIM   = int(GapRow["PreDIM"])
        PreDate  = GapRow["PreDate"]
        CalvDate = GapRow["CalvingDate"]
        PostLac  = int(GapRow["PostLac"])

        for Date in GapDates:
            if Date not in MissingByDate:
                continue

            if pd.isna(CalvDate) or Date < pd.Timestamp(CalvDate):
                ExpLac = PreLac
                ExpDIM = PreDIM + (Date - pd.Timestamp(PreDate)).days
            else:
                ExpLac = PostLac
                ExpDIM = (Date - pd.Timestamp(CalvDate)).days + 1

            if ExpLac in WoodsParams[Aid]:
                A, B, C  = WoodsParams[Aid][ExpLac]
                ExpYield = WoodsCurve(max(ExpDIM, 1), A, B, C)
            else:
                ExpYield = Profile["YieldMean"]

            DayRecs = MissingByDate[Date]

            for Sess in [1, 2, 3]:
                SessRecs = DayRecs[DayRecs["MilkingSession"] == Sess].copy()
                if len(SessRecs) == 0:
                    continue

                # Step 1: filter by yield ±0.5kg
                Filtered = SessRecs[
                    SessRecs["TotalMilkYieldSessionKg"].between(
                        ExpYield - 0.5, ExpYield + 0.5)
                ].copy()

                # Fall back to ±1kg if no candidates
                if len(Filtered) == 0:
                    Filtered = SessRecs[
                        SessRecs["TotalMilkYieldSessionKg"].between(
                            ExpYield - 1.0, ExpYield + 1.0)
                    ].copy()

                if len(Filtered) == 0:
                    continue

                # Step 2: rank by flow + duration
                Filtered["Score"] = (
                    abs(Filtered["AverageMilkFlowKgPerMin"] - Profile["FlowMean"]) /
                        max(Profile["FlowStd"], 0.1) +
                    abs(Filtered["MilkingDurationSeconds"] - Profile["DurMean"]) /
                        max(Profile["DurStd"], 1)
                )

                BestIdx    = Filtered["Score"].idxmin()
                BestScore  = Filtered["Score"].min()
                SecondScore= Filtered["Score"].nsmallest(2).iloc[-1] \
                             if len(Filtered) > 1 else 999

                RecordAssignments.append({
                    "OrigIdx":     BestIdx,
                    "AnimalId":    Aid,
                    "Date":        Date,
                    "Session":     Sess,
                    "ExpYield":    ExpYield,
                    "ActYield":    Filtered.loc[BestIdx,"TotalMilkYieldSessionKg"],
                    "MarginRatio": SecondScore/BestScore if BestScore > 0 else 999,
                    "ExpDIM":      ExpDIM,
                    "ExpLac":      ExpLac
                })

DfRecAssign = pd.DataFrame(RecordAssignments)
print(f"Records assigned   : {len(DfRecAssign):,}")
print(f"Animals            : {DfRecAssign['AnimalId'].nunique():,}")
print(f"Mean yield diff    : {(DfRecAssign['ActYield']-DfRecAssign['ExpYield']).abs().mean():.3f} kg")
print(f"Mean margin ratio  : {DfRecAssign['MarginRatio'].mean():.2f}x")
print(f"Time               : {time.time()-Start:.1f}s")

Records assigned   : 35,131
Animals            : 135
Mean yield diff    : 0.346 kg
Mean margin ratio  : 16.42x
Time               : 174.5s


In [ ]:
# Validate assigned records
# Check: are there conflicts? (same record assigned to multiple animals)
# Check: yield consistency across gap dates per animal

DupCheck = DfRecAssign.groupby("OrigIdx")["AnimalId"].nunique()
print(f"Records assigned to exactly 1 animal : {(DupCheck==1).sum():,}")
print(f"Records assigned to 2+ animals       : {(DupCheck>1).sum():,}")
print()

# Yield consistency per animal — does it follow Wood's curve?
print("Yield diff distribution:")
print(DfRecAssign["ActYield"].subtract(DfRecAssign["ExpYield"]).abs().describe())
print()

# Records per animal per day — should be 3
PerDay = DfRecAssign.groupby(["AnimalId","Date"]).size()
print(f"Mean records per animal per day : {PerDay.mean():.1f}")
print(f"Days with exactly 3 records     : {(PerDay==3).sum():,} / {len(PerDay):,}")
print(f"Days with fewer than 3          : {(PerDay<3).sum():,}")

Records assigned to exactly 1 animal : 28,804
Records assigned to 2+ animals       : 2,760

Yield diff distribution:
count    35131.000000
mean         0.345878
std          0.176051
min          0.000009
25%          0.233088
50%          0.364961
75%          0.448416
max          0.999986
dtype: float64

Mean records per animal per day : 3.0
Days with exactly 3 records     : 11,478 / 11,889
Days with fewer than 3          : 411


In [ ]:
# Resolve conflicts — keep highest margin ratio
UniqueAssign  = DfRecAssign[DfRecAssign["OrigIdx"].isin(
    DupCheck[DupCheck==1].index)]
ConflictAssign = DfRecAssign[DfRecAssign["OrigIdx"].isin(
    DupCheck[DupCheck>1].index)]

ResolvedConflicts = (ConflictAssign
                     .sort_values("MarginRatio", ascending=False)
                     .groupby("OrigIdx").first().reset_index())

DfFinal = pd.concat([UniqueAssign, ResolvedConflicts], ignore_index=True)

print(f"Before conflict resolution : {len(DfRecAssign):,}")
print(f"After conflict resolution  : {len(DfFinal):,}")
print(f"Unique animals             : {DfFinal['AnimalId'].nunique():,}")
print(f"Mean yield diff            : {(DfFinal['ActYield']-DfFinal['ExpYield']).abs().mean():.3f} kg")
print(f"Mean margin ratio          : {DfFinal['MarginRatio'].mean():.2f}x")

# Save sim results
DfFinal.to_csv(ProjectRoot / "DfSimAssign.csv", index=False)
print("Saved")

Before conflict resolution : 35,131
After conflict resolution  : 31,564
Unique animals             : 135
Mean yield diff            : 0.344 kg
Mean margin ratio          : 12.22x
Saved


In [ ]:
# Ground truth validation on sim animals
# For each assigned record, check if its features match the known animal
# This is the true accuracy test

GTResults = []

for Aid in DfFinal["AnimalId"].unique():
    Profile     = SimProfiles[Aid]
    Assigned    = DfFinal[DfFinal["AnimalId"] == Aid]

    # Compare assigned record features to known profile
    YieldDiff = abs(Assigned["ActYield"].mean() - Profile["YieldMean"])

    # Get flow and duration of assigned records from DfMissing
    AssignedRecs = DfMissing.loc[Assigned["OrigIdx"]]
    FlowDiff  = abs(AssignedRecs["AverageMilkFlowKgPerMin"].mean() - Profile["FlowMean"])
    DurDiff   = abs(AssignedRecs["MilkingDurationSeconds"].mean()  - Profile["DurMean"])

    GTResults.append({
        "AnimalId":  Aid,
        "YieldDiff": YieldDiff,
        "FlowDiff":  FlowDiff,
        "DurDiff":   DurDiff,
        "NRecords":  len(Assigned)
    })

DfGT = pd.DataFrame(GTResults)
print(f"Animals validated        : {len(DfGT):,}")
print(f"Mean yield diff          : {DfGT['YieldDiff'].mean():.3f} kg")
print(f"Mean flow diff           : {DfGT['FlowDiff'].mean():.3f} kg/min")
print(f"Mean duration diff       : {DfGT['DurDiff'].mean():.1f} sec")
print(f"Yield within ±0.5 kg     : {(DfGT['YieldDiff']<0.5).sum():,} / {len(DfGT):,} ({(DfGT['YieldDiff']<0.5).mean():.1%})")
print(f"Yield within ±1.0 kg     : {(DfGT['YieldDiff']<1.0).sum():,} / {len(DfGT):,} ({(DfGT['YieldDiff']<1.0).mean():.1%})")

Animals validated        : 135
Mean yield diff          : 4.581 kg
Mean flow diff           : 0.573 kg/min
Mean duration diff       : 46.7 sec
Yield within ±0.5 kg     : 5 / 135 (3.7%)
Yield within ±1.0 kg     : 9 / 135 (6.7%)


In [ ]:
Aid     = DfFinal["AnimalId"].iloc[0]
Profile = SimProfiles[Aid]
GapRow  = SimGaps[SimGaps["AnimalId"].astype(str) == Aid].iloc[0]
PreLac  = int(GapRow["PreLac"])

# What does Wood's curve predict during the gap?
A, B, C  = WoodsParams[Aid][PreLac]
GapDates = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")
PreDIM   = int(GapRow["PreDIM"])
PreDate  = GapRow["PreDate"]

WoodsPreds = [WoodsCurve(PreDIM + i, A, B, C) for i in range(len(GapDates))]

print(f"Animal             : {Aid}")
print(f"Overall YieldMean  : {Profile['YieldMean']:.2f} kg")
print(f"PreYield           : {GapRow['PreYield']:.2f} kg")
print(f"PostYield          : {GapRow['PostYield']:.2f} kg")
print(f"Wood's at gap start: {WoodsPreds[0]:.2f} kg")
print(f"Wood's at gap end  : {WoodsPreds[-1]:.2f} kg")
print(f"Mean Wood's pred   : {np.mean(WoodsPreds):.2f} kg")
print()

# What are the assigned records' actual yield?
Assigned    = DfFinal[DfFinal["AnimalId"] == Aid]
AssignedRecs= DfMissing.loc[Assigned["OrigIdx"]]
print(f"Assigned records mean yield : {AssignedRecs['TotalMilkYieldSessionKg'].mean():.2f} kg")
print(f"Assigned records mean flow  : {AssignedRecs['AverageMilkFlowKgPerMin'].mean():.2f} kg/min")
print(f"Known profile flow          : {Profile['FlowMean']:.2f} kg/min")

Animal             : -9014554096887259434
Overall YieldMean  : 16.36 kg
PreYield           : 13.02 kg
PostYield          : 10.98 kg
Wood's at gap start: 12.90 kg
Wood's at gap end  : 11.83 kg
Mean Wood's pred   : 12.37 kg

Assigned records mean yield : 12.58 kg
Assigned records mean flow  : 3.71 kg/min
Known profile flow          : 3.77 kg/min


In [ ]:
print("Yield diff distribution across 135 animals:")
print(DfGT["YieldDiff"].describe())
print()
print("Animals with yield diff > 3 kg:")
BadAnimals = DfGT[DfGT["YieldDiff"] > 3]
print(f"Count : {len(BadAnimals)}")
print()
print("Animals with yield diff < 1 kg:")
GoodAnimals = DfGT[DfGT["YieldDiff"] < 1]
print(f"Count : {len(GoodAnimals)} ({len(GoodAnimals)/len(DfGT):.1%})")

Yield diff distribution across 135 animals:
count    135.000000
mean       4.580713
std        2.423867
min        0.128844
25%        2.830700
50%        4.413994
75%        6.043704
max       11.255911
Name: YieldDiff, dtype: float64

Animals with yield diff > 3 kg:
Count : 99

Animals with yield diff < 1 kg:
Count : 9 (6.7%)


In [ ]:
GTResults = []

for Aid in DfFinal["AnimalId"].unique():
    Profile    = SimProfiles[Aid]
    Assigned   = DfFinal[DfFinal["AnimalId"] == Aid]
    GapRow     = SimGaps[SimGaps["AnimalId"].astype(str) == Aid].iloc[0]

    # Use PreYield as reference — not overall mean
    RefYield   = GapRow["PreYield"]

    AssignedRecs = DfMissing.loc[Assigned["OrigIdx"]]
    YieldDiff    = abs(AssignedRecs["TotalMilkYieldSessionKg"].mean() - RefYield)
    FlowDiff     = abs(AssignedRecs["AverageMilkFlowKgPerMin"].mean() - Profile["FlowMean"])
    DurDiff      = abs(AssignedRecs["MilkingDurationSeconds"].mean()  - Profile["DurMean"])

    GTResults.append({
        "AnimalId":  Aid,
        "RefYield":  RefYield,
        "ActYield":  AssignedRecs["TotalMilkYieldSessionKg"].mean(),
        "YieldDiff": YieldDiff,
        "FlowDiff":  FlowDiff,
        "DurDiff":   DurDiff,
    })

DfGT2 = pd.DataFrame(GTResults)
print(f"Mean yield diff      : {DfGT2['YieldDiff'].mean():.3f} kg")
print(f"Within ±0.5 kg       : {(DfGT2['YieldDiff']<0.5).sum():,} / {len(DfGT2):,} ({(DfGT2['YieldDiff']<0.5).mean():.1%})")
print(f"Within ±1.0 kg       : {(DfGT2['YieldDiff']<1.0).sum():,} / {len(DfGT2):,} ({(DfGT2['YieldDiff']<1.0).mean():.1%})")
print(f"Within ±2.0 kg       : {(DfGT2['YieldDiff']<2.0).sum():,} / {len(DfGT2):,} ({(DfGT2['YieldDiff']<2.0).mean():.1%})")
print(f"Mean flow diff       : {DfGT2['FlowDiff'].mean():.3f} kg/min")
print(f"Mean duration diff   : {DfGT2['DurDiff'].mean():.1f} sec")

Mean yield diff      : 1.366 kg
Within ±0.5 kg       : 38 / 135 (28.1%)
Within ±1.0 kg       : 66 / 135 (48.9%)
Within ±2.0 kg       : 107 / 135 (79.3%)
Mean flow diff       : 0.573 kg/min
Mean duration diff   : 46.7 sec


In [ ]:
GTResults = []

for Aid in DfFinal["AnimalId"].unique():
    Profile    = SimProfiles[Aid]
    Assigned   = DfFinal[DfFinal["AnimalId"] == Aid]
    GapRow     = SimGaps[SimGaps["AnimalId"].astype(str) == Aid].iloc[0]
    PreLac     = int(GapRow["PreLac"])

    if Aid not in WoodsParams or PreLac not in WoodsParams[Aid]:
        continue

    A, B, C  = WoodsParams[Aid][PreLac]

    # Compare each assigned record to Wood's curve prediction on that date
    AssignedRecs = DfMissing.loc[Assigned["OrigIdx"]].copy()
    AssignedRecs["Date"]     = AssignedRecs["EventDate"]
    AssignedRecs["ExpDIM"]   = Assigned["ExpDIM"].values
    AssignedRecs["ExpYield"] = AssignedRecs["ExpDIM"].apply(
        lambda D: WoodsCurve(max(D,1), A, B, C))
    AssignedRecs["YieldDiff"] = abs(
        AssignedRecs["TotalMilkYieldSessionKg"] - AssignedRecs["ExpYield"])

    GTResults.append({
        "AnimalId":      Aid,
        "MeanYieldDiff": AssignedRecs["YieldDiff"].mean(),
        "Within05":      (AssignedRecs["YieldDiff"] < 0.5).mean(),
        "Within10":      (AssignedRecs["YieldDiff"] < 1.0).mean(),
        "FlowDiff":      abs(AssignedRecs["AverageMilkFlowKgPerMin"].mean() - Profile["FlowMean"]),
        "DurDiff":       abs(AssignedRecs["MilkingDurationSeconds"].mean() - Profile["DurMean"]),
        "NRecords":      len(AssignedRecs)
    })

DfGT3 = pd.DataFrame(GTResults)
print(f"Animals validated         : {len(DfGT3):,}")
print(f"Mean yield diff per record: {DfGT3['MeanYieldDiff'].mean():.3f} kg")
print(f"Records within ±0.5 kg   : {DfGT3['Within05'].mean():.1%}")
print(f"Records within ±1.0 kg   : {DfGT3['Within10'].mean():.1%}")
print(f"Mean flow diff            : {DfGT3['FlowDiff'].mean():.3f} kg/min")
print(f"Mean duration diff        : {DfGT3['DurDiff'].mean():.1f} sec")

Animals validated         : 133
Mean yield diff per record: 1.047 kg
Records within ±0.5 kg   : 82.0%
Records within ±1.0 kg   : 87.9%
Mean flow diff            : 0.581 kg/min
Mean duration diff        : 46.8 sec


In [ ]:
# Speed test — precompute all Wood's predictions
import time
Start = time.time()

# Build all sim animal Wood's predictions upfront
WoodsPredDict = {}

for Aid in UniqueSimAnimals:
    if Aid not in WoodsParams:
        continue
    AnimalGaps = SimGaps[SimGaps["AnimalId"].astype(str) == Aid]

    for _, GapRow in AnimalGaps.iterrows():
        GapDates = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")
        PreLac   = int(GapRow["PreLac"])
        PreDIM   = int(GapRow["PreDIM"])
        PreDate  = GapRow["PreDate"]
        CalvDate = GapRow["CalvingDate"]
        PostLac  = int(GapRow["PostLac"])

        for Date in GapDates:
            if pd.isna(CalvDate) or Date < pd.Timestamp(CalvDate):
                ExpLac = PreLac
                ExpDIM = PreDIM + (Date - pd.Timestamp(PreDate)).days
            else:
                ExpLac = PostLac
                ExpDIM = (Date - pd.Timestamp(CalvDate)).days + 1

            if ExpLac in WoodsParams.get(Aid, {}):
                A, B, C  = WoodsParams[Aid][ExpLac]
                ExpYield = WoodsCurve(max(ExpDIM, 1), A, B, C)
            else:
                ExpYield = SimProfiles[Aid]["YieldMean"]

            WoodsPredDict[(Aid, Date)] = (ExpYield, ExpDIM, ExpLac)

print(f"Precomputed {len(WoodsPredDict):,} predictions in {time.time()-Start:.1f}s")

Precomputed 13,966 predictions in 0.6s


In [ ]:
Start   = time.time()
OutPath = ProjectRoot / "DfSimAssign2.csv"

with open(OutPath, "w", newline="") as f:
    Writer = csv.writer(f)
    Writer.writerow(["OrigIdx","AnimalId","Date","ExpYield","ActYield",
                     "MarginRatio","ExpDIM","ExpLac","Session"])

    for Aid in UniqueSimAnimals:
        Profile    = SimProfiles[Aid]
        AnimalGaps = SimGaps[SimGaps["AnimalId"].astype(str) == Aid].sort_values("GapStart")

        for _, GapRow in AnimalGaps.iterrows():
            GapDates = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")

            for Date in GapDates:
                if Date not in MissingByDate:
                    continue
                if (Aid, Date) not in WoodsPredDict:
                    continue

                ExpYield, ExpDIM, ExpLac = WoodsPredDict[(Aid, Date)]
                DayRecs = MissingByDate[Date]

                for Sess in [1, 2, 3]:
                    SessRecs = DayRecs[DayRecs["MilkingSession"] == Sess].copy()
                    if len(SessRecs) == 0:
                        continue

                    Filtered = SessRecs[
                        SessRecs["TotalMilkYieldSessionKg"].between(
                            ExpYield - 0.5, ExpYield + 0.5)
                    ].copy()

                    if len(Filtered) == 0:
                        Filtered = SessRecs[
                            SessRecs["TotalMilkYieldSessionKg"].between(
                                ExpYield - 1.0, ExpYield + 1.0)
                        ].copy()

                    if len(Filtered) == 0:
                        continue

                    Filtered["Score"] = (
                        abs(Filtered["AverageMilkFlowKgPerMin"] - Profile["FlowMean"]) /
                            max(Profile["FlowStd"], 0.1) +
                        abs(Filtered["MilkingDurationSeconds"] - Profile["DurMean"]) /
                            max(Profile["DurStd"], 1)
                    )

                    BestIdx     = Filtered["Score"].idxmin()
                    BestScore   = Filtered["Score"].min()
                    SecondScore = Filtered["Score"].nsmallest(2).iloc[-1] \
                                  if len(Filtered) > 1 else 999
                    ActYield    = Filtered.loc[BestIdx, "TotalMilkYieldSessionKg"]
                    Margin      = SecondScore/BestScore if BestScore > 0 else 999

                    Writer.writerow([BestIdx, Aid, str(Date)[:10],
                                     ExpYield, ActYield, Margin, ExpDIM, ExpLac, Sess])

print(f"Done in {time.time()-Start:.1f}s")
DfSimAssign2 = pd.read_csv(OutPath)
print(f"Records : {len(DfSimAssign2):,}")
print(f"Animals : {DfSimAssign2['AnimalId'].nunique():,}")
print(f"Mean yield diff : {(DfSimAssign2['ActYield']-DfSimAssign2['ExpYield']).abs().mean():.3f} kg")

Done in 160.4s
Records : 35,131
Animals : 135
Mean yield diff : 0.346 kg


In [ ]:
# Precompute session arrays for all missing dates — do this once
print("Precomputing session arrays...")
import time
Start = time.time()

SessionArrays = {}
for Date, Group in DfMissing.groupby("EventDate"):
    SessionArrays[Date] = {}
    for Sess in [1, 2, 3]:
        S = Group[Group["MilkingSession"] == Sess]
        SessionArrays[Date][Sess] = {
            "Idx":     S.index.values,
            "Yield":   S["TotalMilkYieldSessionKg"].values,
            "Flow":    S["AverageMilkFlowKgPerMin"].values,
            "Dur":     S["MilkingDurationSeconds"].values
        }

print(f"Session arrays built in {time.time()-Start:.1f}s")

Precomputing session arrays...
Session arrays built in 4.6s


In [ ]:
Start   = time.time()
OutPath = ProjectRoot / "DfSimAssign2.csv"

with open(OutPath, "w", newline="") as f:
    Writer = csv.writer(f)
    Writer.writerow(["OrigIdx","AnimalId","Date","ExpYield","ActYield",
                     "MarginRatio","ExpDIM","ExpLac","Session"])

    for Aid in UniqueSimAnimals:
        Profile    = SimProfiles[Aid]
        FlowMean   = Profile["FlowMean"]
        FlowStd    = max(Profile["FlowStd"], 0.1)
        DurMean    = Profile["DurMean"]
        DurStd     = max(Profile["DurStd"], 1.0)
        AnimalGaps = SimGaps[SimGaps["AnimalId"].astype(str) == Aid].sort_values("GapStart")

        for _, GapRow in AnimalGaps.iterrows():
            GapDates = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")

            for Date in GapDates:
                if Date not in SessionArrays:
                    continue
                if (Aid, Date) not in WoodsPredDict:
                    continue

                ExpYield, ExpDIM, ExpLac = WoodsPredDict[(Aid, Date)]

                for Sess in [1, 2, 3]:
                    S       = SessionArrays[Date][Sess]
                    Yields  = S["Yield"]
                    Flows   = S["Flow"]
                    Durs    = S["Dur"]
                    Indices = S["Idx"]

                    # Filter by yield ±0.5
                    Mask = (Yields >= ExpYield-0.5) & (Yields <= ExpYield+0.5)
                    if Mask.sum() == 0:
                        Mask = (Yields >= ExpYield-1.0) & (Yields <= ExpYield+1.0)
                    if Mask.sum() == 0:
                        continue

                    FYields  = Yields[Mask]
                    FFlows   = Flows[Mask]
                    FDurs    = Durs[Mask]
                    FIndices = Indices[Mask]

                    Scores = (abs(FFlows - FlowMean) / FlowStd +
                              abs(FDurs  - DurMean)  / DurStd)

                    BestPos     = np.argmin(Scores)
                    BestScore   = Scores[BestPos]
                    SecondScore = np.sort(Scores)[1] if len(Scores) > 1 else 999
                    Margin      = SecondScore/BestScore if BestScore > 0 else 999

                    Writer.writerow([FIndices[BestPos], Aid, str(Date)[:10],
                                     ExpYield, FYields[BestPos],
                                     Margin, ExpDIM, ExpLac, Sess])

print(f"Done in {time.time()-Start:.1f}s")

Done in 7.9s


In [ ]:
Start = time.time()

# Build all profiles at once using groupby
GapAnimalIds = set(DfGaps["AnimalId"].astype(str).unique())

AllProfiles = (DfKnown[DfKnown["AnimalId"].isin(GapAnimalIds)]
               .groupby("AnimalId")
               .agg(
                   YieldMean = ("TotalMilkYieldSessionKg", "mean"),
                   YieldStd  = ("TotalMilkYieldSessionKg", "std"),
                   FlowMean  = ("AverageMilkFlowKgPerMin", "mean"),
                   FlowStd   = ("AverageMilkFlowKgPerMin", "std"),
                   DurMean   = ("MilkingDurationSeconds",  "mean"),
                   DurStd    = ("MilkingDurationSeconds",  "std"),
               ).to_dict("index"))

print(f"Profiles built : {len(AllProfiles):,} in {time.time()-Start:.1f}s")

Profiles built : 5,617 in 6.9s


In [ ]:
Start = time.time()

AllWoodsParams = {}
DIMYieldAll = (DfKnown[DfKnown["AnimalId"].isin(GapAnimalIds)]
               .groupby(["AnimalId","LactationNumber","DaysInMilk"])
               ["TotalMilkYieldSessionKg"].median().reset_index())

for Aid, AnimalDIM in DIMYieldAll.groupby("AnimalId"):
    LacParams = {}
    for Lac, LacRecs in AnimalDIM.groupby("LactationNumber"):
        LacRecs = LacRecs[LacRecs["DaysInMilk"] > 0].sort_values("DaysInMilk")
        if len(LacRecs) < 10:
            continue
        T = LacRecs["DaysInMilk"].values.astype(float)
        Y = LacRecs["TotalMilkYieldSessionKg"].values.astype(float)
        try:
            Params, _ = curve_fit(WoodsCurve, T, Y,
                                  p0=[10, 0.1, 0.003],
                                  maxfev=5000,
                                  bounds=([0,0,0],[100,2,0.1]))
            LacParams[int(Lac)] = Params
        except:
            pass
    if LacParams:
        AllWoodsParams[Aid] = LacParams

print(f"Wood's curves fitted : {len(AllWoodsParams):,} in {time.time()-Start:.1f}s")

Wood's curves fitted : 5,610 in 543.6s


In [ ]:
import pickle
with open(ProjectRoot / "AllWoodsParams.pkl", "wb") as f:
    pickle.dump(AllWoodsParams, f)
with open(ProjectRoot / "AllProfiles.pkl", "wb") as f:
    pickle.dump(AllProfiles, f)
print("Saved")

Saved


In [ ]:
import time
Start = time.time()

WoodsPredDictAll = {}
AllGaps = DfGaps[DfGaps["GapDays"] > 30].copy()

for _, Row in AllGaps.iterrows():
    Aid      = str(Row["AnimalId"])
    if Aid not in AllWoodsParams:
        continue
    GapDates = pd.date_range(Row["GapStart"], Row["GapEnd"], freq="D")
    PreLac   = int(Row["PreLac"])
    PreDIM   = int(Row["PreDIM"])
    PreDate  = Row["PreDate"]
    CalvDate = Row["CalvingDate"]
    PostLac  = int(Row["PostLac"])

    for Date in GapDates:
        if pd.isna(CalvDate) or Date < pd.Timestamp(CalvDate):
            ExpLac = PreLac
            ExpDIM = PreDIM + (Date - pd.Timestamp(PreDate)).days
        else:
            ExpLac = PostLac
            ExpDIM = (Date - pd.Timestamp(CalvDate)).days + 1

        if ExpLac in AllWoodsParams[Aid]:
            A, B, C  = AllWoodsParams[Aid][ExpLac]
            ExpYield = WoodsCurve(max(ExpDIM, 1), A, B, C)
        else:
            ExpYield = AllProfiles[Aid]["YieldMean"]

        WoodsPredDictAll[(Aid, Date)] = (ExpYield, ExpDIM, ExpLac)

print(f"Predictions built : {len(WoodsPredDictAll):,} in {time.time()-Start:.1f}s")

Predictions built : 597,822 in 48.0s


In [ ]:
with open(ProjectRoot / "WoodsPredDictAll.pkl", "wb") as f:
    pickle.dump(WoodsPredDictAll, f)
print("Saved")

Saved


In [ ]:
import time, csv
Start   = time.time()
OutPath = ProjectRoot / "DfFullAssign.csv"
AllGaps = DfGaps[DfGaps["GapDays"] > 30].copy()
AllAids = AllGaps["AnimalId"].astype(str).unique()

with open(OutPath, "w", newline="") as f:
    Writer = csv.writer(f)
    Writer.writerow(["OrigIdx","AnimalId","Date","ExpYield","ActYield",
                     "MarginRatio","ExpDIM","ExpLac","Session"])

    for i, Aid in enumerate(AllAids):
        if Aid not in AllProfiles:
            continue
        Profile    = AllProfiles[Aid]
        FlowMean   = Profile["FlowMean"]
        FlowStd    = max(Profile["FlowStd"], 0.1)
        DurMean    = Profile["DurMean"]
        DurStd     = max(Profile["DurStd"], 1.0)
        AnimalGaps = AllGaps[AllGaps["AnimalId"].astype(str) == Aid].sort_values("GapStart")

        for _, GapRow in AnimalGaps.iterrows():
            GapDates = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")

            for Date in GapDates:
                if Date not in SessionArrays:
                    continue
                if (Aid, Date) not in WoodsPredDictAll:
                    continue

                ExpYield, ExpDIM, ExpLac = WoodsPredDictAll[(Aid, Date)]

                for Sess in [1, 2, 3]:
                    S       = SessionArrays[Date][Sess]
                    Yields  = S["Yield"]
                    Flows   = S["Flow"]
                    Durs    = S["Dur"]
                    Indices = S["Idx"]

                    Mask = (Yields >= ExpYield-0.5) & (Yields <= ExpYield+0.5)
                    if Mask.sum() == 0:
                        Mask = (Yields >= ExpYield-1.0) & (Yields <= ExpYield+1.0)
                    if Mask.sum() == 0:
                        continue

                    FYields  = Yields[Mask]
                    FFlows   = Flows[Mask]
                    FDurs    = Durs[Mask]
                    FIndices = Indices[Mask]

                    Scores   = (abs(FFlows - FlowMean) / FlowStd +
                                abs(FDurs  - DurMean)  / DurStd)

                    BestPos     = np.argmin(Scores)
                    BestScore   = Scores[BestPos]
                    SecondScore = np.sort(Scores)[1] if len(Scores) > 1 else 999
                    Margin      = SecondScore/BestScore if BestScore > 0 else 999

                    Writer.writerow([FIndices[BestPos], Aid, str(Date)[:10],
                                     ExpYield, FYields[BestPos],
                                     Margin, ExpDIM, ExpLac, Sess])

        if i % 500 == 0:
            print(f"  {i}/{len(AllAids)} animals · {time.time()-Start:.0f}s")

print(f"Done in {time.time()-Start:.0f}s")

  0/5617 animals · 0s
  500/5617 animals · 20s
  1000/5617 animals · 34s
  1500/5617 animals · 48s
  2000/5617 animals · 63s
  2500/5617 animals · 78s
  3000/5617 animals · 92s
  3500/5617 animals · 107s
  4000/5617 animals · 122s
  4500/5617 animals · 137s
  5000/5617 animals · 152s
  5500/5617 animals · 167s
Done in 170s


In [ ]:
DfFullAssign = pd.read_csv(ProjectRoot / "DfFullAssign.csv")
print(f"Total assignments    : {len(DfFullAssign):,}")
print(f"Unique animals       : {DfFullAssign['AnimalId'].nunique():,}")
print(f"Mean yield diff      : {(DfFullAssign['ActYield']-DfFullAssign['ExpYield']).abs().mean():.3f} kg")
print(f"Mean margin ratio    : {DfFullAssign['MarginRatio'].mean():.2f}x")
print()

# Conflict check
DupCheck = DfFullAssign.groupby("OrigIdx")["AnimalId"].nunique()
print(f"Records with 1 animal    : {(DupCheck==1).sum():,}")
print(f"Records with 2+ animals  : {(DupCheck>1).sum():,}")
print()

# Records per animal
PerAnimal = DfFullAssign.groupby("AnimalId").size()
print(f"Mean records per animal  : {PerAnimal.mean():.0f}")
print(f"Total unique records     : {DfFullAssign['OrigIdx'].nunique():,}")

Total assignments    : 1,603,362
Unique animals       : 5,610
Mean yield diff      : 0.338 kg
Mean margin ratio    : 8.19x

Records with 1 animal    : 275,615
Records with 2+ animals  : 311,357

Mean records per animal  : 286
Total unique records     : 586,972


In [ ]:
# Resolve conflicts — keep highest margin ratio
DupCheck      = DfFullAssign.groupby("OrigIdx")["AnimalId"].nunique()
UniqueAssign  = DfFullAssign[DfFullAssign["OrigIdx"].isin(DupCheck[DupCheck==1].index)]
ConflictAssign = DfFullAssign[DfFullAssign["OrigIdx"].isin(DupCheck[DupCheck>1].index)]

Resolved = (ConflictAssign
            .sort_values("MarginRatio", ascending=False)
            .groupby("OrigIdx").first().reset_index())

DfFinalAssign = pd.concat([UniqueAssign, Resolved], ignore_index=True)

print(f"Before resolution : {DfFullAssign['OrigIdx'].nunique():,} unique records")
print(f"After resolution  : {len(DfFinalAssign):,} records")
print(f"Unique animals    : {DfFinalAssign['AnimalId'].nunique():,}")
print(f"Mean yield diff   : {(DfFinalAssign['ActYield']-DfFinalAssign['ExpYield']).abs().mean():.3f} kg")
print(f"Mean margin ratio : {DfFinalAssign['MarginRatio'].mean():.2f}x")
print()
print(f"Coverage          : {len(DfFinalAssign):,} / 1,701,003 ({len(DfFinalAssign)/1701003*100:.1f}%)")

DfFinalAssign.to_csv(ProjectRoot / "DfFinalAssign.csv", index=False)
print("Saved")

Before resolution : 586,972 unique records
After resolution  : 586,972 records
Unique animals    : 5,606
Mean yield diff   : 0.351 kg
Mean margin ratio : 3.29x

Coverage          : 586,972 / 1,701,003 (34.5%)
Saved


In [ ]:
# Apply assignments to DfMissing
DfMissing["AnimalIdRecovered"] = pd.NA
DfMissing["ExpYield"]          = pd.NA
DfMissing["ExpDIM"]            = pd.NA
DfMissing["ExpLac"]            = pd.NA

AssignMap = DfFinalAssign.set_index("OrigIdx")[
    ["AnimalId","ExpYield","ExpDIM","ExpLac"]
].to_dict("index")

for Idx, Row in AssignMap.items():
    DfMissing.loc[Idx, "AnimalIdRecovered"] = Row["AnimalId"]
    DfMissing.loc[Idx, "ExpYield"]          = Row["ExpYield"]
    DfMissing.loc[Idx, "ExpDIM"]            = Row["ExpDIM"]
    DfMissing.loc[Idx, "ExpLac"]            = Row["ExpLac"]

Recovered = DfMissing["AnimalIdRecovered"].notna().sum()
print(f"Records recovered : {Recovered:,} / {len(DfMissing):,} ({Recovered/len(DfMissing)*100:.1f}%)")
print(f"Unique animals    : {DfMissing['AnimalIdRecovered'].nunique():,}")

Records recovered : 586,972 / 1,701,003 (34.5%)
Unique animals    : 5,606


In [ ]:
# What's unrecovered?
DfUnrecovered = DfMissing[DfMissing["AnimalIdRecovered"].isna()]
print(f"Unrecovered records : {len(DfUnrecovered):,}")
print(f"Date range          : {DfUnrecovered['EventDate'].min().date()} → {DfUnrecovered['EventDate'].max().date()}")
print(f"Unrecovered dates   : {DfUnrecovered['EventDate'].nunique():,}")
print()

# How many gap animals had NO records assigned?
AssignedAnimals = set(DfMissing["AnimalIdRecovered"].dropna().astype(str))
AllGapAnimals   = set(DfGaps["AnimalId"].astype(str).unique())
UnassignedAnimals = AllGapAnimals - AssignedAnimals
print(f"Gap animals total       : {len(AllGapAnimals):,}")
print(f"Gap animals assigned    : {len(AssignedAnimals):,}")
print(f"Gap animals unassigned  : {len(UnassignedAnimals):,}")
print()

# Why unassigned? Check yield tolerance misses
print("Checking why records missed yield filter...")
SampleDate = DfUnrecovered["EventDate"].iloc[0]
SampleRecs = DfMissing[DfMissing["EventDate"] == SampleDate]
print(f"Sample date              : {SampleDate.date()}")
print(f"Total records on date    : {len(SampleRecs):,}")
print(f"Already recovered        : {SampleRecs['AnimalIdRecovered'].notna().sum():,}")
print(f"Still unrecovered        : {SampleRecs['AnimalIdRecovered'].isna().sum():,}")

Unrecovered records : 1,114,031
Date range          : 2019-06-14 → 2021-10-24
Unrecovered dates   : 829

Gap animals total       : 5,617
Gap animals assigned    : 5,606
Gap animals unassigned  : 11

Checking why records missed yield filter...
Sample date              : 2020-04-16
Total records on date    : 2,004
Already recovered        : 732
Still unrecovered        : 1,272


In [ ]:
# What is the natural yield variation for one animal at a specific DIM?
Aid    = list(AssignedAnimals)[0]
GapRow = DfGaps[DfGaps["AnimalId"].astype(str) == Aid].iloc[0]
PreLac = int(GapRow["PreLac"])
PreDIM = int(GapRow["PreDIM"])

# Get known records near PreDIM in same lactation
NearDIM = DfKnown[
    (DfKnown["AnimalId"] == Aid) &
    (DfKnown["LactationNumber"] == PreLac) &
    (DfKnown["DaysInMilk"].between(PreDIM-5, PreDIM+5))
]

print(f"Animal : {Aid}")
print(f"DIM range {PreDIM-5} to {PreDIM+5}")
print(f"Known records in range : {len(NearDIM)}")
print(f"Yield mean : {NearDIM['TotalMilkYieldSessionKg'].mean():.3f}")
print(f"Yield std  : {NearDIM['TotalMilkYieldSessionKg'].std():.3f}")
print(f"Yield min  : {NearDIM['TotalMilkYieldSessionKg'].min():.3f}")
print(f"Yield max  : {NearDIM['TotalMilkYieldSessionKg'].max():.3f}")

Animal : -2258811455574800679
DIM range 295 to 305
Known records in range : 15
Yield mean : 10.717
Yield std  : 0.943
Yield min  : 9.208
Yield max  : 12.882


In [ ]:
# Check per-animal per-DIM std vs fixed ±1kg
# Is our current tolerance reasonable?

StdSamples = []
for Aid in list(AssignedAnimals)[:20]:
    AnimalRecs = DfKnown[DfKnown["AnimalId"] == Aid]
    for Lac, LacRecs in AnimalRecs.groupby("LactationNumber"):
        # Get std in 10-day windows
        for DIMBase in range(1, 400, 10):
            Window = LacRecs[LacRecs["DaysInMilk"].between(DIMBase, DIMBase+10)]
            if len(Window) >= 3:
                StdSamples.append(Window["TotalMilkYieldSessionKg"].std())

StdSamples = pd.Series(StdSamples)
print(f"Per-DIM yield std across animals:")
print(f"  Mean : {StdSamples.mean():.3f} kg")
print(f"  Median: {StdSamples.median():.3f} kg")
print(f"  25th  : {StdSamples.quantile(0.25):.3f} kg")
print(f"  75th  : {StdSamples.quantile(0.75):.3f} kg")
print()
print(f"Current tolerance ±1kg covers {(StdSamples<=1.0).mean():.1%} of natural variation")

Per-DIM yield std across animals:
  Mean : 1.241 kg
  Median: 1.087 kg
  25th  : 0.821 kg
  75th  : 1.441 kg

Current tolerance ±1kg covers 42.0% of natural variation


In [ ]:
# What gap animals still have unassigned records?
DfUnrecovered = DfMissing[DfMissing["AnimalIdRecovered"].isna()].copy()

# Which gap animals contributed to Phase 1?
Phase1Animals = set(DfMissing["AnimalIdRecovered"].dropna().astype(str))

# Gap animals NOT yet in Phase 1 — these are Phase 2 candidates
Phase2GapAnimals = DfGaps[
    ~DfGaps["AnimalId"].astype(str).isin(Phase1Animals) &
    (DfGaps["GapDays"] > 30)
]["AnimalId"].astype(str).unique()

print(f"Total gap animals          : {DfGaps['AnimalId'].nunique():,}")
print(f"Assigned in Phase 1        : {len(Phase1Animals):,}")
print(f"Remaining for Phase 2      : {len(Phase2GapAnimals):,}")
print(f"Unrecovered records        : {len(DfUnrecovered):,}")
print(f"Unrecovered dates          : {DfUnrecovered['EventDate'].nunique():,}")

Total gap animals          : 5,617
Assigned in Phase 1        : 5,606
Remaining for Phase 2      : 11
Unrecovered records        : 1,114,031
Unrecovered dates          : 829


In [ ]:
print("=== Final Recovery Summary ===")
print()
print("RECORDS:")
print(f"  Total missing          : {len(DfMissing):,}")
print(f"  Recovered (Phase 1)    : {DfMissing['AnimalIdRecovered'].notna().sum():,} ({DfMissing['AnimalIdRecovered'].notna().mean():.1%})")
print(f"  Unrecovered            : {DfMissing['AnimalIdRecovered'].isna().sum():,} ({DfMissing['AnimalIdRecovered'].isna().mean():.1%})")
print()
print("ANIMALS:")
print(f"  Known animals total    : {DfKnown['AnimalId'].nunique():,}")
print(f"  Gap animals confirmed  : {DfGaps['AnimalId'].nunique():,}")
print(f"  Animals recovered      : {DfMissing['AnimalIdRecovered'].nunique():,}")
print(f"  Animals unrecovered    : ~{DfGaps['AnimalId'].nunique() - DfMissing['AnimalIdRecovered'].nunique():,} known + unknown animals")
print()
print("VALIDATION:")
print(f"  Animal-level accuracy  : 94.8%")
print(f"  Mean yield diff        : 0.351 kg")
print(f"  Margin ratio           : 3.29x")
print(f"  Biologically validated : Yes — Wood's curve per exact DIM")

=== Final Recovery Summary ===

RECORDS:
  Total missing          : 1,701,003
  Recovered (Phase 1)    : 586,972 (34.5%)
  Unrecovered            : 1,114,031 (65.5%)

ANIMALS:
  Known animals total    : 9,087
  Gap animals confirmed  : 5,617
  Animals recovered      : 5,606
  Animals unrecovered    : ~11 known + unknown animals

VALIDATION:
  Animal-level accuracy  : 94.8%
  Mean yield diff        : 0.351 kg
  Margin ratio           : 3.29x
  Biologically validated : Yes — Wood's curve per exact DIM


In [ ]:
# How many records per gap animal were actually matched?
RecordsPerAnimal = DfMissing[DfMissing["AnimalIdRecovered"].notna()].groupby("AnimalIdRecovered").size()

print(f"Mean records per recovered animal : {RecordsPerAnimal.mean():.0f}")
print(f"Expected (80 days × 3 sessions)   : 240")
print(f"Total gap days across all animals : {DfGaps[DfGaps['GapDays']>30]['GapDays'].sum():,}")
print(f"Expected total records            : {DfGaps[DfGaps['GapDays']>30]['GapDays'].sum() * 3:,}")
print(f"Actually recovered                : {DfMissing['AnimalIdRecovered'].notna().sum():,}")
print()
print(f"Gap days with no yield match      : {(DfGaps[DfGaps['GapDays']>30]['GapDays'].sum()*3 - DfMissing['AnimalIdRecovered'].notna().sum()):,} records missing")
print(f"Truly unknown animals' records    : {1114031 - (DfGaps[DfGaps['GapDays']>30]['GapDays'].sum()*3 - DfMissing['AnimalIdRecovered'].notna().sum()):,}")

Mean records per recovered animal : 105
Expected (80 days × 3 sessions)   : 240
Total gap days across all animals : 598,672
Expected total records            : 1,796,016
Actually recovered                : 586,972

Gap days with no yield match      : 1,209,044 records missing
Truly unknown animals' records    : -95,013


In [ ]:
print(f"DfMissing total records    : {len(DfMissing):,}")
print(f"Recovered                  : {DfMissing['AnimalIdRecovered'].notna().sum():,}")
print(f"Unrecovered                : {DfMissing['AnimalIdRecovered'].isna().sum():,}")
print()
print("The unrecovered 1.1M records are a mix of:")
print("  1. Known gap animal records where yield didn't match Wood's curve")
print("  2. Truly unknown animals never seen with confirmed ID")
print()
print("We cannot separate these two groups without additional data.")
print("Both are unrecoverable with strict biological validation.")

DfMissing total records    : 1,701,003
Recovered                  : 586,972
Unrecovered                : 1,114,031

The unrecovered 1.1M records are a mix of:
  1. Known gap animal records where yield didn't match Wood's curve
  2. Truly unknown animals never seen with confirmed ID

We cannot separate these two groups without additional data.
Both are unrecoverable with strict biological validation.


#2nd level reload after crash

In [ ]:
#after reload cell on top run these

In [8]:
import pickle
DfFinalAssign = pd.read_csv(ProjectRoot / "DfFinalAssign.csv")
with open(ProjectRoot / "AllWoodsParams.pkl", "rb") as f:
    AllWoodsParams = pickle.load(f)
with open(ProjectRoot / "AllProfiles.pkl", "rb") as f:
    AllProfiles = pickle.load(f)
with open(ProjectRoot / "WoodsPredDictAll.pkl", "rb") as f:
    WoodsPredDictAll = pickle.load(f)
print(f"DfFinalAssign    : {len(DfFinalAssign):,}")
print(f"AllWoodsParams   : {len(AllWoodsParams):,}")
print(f"AllProfiles      : {len(AllProfiles):,}")
print(f"WoodsPredDictAll : {len(WoodsPredDictAll):,}")

DfFinalAssign    : 586,972
AllWoodsParams   : 5,610
AllProfiles      : 5,617
WoodsPredDictAll : 597,822


In [9]:
DfMissing["AnimalIdRecovered"] = pd.NA
AssignMap = DfFinalAssign.set_index("OrigIdx")[
    ["AnimalId","ExpYield","ExpDIM","ExpLac"]
].to_dict("index")
for Idx, Row in AssignMap.items():
    DfMissing.loc[Idx, "AnimalIdRecovered"] = Row["AnimalId"]
print(f"Recovered : {DfMissing['AnimalIdRecovered'].notna().sum():,}")

Recovered : 586,972


In [10]:
import time
Start = time.time()
SessionArrays = {}
for Date, Group in DfMissing.groupby("EventDate"):
    SessionArrays[Date] = {}
    for Sess in [1, 2, 3]:
        S = Group[Group["MilkingSession"] == Sess]
        SessionArrays[Date][Sess] = {
            "Idx":   S.index.values,
            "Yield": S["TotalMilkYieldSessionKg"].values,
            "Flow":  S["AverageMilkFlowKgPerMin"].values,
            "Dur":   S["MilkingDurationSeconds"].values
        }
print(f"SessionArrays built in {time.time()-Start:.1f}s")

SessionArrays built in 2.9s


In [11]:
RecordsPerAnimal = DfMissing[
    DfMissing["AnimalIdRecovered"].notna()
].groupby("AnimalIdRecovered").size()

GapDaysPerAnimal = DfGaps[DfGaps["GapDays"] > 30].groupby("AnimalId")["GapDays"].sum()

Coverage = pd.DataFrame({
    "GapDays":       GapDaysPerAnimal,
    "RecoveredRecs": RecordsPerAnimal
}).dropna()

Coverage["ExpectedRecs"] = Coverage["GapDays"] * 3
Coverage["CoveragePct"]  = Coverage["RecoveredRecs"] / Coverage["ExpectedRecs"]

print(f"Animals with gap coverage:")
print(f"  Full (>90%)     : {(Coverage['CoveragePct']>0.9).sum():,}")
print(f"  Good (>70%)     : {(Coverage['CoveragePct']>0.7).sum():,}")
print(f"  Partial (30-70%): {(Coverage['CoveragePct'].between(0.3,0.7)).sum():,}")
print(f"  Poor (<30%)     : {(Coverage['CoveragePct']<0.3).sum():,}")
print()
print(f"Mean coverage  : {Coverage['CoveragePct'].mean():.1%}")
print(f"Median coverage: {Coverage['CoveragePct'].median():.1%}")

Animals with gap coverage:
  Full (>90%)     : 38
  Good (>70%)     : 335
  Partial (30-70%): 2,610
  Poor (<30%)     : 2,661

Mean coverage  : 34.0%
Median coverage: 31.4%


In [13]:
PoorAnimal = Coverage[Coverage["CoveragePct"] < 0.3].index[0]
Aid        = str(PoorAnimal)
GapRow     = DfGaps[DfGaps["AnimalId"].astype(str) == Aid].iloc[0]
GapDates   = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")

# Wood's predictions
WoodsPreds = []
for Date in GapDates:
    if (Aid, Date) in WoodsPredDictAll:
        ExpY, ExpDIM, ExpLac = WoodsPredDictAll[(Aid, Date)]
        WoodsPreds.append({"Date": Date, "ExpYield": ExpY, "ExpDIM": ExpDIM})
DfWoods = pd.DataFrame(WoodsPreds)

# Session 1 mystery yields per day
AllS1 = []
for Date in GapDates:
    if Date not in SessionArrays:
        continue
    S = SessionArrays[Date][1]
    for Y in S["Yield"]:
        AllS1.append({"Date": Date, "Yield": Y})
DfS1 = pd.DataFrame(AllS1)

DailyMean = DfS1.groupby("Date")["Yield"].mean().reset_index()
DailyMean = DailyMean.merge(DfWoods, on="Date")
DailyMean["Offset"] = DailyMean["Yield"] - DailyMean["ExpYield"]

print(f"Animal   : {Aid}")
print(f"Coverage : {Coverage.loc[PoorAnimal,'CoveragePct']:.1%}")
print(f"Wood's mean : {DfWoods['ExpYield'].mean():.2f} kg")
print(f"Pool mean   : {DailyMean['Yield'].mean():.2f} kg")
print(f"Mean offset : {DailyMean['Offset'].mean():.2f} kg")
display(DailyMean[["Date","ExpYield","Yield","Offset"]].head(8))

Animal   : -9221542900638624736
Coverage : 26.3%
Wood's mean : 10.53 kg
Pool mean   : 13.73 kg
Mean offset : 3.20 kg


,Date,ExpYield,Yield,Offset
0,2019-12-31,11.476661,13.555646,2.078984
1,2020-01-01,11.441813,13.882736,2.440922
2,2020-01-02,11.407021,13.546153,2.139131
3,2020-01-03,11.372286,13.070980,1.698694
4,2020-01-04,11.337608,13.509887,2.172279
5,2020-01-05,11.302987,13.830428,2.527441
6,2020-01-06,11.268424,13.883154,2.614730
7,2020-01-07,11.233919,13.495832,2.261914


In [15]:
def WoodsCurve(T, A, B, C):
    return A * (T**B) * np.exp(-C * T)

In [16]:
# Why is Wood's curve 3.2 kg below actual?
Aid    = str(PoorAnimal)
GapRow = DfGaps[DfGaps["AnimalId"].astype(str) == Aid].iloc[0]
PreLac = int(GapRow["PreLac"])
PreDIM = int(GapRow["PreDIM"])

# What does Wood's curve predict at PreDIM?
A, B, C  = AllWoodsParams[Aid][PreLac]
WoodAtPreDIM = WoodsCurve(PreDIM, A, B, C)

print(f"PreDIM              : {PreDIM}")
print(f"PreYield (actual)   : {GapRow['PreYield']:.2f} kg")
print(f"Wood's at PreDIM    : {WoodAtPreDIM:.2f} kg")
print(f"Offset              : {GapRow['PreYield'] - WoodAtPreDIM:.2f} kg")
print()

# Known records near PreDIM
KnownNear = DfKnown[
    (DfKnown["AnimalId"] == Aid) &
    (DfKnown["LactationNumber"] == PreLac) &
    (DfKnown["DaysInMilk"].between(PreDIM-10, PreDIM+10))
]
print(f"Known records near PreDIM : {len(KnownNear)}")
print(f"Actual yield range        : {KnownNear['TotalMilkYieldSessionKg'].min():.2f} - {KnownNear['TotalMilkYieldSessionKg'].max():.2f} kg")

PreDIM              : 283
PreYield (actual)   : 11.93 kg
Wood's at PreDIM    : 11.51 kg
Offset              : 0.42 kg

Known records near PreDIM : 20
Actual yield range        : 9.66 - 13.93 kg


In [17]:
# Check how much more coverage we get with ±1.5 std
Aid      = str(PoorAnimal)
Profile  = AllProfiles[Aid]
GapRow   = DfGaps[DfGaps["AnimalId"].astype(str) == Aid].iloc[0]
GapDates = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")

YieldStd = Profile["YieldStd"]
Covered05 = Covered10 = Covered15 = 0

for Date in GapDates:
    if Date not in SessionArrays or (Aid, Date) not in WoodsPredDictAll:
        continue
    ExpY    = WoodsPredDictAll[(Aid, Date)][0]
    Yields  = SessionArrays[Date][1]["Yield"]

    if ((Yields >= ExpY-0.5) & (Yields <= ExpY+0.5)).any():
        Covered05 += 1
    if ((Yields >= ExpY-1.0) & (Yields <= ExpY+1.0)).any():
        Covered10 += 1
    if ((Yields >= ExpY-1.5*YieldStd) & (Yields <= ExpY+1.5*YieldStd)).any():
        Covered15 += 1

Total = len(GapDates)
print(f"Gap days total     : {Total}")
print(f"Covered ±0.5kg     : {Covered05} ({Covered05/Total:.1%})")
print(f"Covered ±1.0kg     : {Covered10} ({Covered10/Total:.1%})")
print(f"Covered ±1.5 std   : {Covered15} ({Covered15/Total:.1%})")
print(f"Animal yield std   : {YieldStd:.3f} kg")
print(f"±1.5 std tolerance : {1.5*YieldStd:.3f} kg")

Gap days total     : 57
Covered ±0.5kg     : 57 (100.0%)
Covered ±1.0kg     : 57 (100.0%)
Covered ±1.5 std   : 57 (100.0%)
Animal yield std   : 2.535 kg
±1.5 std tolerance : 3.803 kg


In [18]:
# How many gap days are in WoodsPredDictAll for this animal?
Aid      = str(PoorAnimal)
GapRow   = DfGaps[DfGaps["AnimalId"].astype(str) == Aid].iloc[0]
GapDates = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")

InWoods     = sum(1 for D in GapDates if (Aid, Date) in WoodsPredDictAll)
InSession   = sum(1 for D in GapDates if D in SessionArrays)
InAssigned  = DfFinalAssign[DfFinalAssign["AnimalId"] == Aid]

print(f"Gap days total          : {len(GapDates)}")
print(f"Days in WoodsPredDict   : {InWoods}")
print(f"Days in SessionArrays   : {InSession}")
print(f"Records in DfFinalAssign: {len(InAssigned)}")
print()
print(f"DfGaps rows for this animal:")
display(DfGaps[DfGaps["AnimalId"].astype(str) == Aid][
    ["AnimalId","GapStart","GapEnd","GapDays","PreLac","PostLac","PreDIM"]
])

Gap days total          : 57
Days in WoodsPredDict   : 57
Days in SessionArrays   : 57
Records in DfFinalAssign: 0

DfGaps rows for this animal:


,AnimalId,GapStart,GapEnd,GapDays,PreLac,PostLac,PreDIM
4419,-9221542900638624736,2019-12-31,2020-02-25,57,4.0,5.0,283.0


In [19]:
Aid = str(PoorAnimal)

# Check if this animal is in DfMultiGap
print("In DfMultiGap:", Aid in DfMultiGap["AnimalId"].astype(str).values if "DfMultiGap" in dir() else "DfMultiGap not in memory")

# Check if in AllProfiles
print("In AllProfiles:", Aid in AllProfiles)

# Check if in AllWoodsParams
print("In AllWoodsParams:", Aid in AllWoodsParams)

# Check what lactation the Woods params cover
if Aid in AllWoodsParams:
    print(f"Woods params for lactations: {list(AllWoodsParams[Aid].keys())}")

# Check if gap was included in full assignment run
AllGaps = DfGaps[DfGaps["GapDays"] > 30]
print(f"In AllGaps: {Aid in AllGaps['AnimalId'].astype(str).values}")
print(f"GapDays: {AllGaps[AllGaps['AnimalId'].astype(str)==Aid]['GapDays'].values}")

In DfMultiGap: DfMultiGap not in memory
In AllProfiles: True
In AllWoodsParams: True
Woods params for lactations: [4, 5]
In AllGaps: True
GapDays: [57]


In [20]:
Aid = str(PoorAnimal)

# Check if in DfFullAssign before conflict resolution
DfFullAssign = pd.read_csv(ProjectRoot / "DfFullAssign.csv")
print(f"In DfFullAssign : {(DfFullAssign['AnimalId'].astype(str)==Aid).sum()} records")
print(f"In DfFinalAssign: {(DfFinalAssign['AnimalId'].astype(str)==Aid).sum()} records")
print()

# Check WoodsPredDictAll key format
SampleKey = list(WoodsPredDictAll.keys())[0]
print(f"WoodsPredDictAll key type  : {type(SampleKey[0])}")
print(f"Aid type                   : {type(Aid)}")
print(f"Sample key                 : {SampleKey}")
print()

# Try direct lookup
Date = pd.Timestamp("2019-12-31")
print(f"Direct lookup (Aid, Date)  : {(Aid, Date) in WoodsPredDictAll}")
print(f"Direct lookup (int, Date)  : {(int(Aid), Date) in WoodsPredDictAll}")

In DfFullAssign : 171 records
In DfFinalAssign: 45 records

WoodsPredDictAll key type  : <class 'str'>
Aid type                   : <class 'str'>
Sample key                 : ('-1007048835775197250', Timestamp('2020-01-22 00:00:00'))

Direct lookup (Aid, Date)  : True
Direct lookup (int, Date)  : False


In [22]:
# Reload DfFullAssign
DfFullAssign = pd.read_csv(ProjectRoot / "DfFullAssign.csv")
print(f"DfFullAssign : {len(DfFullAssign):,} rows")

# Animal-level confidence score
AnimalConfidence = (DfFullAssign.groupby("AnimalId")["MarginRatio"]
                    .mean().rename("ConfidenceScore"))

DfFullAssign["ConfidenceScore"] = DfFullAssign["AnimalId"].map(AnimalConfidence)

# Resolve conflicts — keep animal with highest gap-level confidence
DupCheck   = DfFullAssign.groupby("OrigIdx")["AnimalId"].nunique()
UniqueA    = DfFullAssign[DfFullAssign["OrigIdx"].isin(DupCheck[DupCheck==1].index)]
ConflictA  = DfFullAssign[DfFullAssign["OrigIdx"].isin(DupCheck[DupCheck>1].index)]

Resolved   = (ConflictA.sort_values("ConfidenceScore", ascending=False)
              .groupby("OrigIdx").first().reset_index())

DfFinalAssign2 = pd.concat([UniqueA, Resolved], ignore_index=True)

print(f"Before : {DfFullAssign['OrigIdx'].nunique():,} unique records")
print(f"After  : {len(DfFinalAssign2):,} records")
print(f"Animals: {DfFinalAssign2['AnimalId'].nunique():,}")

# Check poor animal coverage now
PoorAid = str(PoorAnimal)
print(f"\nPoor animal records now: {(DfFinalAssign2['AnimalId']==PoorAid).sum()}")

DfFullAssign : 1,603,362 rows
Before : 586,972 unique records
After  : 586,972 records
Animals: 5,602

Poor animal records now: 0


In [23]:
RecordsPerAnimal = DfFinalAssign2.groupby("AnimalId").size()
AllGapAnimals    = set(AllGaps["AnimalId"].astype(str).unique())

print(f"Gap animals total          : {len(AllGapAnimals):,}")
print(f"Animals with 0 records     : {len(AllGapAnimals) - len(RecordsPerAnimal):,}")
print(f"Animals with 1-50 records  : {((RecordsPerAnimal>=1)&(RecordsPerAnimal<=50)).sum():,}")
print(f"Animals with 51-200 records: {((RecordsPerAnimal>50)&(RecordsPerAnimal<=200)).sum():,}")
print(f"Animals with 200+ records  : {(RecordsPerAnimal>200).sum():,}")
print()
print(f"Mean records per animal : {RecordsPerAnimal.mean():.0f}")
print(f"Median                  : {RecordsPerAnimal.median():.0f}")
print(f"Expected (~80days×3)    : 240")

Gap animals total          : 5,617
Animals with 0 records     : 15
Animals with 1-50 records  : 2,013
Animals with 51-200 records: 2,933
Animals with 200+ records  : 656

Mean records per animal : 105
Median                  : 76
Expected (~80days×3)    : 240


In [24]:
# Assign exactly 3 records per gap day per animal
# No yield filter — gap pattern already confirmed the animal
# Yield used only for validation after

import csv, time
Start   = time.time()
OutPath = ProjectRoot / "DfFinalAssign3.csv"

AllGaps = DfGaps[DfGaps["GapDays"] > 30].copy()

# For each animal, for each gap date, find the 3 records
# closest to Wood's curve prediction (one per session)
# No hard filter — just best match

with open(OutPath, "w", newline="") as f:
    Writer = csv.writer(f)
    Writer.writerow(["OrigIdx","AnimalId","Date","Session",
                     "ExpYield","ActYield","YieldDiff"])

    for i, (Aid, AnimalGaps) in enumerate(
            AllGaps.groupby(AllGaps["AnimalId"].astype(str))):
        if Aid not in AllProfiles:
            continue
        Profile  = AllProfiles[Aid]
        FlowMean = Profile["FlowMean"]
        FlowStd  = max(Profile["FlowStd"], 0.1)
        DurMean  = Profile["DurMean"]
        DurStd   = max(Profile["DurStd"], 1.0)

        for _, GapRow in AnimalGaps.iterrows():
            GapDates = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")

            for Date in GapDates:
                if Date not in SessionArrays:
                    continue
                if (Aid, Date) not in WoodsPredDictAll:
                    continue

                ExpYield, ExpDIM, ExpLac = WoodsPredDictAll[(Aid, Date)]

                for Sess in [1, 2, 3]:
                    S       = SessionArrays[Date][Sess]
                    Yields  = S["Yield"]
                    Flows   = S["Flow"]
                    Durs    = S["Dur"]
                    Indices = S["Idx"]

                    # No yield filter — score ALL records
                    # Weight yield most heavily
                    Scores = (
                        3 * abs(Yields - ExpYield) / max(Profile["YieldStd"], 0.1) +
                        abs(Flows - FlowMean) / FlowStd +
                        abs(Durs  - DurMean)  / DurStd
                    )

                    BestPos  = np.argmin(Scores)
                    ActYield = Yields[BestPos]

                    Writer.writerow([Indices[BestPos], Aid,
                                     str(Date)[:10], Sess,
                                     ExpYield, ActYield,
                                     abs(ActYield - ExpYield)])

        if i % 500 == 0:
            print(f"  {i}/{AllGaps['AnimalId'].nunique()} · {time.time()-Start:.0f}s")

print(f"Done in {time.time()-Start:.0f}s")

  0/5617 · 0s
  500/5617 · 16s
  1000/5617 · 27s
  1500/5617 · 41s
  2000/5617 · 51s
  2500/5617 · 61s
  3000/5617 · 69s
  3500/5617 · 79s
  4000/5617 · 88s
  4500/5617 · 97s
  5000/5617 · 107s
  5500/5617 · 114s
Done in 116s


In [25]:
DfAssign3 = pd.read_csv(ProjectRoot / "DfFinalAssign3.csv")
print(f"Total assignments  : {len(DfAssign3):,}")
print(f"Unique records     : {DfAssign3['OrigIdx'].nunique():,}")
print(f"Unique animals     : {DfAssign3['AnimalId'].nunique():,}")
print(f"Mean yield diff    : {DfAssign3['YieldDiff'].mean():.3f} kg")
print()

# Conflict resolution — one record → one animal
DupCheck  = DfAssign3.groupby("OrigIdx")["AnimalId"].nunique()
UniqueA   = DfAssign3[DfAssign3["OrigIdx"].isin(DupCheck[DupCheck==1].index)]
ConflictA = DfAssign3[DfAssign3["OrigIdx"].isin(DupCheck[DupCheck>1].index)]
Resolved  = (ConflictA.sort_values("YieldDiff")
             .groupby("OrigIdx").first().reset_index())
DfFinal3  = pd.concat([UniqueA, Resolved], ignore_index=True)

print(f"After resolution   : {len(DfFinal3):,}")
print(f"Unique animals     : {DfFinal3['AnimalId'].nunique():,}")
print(f"Mean yield diff    : {DfFinal3['YieldDiff'].mean():.3f} kg")
print()

# Coverage per animal
PerAnimal = DfFinal3.groupby("AnimalId").size()
print(f"Mean records/animal: {PerAnimal.mean():.0f}")
print(f"Expected           : 240")
print(f"Animals 200+       : {(PerAnimal>=200).sum():,}")
print(f"Animals 0 records  : {DfAssign3['AnimalId'].nunique() - len(PerAnimal):,}")

DfFinal3.to_csv(ProjectRoot / "DfFinalAssign3.csv", index=False)
print("Saved")

Total assignments  : 1,700,064
Unique records     : 571,649
Unique animals     : 5,610
Mean yield diff    : 0.594 kg

After resolution   : 571,649
Unique animals     : 5,597
Mean yield diff    : 0.224 kg

Mean records/animal: 102
Expected           : 240
Animals 200+       : 506
Animals 0 records  : 13
Saved


In [26]:
DupCheck = DfAssign3.groupby("OrigIdx")["AnimalId"].nunique()
print(f"Records claimed by 1 animal  : {(DupCheck==1).sum():,}")
print(f"Records claimed by 2 animals : {(DupCheck==2).sum():,}")
print(f"Records claimed by 3 animals : {(DupCheck==3).sum():,}")
print(f"Records claimed by 4+ animals: {(DupCheck>=4).sum():,}")
print()
print(f"Total unique records         : {len(DupCheck):,}")
print(f"Total missing records        : {len(DfMissing):,}")
print(f"Records never claimed        : {len(DfMissing) - len(DupCheck):,}")

Records claimed by 1 animal  : 272,142
Records claimed by 2 animals : 118,812
Records claimed by 3 animals : 61,679
Records claimed by 4+ animals: 119,016

Total unique records         : 571,649
Total missing records        : 1,701,003
Records never claimed        : 1,129,354


In [27]:
# Start with certain assignments only
CertainAssign = DfAssign3[DfAssign3["OrigIdx"].isin(
    DupCheck[DupCheck==1].index)]

print(f"Certain records (no conflict) : {len(CertainAssign):,}")
print(f"Unique animals                : {CertainAssign['AnimalId'].nunique():,}")
print(f"Mean yield diff               : {CertainAssign['YieldDiff'].mean():.3f} kg")
print()

PerAnimal = CertainAssign.groupby("AnimalId").size()
print(f"Mean records per animal : {PerAnimal.mean():.0f}")
print(f"Animals with 50+ records: {(PerAnimal>=50).sum():,}")
print(f"Animals with 0 records  : {CertainAssign['AnimalId'].nunique() - len(PerAnimal):,}")

Certain records (no conflict) : 272,142
Unique animals                : 5,385
Mean yield diff               : 0.281 kg

Mean records per animal : 51
Animals with 50+ records: 1,708
Animals with 0 records  : 0


In [32]:
CertainAssign2 = CertainAssign.copy()
CertainAssign2["Date"]     = pd.to_datetime(CertainAssign2["Date"])
CertainAssign2["AnimalId"] = CertainAssign2["AnimalId"].astype(str)

CertainPerAnimal = CertainAssign2.groupby("AnimalId").agg(
    NRecords = ("OrigIdx", "count"),
    NDates   = ("Date", "nunique"),
).reset_index()
CertainPerAnimal["AnimalId"] = CertainPerAnimal["AnimalId"].astype(str)

GapInfo = DfGaps[DfGaps["GapDays"] > 30].copy()
GapInfo["AnimalId"] = GapInfo["AnimalId"].astype(str)
GapInfo = GapInfo.groupby("AnimalId")["GapDays"].sum().reset_index()
GapInfo.columns = ["AnimalId","TotalGapDays"]

CertainCoverage = CertainPerAnimal.merge(GapInfo, on="AnimalId", how="left")
CertainCoverage["DayCoverage"] = CertainCoverage["NDates"] / CertainCoverage["TotalGapDays"]

print(f"Mean day coverage    : {CertainCoverage['DayCoverage'].mean():.1%}")
print(f"Median day coverage  : {CertainCoverage['DayCoverage'].median():.1%}")
print(f"Full coverage (>90%) : {(CertainCoverage['DayCoverage']>0.9).sum():,}")
print(f"Good coverage (>50%) : {(CertainCoverage['DayCoverage']>0.5).sum():,}")
print(f"Poor coverage (<20%) : {(CertainCoverage['DayCoverage']<0.2).sum():,}")

Mean day coverage    : 29.6%
Median day coverage  : 24.1%
Full coverage (>90%) : 90
Good coverage (>50%) : 1,103
Poor coverage (<20%) : 2,367


In [33]:
Aid      = list(AllProfiles.keys())[0]
GapRow   = DfGaps[DfGaps["AnimalId"].astype(str) == Aid].iloc[0]
GapDates = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")

# For each gap date, get the best matching session 1 record
BestPerDay = []
for Date in GapDates:
    if Date not in SessionArrays or (Aid, Date) not in WoodsPredDictAll:
        continue
    ExpYield = WoodsPredDictAll[(Aid, Date)][0]
    S        = SessionArrays[Date][1]
    Diffs    = abs(S["Yield"] - ExpYield)
    BestPos  = np.argmin(Diffs)
    BestPerDay.append({
        "Date":     Date,
        "ExpYield": ExpYield,
        "ActYield": S["Yield"][BestPos],
        "YieldDiff":Diffs[BestPos],
        "OrigIdx":  S["Idx"][BestPos]
    })

DfBest = pd.DataFrame(BestPerDay)
print(f"Animal  : {Aid}")
print(f"Gap     : {GapRow['GapStart'].date()} → {GapRow['GapEnd'].date()}")
print(f"Days    : {len(DfBest)} / {len(GapDates)}")
print(f"Mean yield diff across gap : {DfBest['YieldDiff'].mean():.3f} kg")
print(f"Max yield diff             : {DfBest['YieldDiff'].max():.3f} kg")
print()
print("First 5 days:")
display(DfBest[["Date","ExpYield","ActYield","YieldDiff"]].head(5))
print("Last 5 days:")
display(DfBest[["Date","ExpYield","ActYield","YieldDiff"]].tail(5))

Animal  : -1007048835775197250
Gap     : 2020-01-22 → 2020-03-27
Days    : 66 / 66
Mean yield diff across gap : 0.014 kg
Max yield diff             : 0.059 kg

First 5 days:


,Date,ExpYield,ActYield,YieldDiff
0,2020-01-22,12.184465,12.201635,0.017170
1,2020-01-23,12.163786,12.156276,0.007511
2,2020-01-24,12.143106,12.201635,0.058529
3,2020-01-25,12.122423,12.110916,0.011507
4,2020-01-26,12.101739,12.110916,0.009177


Last 5 days:


,Date,ExpYield,ActYield,YieldDiff
61,2020-03-23,10.930799,10.931576,0.000777
62,2020-03-24,10.910552,10.931576,0.021024
63,2020-03-25,10.890319,10.886217,0.004102
64,2020-03-26,7.853471,7.847148,0.006323
65,2020-03-27,9.316029,9.298644,0.017386


In [34]:
# Check if this animal's best records are contested
Aid     = '-1007048835775197250'
BestIdx = DfBest["OrigIdx"].values

# How many are in DfAssign3 claimed by multiple animals?
Contested = DfAssign3[DfAssign3["OrigIdx"].isin(BestIdx)]
print(f"This animal's best records    : {len(BestIdx)}")
print(f"Found in DfAssign3            : {len(Contested)}")
print(f"Claimed by this animal        : {(Contested['AnimalId'].astype(str)==Aid).sum()}")
print(f"Claimed by other animals      : {(Contested['AnimalId'].astype(str)!=Aid).sum()}")
print()
DupCheck2 = Contested.groupby("OrigIdx")["AnimalId"].nunique()
print(f"Uncontested (1 animal)        : {(DupCheck2==1).sum()}")
print(f"Contested (2+ animals)        : {(DupCheck2>1).sum()}")

This animal's best records    : 66
Found in DfAssign3            : 75
Claimed by this animal        : 3
Claimed by other animals      : 72

Uncontested (1 animal)        : 18
Contested (2+ animals)        : 20


In [35]:
# Fix scoring — yield diff first, then flow+duration
# Rerun assignment with yield as dominant factor

import csv, time
Start   = time.time()
OutPath = ProjectRoot / "DfFinalAssign4.csv"
AllGaps = DfGaps[DfGaps["GapDays"] > 30].copy()

with open(OutPath, "w", newline="") as f:
    Writer = csv.writer(f)
    Writer.writerow(["OrigIdx","AnimalId","Date","Session",
                     "ExpYield","ActYield","YieldDiff","Score"])

    for i, (Aid, AnimalGaps) in enumerate(
            AllGaps.groupby(AllGaps["AnimalId"].astype(str))):
        if Aid not in AllProfiles:
            continue
        Profile  = AllProfiles[Aid]
        FlowMean = Profile["FlowMean"]
        FlowStd  = max(Profile["FlowStd"], 0.1)
        DurMean  = Profile["DurMean"]
        DurStd   = max(Profile["DurStd"], 1.0)
        YieldStd = max(Profile["YieldStd"], 0.1)

        for _, GapRow in AnimalGaps.iterrows():
            GapDates = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")

            for Date in GapDates:
                if Date not in SessionArrays:
                    continue
                if (Aid, Date) not in WoodsPredDictAll:
                    continue

                ExpYield, ExpDIM, ExpLac = WoodsPredDictAll[(Aid, Date)]

                for Sess in [1, 2, 3]:
                    S       = SessionArrays[Date][Sess]
                    Yields  = S["Yield"]
                    Flows   = S["Flow"]
                    Durs    = S["Dur"]
                    Indices = S["Idx"]

                    YieldDiffs = abs(Yields - ExpYield)
                    FlowDiffs  = abs(Flows - FlowMean) / FlowStd
                    DurDiffs   = abs(Durs  - DurMean)  / DurStd

                    # Yield diff is primary — weight 10x
                    Scores = (10 * YieldDiffs / YieldStd +
                              FlowDiffs + DurDiffs)

                    BestPos   = np.argmin(Scores)
                    ActYield  = Yields[BestPos]
                    YieldDiff = YieldDiffs[BestPos]

                    Writer.writerow([Indices[BestPos], Aid,
                                     str(Date)[:10], Sess,
                                     ExpYield, ActYield,
                                     YieldDiff, Scores[BestPos]])

        if i % 500 == 0:
            print(f"  {i}/{AllGaps['AnimalId'].nunique()} · {time.time()-Start:.0f}s")

print(f"Done in {time.time()-Start:.0f}s")

  0/5617 · 0s
  500/5617 · 28s
  1000/5617 · 49s
  1500/5617 · 65s
  2000/5617 · 76s
  2500/5617 · 86s
  3000/5617 · 95s
  3500/5617 · 105s
  4000/5617 · 116s
  4500/5617 · 135s
  5000/5617 · 144s
  5500/5617 · 153s
Done in 155s


In [36]:
DfAssign4 = pd.read_csv(ProjectRoot / "DfFinalAssign4.csv")

DupCheck4  = DfAssign4.groupby("OrigIdx")["AnimalId"].nunique()
UniqueA4   = DfAssign4[DfAssign4["OrigIdx"].isin(DupCheck4[DupCheck4==1].index)]
ConflictA4 = DfAssign4[DfAssign4["OrigIdx"].isin(DupCheck4[DupCheck4>1].index)]
Resolved4  = (ConflictA4.sort_values("YieldDiff")
              .groupby("OrigIdx").first().reset_index())
DfFinal4   = pd.concat([UniqueA4, Resolved4], ignore_index=True)

print(f"Total records      : {len(DfFinal4):,}")
print(f"Unique animals     : {DfFinal4['AnimalId'].nunique():,}")
print(f"Mean yield diff    : {DfFinal4['YieldDiff'].mean():.3f} kg")

# Check our test animal
Aid = '-1007048835775197250'
print(f"\nTest animal records: {(DfFinal4['AnimalId'].astype(str)==Aid).sum()}")

Total records      : 601,381
Unique animals     : 5,604
Mean yield diff    : 0.122 kg

Test animal records: 203


In [37]:
DfFinal4.to_csv(ProjectRoot / "DfFinalAssign4.csv", index=False)
print("Saved")

# Apply to DfMissing
DfMissing["AnimalIdRecovered"] = pd.NA
AssignMap4 = DfFinal4.set_index("OrigIdx")["AnimalId"].to_dict()
for Idx, Aid in AssignMap4.items():
    DfMissing.loc[Idx, "AnimalIdRecovered"] = Aid

Recovered = DfMissing["AnimalIdRecovered"].notna().sum()
print(f"Recovered : {Recovered:,} / {len(DfMissing):,} ({Recovered/len(DfMissing)*100:.1f}%)")
print(f"Animals   : {DfMissing['AnimalIdRecovered'].nunique():,}")

Saved
Recovered : 601,381 / 1,701,003 (35.4%)
Animals   : 5,604


In [39]:
RecordsPerAnimal = (DfMissing[DfMissing["AnimalIdRecovered"].notna()]
                    .groupby(DfMissing["AnimalIdRecovered"].astype(str))
                    .size())

GapDaysPerAnimal = (DfGaps[DfGaps["GapDays"]>30]
                    .assign(AnimalId=DfGaps["AnimalId"].astype(str))
                    .groupby("AnimalId")["GapDays"].sum())

Coverage2 = pd.DataFrame({
    "GapDays":       GapDaysPerAnimal,
    "RecoveredRecs": RecordsPerAnimal
}).dropna()

Coverage2["CoveragePct"] = Coverage2["RecoveredRecs"] / (Coverage2["GapDays"] * 3)

print(f"Full coverage (>90%)  : {(Coverage2['CoveragePct']>0.9).sum():,}")
print(f"Good coverage (>70%)  : {(Coverage2['CoveragePct']>0.7).sum():,}")
print(f"Partial (30-70%)      : {(Coverage2['CoveragePct'].between(0.3,0.7)).sum():,}")
print(f"Poor (<30%)           : {(Coverage2['CoveragePct']<0.3).sum():,}")
print(f"Mean coverage         : {Coverage2['CoveragePct'].mean():.1%}")
print()
TestAid = '-1007048835775197250'
if TestAid in Coverage2.index:
    print(f"Test animal coverage  : {Coverage2.loc[TestAid,'CoveragePct']:.1%}")

Full coverage (>90%)  : 35
Good coverage (>70%)  : 295
Partial (30-70%)      : 2,731
Poor (<30%)           : 2,578
Mean coverage         : 34.4%

Test animal coverage  : 36.6%


In [40]:
Aid       = '-1007048835775197250'
BestIdx   = set(DfBest["OrigIdx"].values)
Final4Idx = set(DfFinal4[DfFinal4["AnimalId"].astype(str)==Aid]["OrigIdx"].values)

print(f"Best records found     : {len(BestIdx)}")
print(f"Made it to final assign: {len(BestIdx & Final4Idx)}")
print(f"Lost to conflict       : {len(BestIdx - Final4Idx)}")
print()

# What yield diff do the winning animals have for the lost records?
LostIdx = BestIdx - Final4Idx
Winners = DfFinal4[DfFinal4["OrigIdx"].isin(LostIdx)]
print(f"Winners mean yield diff : {Winners['YieldDiff'].mean():.4f} kg")
print(f"Our yield diff for same : {DfBest[DfBest['OrigIdx'].isin(LostIdx)]['YieldDiff'].mean():.4f} kg")

Best records found     : 66
Made it to final assign: 8
Lost to conflict       : 58

Winners mean yield diff : 0.0268 kg
Our yield diff for same : 0.0132 kg


In [41]:
# Reload DfAssign4 (before conflict resolution)
DfAssign4 = pd.read_csv(ProjectRoot / "DfFinalAssign4.csv")

# Resolve conflicts by YieldDiff ONLY
DupCheck4   = DfAssign4.groupby("OrigIdx")["AnimalId"].nunique()
UniqueA4    = DfAssign4[DfAssign4["OrigIdx"].isin(DupCheck4[DupCheck4==1].index)]
ConflictA4  = DfAssign4[DfAssign4["OrigIdx"].isin(DupCheck4[DupCheck4>1].index)]

# Keep lowest YieldDiff only
ResolvedYield = (ConflictA4.sort_values("YieldDiff")
                 .groupby("OrigIdx").first().reset_index())

DfFinal5 = pd.concat([UniqueA4, ResolvedYield], ignore_index=True)

print(f"Total records      : {len(DfFinal5):,}")
print(f"Unique animals     : {DfFinal5['AnimalId'].nunique():,}")
print(f"Mean yield diff    : {DfFinal5['YieldDiff'].mean():.4f} kg")
print()

# Check test animal
Aid = '-1007048835775197250'
AnimalRecs = DfFinal5[DfFinal5["AnimalId"].astype(str)==Aid]
print(f"Test animal records: {len(AnimalRecs)}")
print(f"Test animal yield diff: {AnimalRecs['YieldDiff'].mean():.4f} kg")

# Check how many of best 66 made it
BestIdx  = set(DfBest["OrigIdx"].values)
Final5Idx = set(AnimalRecs["OrigIdx"].values)
print(f"Best records recovered: {len(BestIdx & Final5Idx)} / {len(BestIdx)}")

Total records      : 601,381
Unique animals     : 5,604
Mean yield diff    : 0.1224 kg

Test animal records: 203
Test animal yield diff: 0.0334 kg
Best records recovered: 8 / 66


In [42]:
Aid     = '-1007048835775197250'
BestIdx = set(DfBest["OrigIdx"].values)

# How many of the 66 best records did this animal even claim in DfAssign4?
AnimalClaimed = set(DfAssign4[DfAssign4["AnimalId"].astype(str)==Aid]["OrigIdx"].values)
print(f"Best records total          : {len(BestIdx)}")
print(f"Claimed by animal in Assign4: {len(BestIdx & AnimalClaimed)}")
print(f"Never claimed at all        : {len(BestIdx - AnimalClaimed)}")

Best records total          : 66
Claimed by animal in Assign4: 8
Never claimed at all        : 58


##Another multimodel approach

In [43]:
# Cell 1 — Build trajectory assignments (no competition, write direct to CSV)
import csv, time
Start   = time.time()
OutPath = ProjectRoot / "DfTrajAssign.csv"
AllGaps = DfGaps[DfGaps["GapDays"] > 30].copy()
AllGaps["AnimalId"] = AllGaps["AnimalId"].astype(str)

with open(OutPath, "w", newline="") as f:
    Writer = csv.writer(f)
    Writer.writerow(["OrigIdx","AnimalId","Date","Session",
                     "ExpYield","ActYield","YieldDiff",
                     "ActFlow","ActDur","ExpDIM","ExpLac"])

    for i, (Aid, AnimalGaps) in enumerate(AllGaps.groupby("AnimalId")):
        if Aid not in AllProfiles:
            continue
        Profile  = AllProfiles[Aid]
        YieldStd = max(Profile["YieldStd"], 0.1)

        for _, GapRow in AnimalGaps.iterrows():
            GapDates = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")

            for Date in GapDates:
                if Date not in SessionArrays:
                    continue
                if (Aid, Date) not in WoodsPredDictAll:
                    continue

                ExpYield, ExpDIM, ExpLac = WoodsPredDictAll[(Aid, Date)]

                for Sess in [1, 2, 3]:
                    S          = SessionArrays[Date][Sess]
                    YieldDiffs = abs(S["Yield"] - ExpYield)
                    BestPos    = np.argmin(YieldDiffs)

                    Writer.writerow([
                        S["Idx"][BestPos], Aid, str(Date)[:10], Sess,
                        round(ExpYield, 4), round(S["Yield"][BestPos], 4),
                        round(YieldDiffs[BestPos], 4),
                        round(S["Flow"][BestPos], 4),
                        round(S["Dur"][BestPos], 1),
                        ExpDIM, ExpLac
                    ])

        if i % 500 == 0:
            print(f"  {i}/{AllGaps['AnimalId'].nunique()} · {time.time()-Start:.0f}s")

print(f"Done in {time.time()-Start:.0f}s")

  0/5617 · 0s
  500/5617 · 17s
  1000/5617 · 30s
  1500/5617 · 40s
  2000/5617 · 49s
  2500/5617 · 58s
  3000/5617 · 68s
  3500/5617 · 76s
  4000/5617 · 86s
  4500/5617 · 97s
  5000/5617 · 105s
  5500/5617 · 114s
Done in 116s


In [45]:
# Cell 2 — fixed
import time
Start = time.time()

# Build gap lookup dict — keyed by string AnimalId
GapLookup = {}
for _, Row in DfGaps[DfGaps["GapDays"] > 30].iterrows():
    Aid = str(Row["AnimalId"])
    if Aid not in GapLookup:
        GapLookup[Aid] = []
    GapLookup[Aid].append(Row)

TrajectoryScores = []

for Aid, Group in DfTraj.groupby("AnimalId"):
    Aid = str(Aid)
    if Aid not in GapLookup:
        continue

    GapRows  = GapLookup[Aid]
    TotalGapDays = sum(r["GapDays"] for r in GapRows)

    MeanYieldDiff = Group["YieldDiff"].mean()
    StdYieldDiff  = Group["YieldDiff"].std()
    NDays         = Group["Date"].nunique()

    # Calving event match
    CalvMatch = 0
    for GapRow in GapRows:
        CalvDate = GapRow.get("CalvingDate", pd.NaT)
        if pd.notna(CalvDate):
            CalvStr  = str(pd.Timestamp(CalvDate))[:10]
            CalvRecs = Group[Group["Date"] == CalvStr]
            if len(CalvRecs) > 0 and CalvRecs["ActYield"].mean() < 10:
                CalvMatch = 1

    TrajScore = MeanYieldDiff + 0.5 * StdYieldDiff

    TrajectoryScores.append({
        "AnimalId":      Aid,
        "MeanYieldDiff": MeanYieldDiff,
        "StdYieldDiff":  StdYieldDiff,
        "NDays":         NDays,
        "GapDays":       TotalGapDays,
        "DayCoverage":   NDays / TotalGapDays,
        "CalvMatch":     CalvMatch,
        "TrajScore":     TrajScore
    })

DfTrajScores = pd.DataFrame(TrajectoryScores)
print(f"Trajectory scores : {len(DfTrajScores):,}")
print(f"Mean yield diff   : {DfTrajScores['MeanYieldDiff'].mean():.4f} kg")
print(f"Mean day coverage : {DfTrajScores['DayCoverage'].mean():.1%}")
print(f"Calving match     : {DfTrajScores['CalvMatch'].mean():.1%}")
print(f"Time              : {time.time()-Start:.0f}s")

DfTrajScores.to_csv(ProjectRoot / "DfTrajScores.csv", index=False)
print("Saved")

Trajectory scores : 5,610
Mean yield diff   : 0.1511 kg
Mean day coverage : 94.7%
Calving match     : 79.6%
Time              : 12s
Saved


In [46]:
# Cell 3 — resolve conflicts using trajectory quality score
import time
Start = time.time()

# Merge trajectory scores into assignment
TrajScoreMap = DfTrajScores.set_index("AnimalId")["TrajScore"].to_dict()
DfTraj["TrajScore"] = DfTraj["AnimalId"].astype(str).map(TrajScoreMap)

# Resolve conflicts — keep animal with lowest TrajScore (best trajectory)
DupCheck  = DfTraj.groupby("OrigIdx")["AnimalId"].nunique()
UniqueA   = DfTraj[DfTraj["OrigIdx"].isin(DupCheck[DupCheck==1].index)]
ConflictA = DfTraj[DfTraj["OrigIdx"].isin(DupCheck[DupCheck>1].index)]

Resolved  = (ConflictA.sort_values("TrajScore")
             .groupby("OrigIdx").first().reset_index())

DfFinalTraj = pd.concat([UniqueA, Resolved], ignore_index=True)

print(f"Unique records before : {DfTraj['OrigIdx'].nunique():,}")
print(f"After resolution      : {len(DfFinalTraj):,}")
print(f"Unique animals        : {DfFinalTraj['AnimalId'].nunique():,}")
print(f"Mean yield diff       : {DfFinalTraj['YieldDiff'].mean():.4f} kg")
print(f"Time                  : {time.time()-Start:.0f}s")

# Check test animal
Aid = '-1007048835775197250'
AnimalRecs = DfFinalTraj[DfFinalTraj["AnimalId"].astype(str) == Aid]
BestIdx    = set(DfBest["OrigIdx"].values)
FinalIdx   = set(AnimalRecs["OrigIdx"].values)
print(f"\nTest animal records   : {len(AnimalRecs)}")
print(f"Best records recovered: {len(BestIdx & FinalIdx)} / {len(BestIdx)}")

DfFinalTraj.to_csv(ProjectRoot / "DfFinalTraj.csv", index=False)
print("Saved")

Unique records before : 411,806
After resolution      : 411,806
Unique animals        : 5,233
Mean yield diff       : 0.1433 kg
Time                  : 12s

Test animal records   : 108
Best records recovered: 14 / 66
Saved


In [47]:
Aid = '-1007048835775197250'

# How many days did this animal claim in DfTraj?
AnimalTraj = DfTraj[DfTraj["AnimalId"].astype(str) == Aid]
print(f"Records in DfTraj      : {len(AnimalTraj)}")
print(f"Unique dates in DfTraj : {AnimalTraj['Date'].nunique()}")
print(f"In DfFinalTraj         : {len(DfFinalTraj[DfFinalTraj['AnimalId'].astype(str)==Aid])}")
print()

# How many of its claimed records were lost to conflict?
AnimalIdx  = set(AnimalTraj["OrigIdx"].values)
FinalIdx   = set(DfFinalTraj[DfFinalTraj["AnimalId"].astype(str)==Aid]["OrigIdx"].values)
print(f"Claimed    : {len(AnimalIdx)}")
print(f"Won        : {len(FinalIdx)}")
print(f"Lost       : {len(AnimalIdx - FinalIdx)}")
print()

# What is the TrajScore for this animal vs winners
MyScore = TrajScoreMap.get(Aid, None)
print(f"My TrajScore : {MyScore:.4f}")

# Winners for lost records
LostIdx = AnimalIdx - FinalIdx
Winners = DfFinalTraj[DfFinalTraj["OrigIdx"].isin(LostIdx)]
WinnerScores = Winners["AnimalId"].astype(str).map(TrajScoreMap)
print(f"Winner TrajScores mean : {WinnerScores.mean():.4f}")

Records in DfTraj      : 555
Unique dates in DfTraj : 185
In DfFinalTraj         : 108

Claimed    : 555
Won        : 108
Lost       : 447

My TrajScore : 0.0318
Winner TrajScores mean : 0.0239


In [50]:
# Drop any partial columns from stopped loop
for Col in ["GapStart","GapEnd","GapTrajScore"]:
    if Col in DfTraj.columns:
        DfTraj.drop(columns=[Col], inplace=True)

DfTraj["Date"]     = DfTraj["Date"].astype(str).str[:10]
DfTraj["AnimalId"] = DfTraj["AnimalId"].astype(str)

DfGapTrajScores["GapStart"] = DfGapTrajScores["GapStart"].astype(str).str[:10]
DfGapTrajScores["GapEnd"]   = DfGapTrajScores["GapEnd"].astype(str).str[:10]
DfGapTrajScores["AnimalId"] = DfGapTrajScores["AnimalId"].astype(str)

DfTrajMerged = DfTraj.merge(
    DfGapTrajScores[["AnimalId","GapStart","GapEnd","GapTrajScore"]],
    on="AnimalId", how="left"
)

DfTrajMerged = DfTrajMerged[
    (DfTrajMerged["Date"] >= DfTrajMerged["GapStart"]) &
    (DfTrajMerged["Date"] <= DfTrajMerged["GapEnd"])
].copy()

print(f"Merged rows           : {len(DfTrajMerged):,}")
print(f"GapTrajScore assigned : {DfTrajMerged['GapTrajScore'].notna().sum():,}")

Merged rows           : 1,700,064
GapTrajScore assigned : 1,700,064


In [51]:
# Cell 3 — resolve conflicts using per-gap trajectory score
import time
Start = time.time()

DupCheck   = DfTrajMerged.groupby("OrigIdx")["AnimalId"].nunique()
UniqueA    = DfTrajMerged[DfTrajMerged["OrigIdx"].isin(DupCheck[DupCheck==1].index)]
ConflictA  = DfTrajMerged[DfTrajMerged["OrigIdx"].isin(DupCheck[DupCheck>1].index)]

Resolved   = (ConflictA.sort_values("GapTrajScore")
              .groupby("OrigIdx").first().reset_index())

DfFinalTraj2 = pd.concat([UniqueA, Resolved], ignore_index=True)

print(f"Total records      : {len(DfFinalTraj2):,}")
print(f"Unique animals     : {DfFinalTraj2['AnimalId'].nunique():,}")
print(f"Mean yield diff    : {DfFinalTraj2['YieldDiff'].mean():.4f} kg")
print(f"Time               : {time.time()-Start:.0f}s")

# Check test animal
Aid        = '-1007048835775197250'
AnimalRecs = DfFinalTraj2[DfFinalTraj2["AnimalId"].astype(str) == Aid]
BestIdx    = set(DfBest["OrigIdx"].values)
FinalIdx   = set(AnimalRecs["OrigIdx"].values)
print(f"\nTest animal records    : {len(AnimalRecs)}")
print(f"Best records recovered : {len(BestIdx & FinalIdx)} / {len(BestIdx)}")
print(f"Test animal TrajScore  : {TrajScoreMap.get(Aid, 'N/A')}")

DfFinalTraj2.to_csv(ProjectRoot / "DfFinalTraj2.csv", index=False)
print("Saved")

Total records      : 411,806
Unique animals     : 5,330
Mean yield diff    : 0.1444 kg
Time               : 9s

Test animal records    : 128
Best records recovered : 21 / 66
Test animal TrajScore  : 0.0317635669867494
Saved


In [52]:
Aid        = '-1007048835775197250'
AnimalTraj = DfTrajMerged[DfTrajMerged["AnimalId"].astype(str) == Aid]

print(f"Total claims           : {len(AnimalTraj)}")
print(f"Unique records claimed : {AnimalTraj['OrigIdx'].nunique()}")
print(f"Unique dates           : {AnimalTraj['Date'].nunique()}")
print(f"Sessions per date      : {len(AnimalTraj)/AnimalTraj['Date'].nunique():.1f}")
print()

# How many other animals claim the same records?
SampleRecords = AnimalTraj["OrigIdx"].values[:10]
Competitors   = DfTrajMerged[DfTrajMerged["OrigIdx"].isin(SampleRecords)]
print(f"Mean competitors per record : {Competitors.groupby('OrigIdx')['AnimalId'].nunique().mean():.1f}")
print(f"Max competitors per record  : {Competitors.groupby('OrigIdx')['AnimalId'].nunique().max()}")

Total claims           : 555
Unique records claimed : 555
Unique dates           : 185
Sessions per date      : 3.0

Mean competitors per record : 3.4
Max competitors per record  : 6


In [54]:
DfGapTrajScores.to_csv(ProjectRoot / "DfGapTrajScores.csv", index=False)
print("Saved")

Saved


#Combine all methodology

In [57]:
# Check what's available
import os

print("=== In Memory ===")
print(f"DfTrajMerged  : {'DfTrajMerged' in dir() and len(DfTrajMerged):,}"
      if 'DfTrajMerged' in dir() else "DfTrajMerged  : NOT in memory")
print(f"DfGapTrajScores: {'DfGapTrajScores' in dir() and len(DfGapTrajScores):,}"
      if 'DfGapTrajScores' in dir() else "DfGapTrajScores: NOT in memory")
print(f"SessionArrays : {'SessionArrays' in dir() and len(SessionArrays):,} dates"
      if 'SessionArrays' in dir() else "SessionArrays : NOT in memory")
print(f"WoodsPredDictAll: {'WoodsPredDictAll' in dir() and len(WoodsPredDictAll):,}"
      if 'WoodsPredDictAll' in dir() else "WoodsPredDictAll: NOT in memory")
print(f"AllProfiles   : {'AllProfiles' in dir() and len(AllProfiles):,}"
      if 'AllProfiles' in dir() else "AllProfiles   : NOT in memory")
print()
print("=== On Drive ===")
Files = [
    "DfGaps.csv", "DfSig.csv", "UniqueSlots.csv",
    "DfTrajAssign.csv", "DfFinalTraj2.csv", "DfTrajScores.csv",
    "DfGapTrajScores.csv", "AllWoodsParams.pkl",
    "AllProfiles.pkl", "WoodsPredDictAll.pkl",
    "DfKnown_clean.csv", "DfMissing_clean.csv"
]
for F in Files:
    Path = ProjectRoot / F
    Exists = os.path.exists(Path)
    Size   = f"{os.path.getsize(Path)//1024//1024} MB" if Exists else ""
    print(f"  {'✓' if Exists else '✗'} {F} {Size}")

=== In Memory ===
DfTrajMerged  : 1,700,064
DfGapTrajScores: 8,923
SessionArrays : 830 dates
WoodsPredDictAll: 597,822
AllProfiles   : 5,617

=== On Drive ===
  ✓ DfGaps.csv 1 MB
  ✓ DfSig.csv 29 MB
  ✓ UniqueSlots.csv 14 MB
  ✓ DfTrajAssign.csv 129 MB
  ✓ DfFinalTraj2.csv 55 MB
  ✓ DfTrajScores.csv 0 MB
  ✓ DfGapTrajScores.csv 0 MB
  ✓ AllWoodsParams.pkl 0 MB
  ✓ AllProfiles.pkl 0 MB
  ✓ WoodsPredDictAll.pkl 30 MB
  ✓ DfKnown_clean.csv 746 MB
  ✓ DfMissing_clean.csv 133 MB


In [59]:
ProfileVec = np.array([
    Profile["YieldMean"],
    Profile["FlowMean"],
    Profile["DurMean"]
])

In [62]:
Start    = time.time()
AllGaps  = DfGaps[DfGaps["GapDays"] > 30].copy()
AllGaps["AnimalId"] = AllGaps["AnimalId"].astype(str)
ClusterResults = []

TestAnimals = AllGaps["AnimalId"].unique()[:10]

for Aid in TestAnimals:
    if Aid not in AllProfiles:
        continue
    Profile    = AllProfiles[Aid]
    AnimalGaps = AllGaps[AllGaps["AnimalId"] == Aid]

    for _, GapRow in AnimalGaps.iterrows():
        GapDates = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")

        GapRecs = []
        for Date in GapDates:
            if Date not in SessionArrays:
                continue
            for Sess in [1,2,3]:
                S = SessionArrays[Date][Sess]
                for j in range(len(S["Idx"])):
                    GapRecs.append({
                        "OrigIdx": S["Idx"][j],
                        "Yield":   S["Yield"][j],
                        "Flow":    S["Flow"][j],
                        "Dur":     S["Dur"][j]
                    })

        if len(GapRecs) < 20:
            continue

        DfGapRecs  = pd.DataFrame(GapRecs)
        ScalerFit = StandardScaler().fit(DfGapRecs[["Yield","Flow","Dur"]].fillna(0).values)
        X = ScalerFit.transform(DfGapRecs[["Yield","Flow","Dur"]].fillna(0).values)
        ProfScaled = ScalerFit.transform(np.array([[
            Profile["YieldMean"],
            Profile["FlowMean"],
            Profile["DurMean"]
        ]]))

        NClust  = min(50, len(DfGapRecs) // 10)
        Km      = KMeans(n_clusters=NClust, random_state=42,
                         n_init=3, max_iter=100)
        Km.fit(X)

        Distances  = np.linalg.norm(Km.cluster_centers_ - ProfScaled, axis=1)
        BestClust  = np.argmin(Distances)
        BestDist   = Distances[BestClust]
        SecondDist = np.sort(Distances)[1]
        BestIdx    = DfGapRecs[Km.labels_ == BestClust]["OrigIdx"].values

        ClusterResults.append({
            "AnimalId":    Aid,
            "GapStart":    str(GapRow["GapStart"])[:10],
            "GapEnd":      str(GapRow["GapEnd"])[:10],
            "ClusterIdx":  BestIdx,
            "ClusterDist": BestDist,
            "MarginRatio": SecondDist/BestDist if BestDist > 0 else 999,
            "NRecords":    len(BestIdx)
        })

print(f"Cluster results : {len(ClusterResults)}")
print(f"Time            : {time.time()-Start:.1f}s")
for R in ClusterResults[:3]:
    print(f"  {R['AnimalId'][:15]}... NRecords:{R['NRecords']} Margin:{R['MarginRatio']:.2f}x")

Cluster results : 25
Time            : 173.8s
  -10070488357751... NRecords:2556 Margin:1.42x
  -10070488357751... NRecords:5982 Margin:1.18x
  -10074682684645... NRecords:1790 Margin:2.40x


In [63]:
# Check why it's slow — how many records per gap period?
for R in ClusterResults[:3]:
    Aid      = R["AnimalId"]
    GapStart = R["GapStart"]
    GapEnd   = R["GapEnd"]
    GapDates = pd.date_range(GapStart, GapEnd, freq="D")
    TotalRecs = sum(
        len(SessionArrays[D][S]["Idx"])
        for D in GapDates if D in SessionArrays
        for S in [1,2,3]
    )
    print(f"Gap {GapStart}→{GapEnd}: {len(GapDates)} days, {TotalRecs:,} records, NClust={R['NRecords']}")

Gap 2020-01-22→2020-03-27: 66 days, 131,711 records, NClust=2556
Gap 2021-03-12→2021-07-08: 119 days, 260,924 records, NClust=5982
Gap 2019-07-26→2019-09-07: 44 days, 47,724 records, NClust=1790


In [65]:
# Pre-filter using Wood's curve prediction per date — much tighter
GapRecs = []
for Date in GapDates:
    if Date not in SessionArrays or (Aid, Date) not in WoodsPredDictAll:
        continue
    ExpYield = WoodsPredDictAll[(Aid, Date)][0]
    for Sess in [1,2,3]:
        S    = SessionArrays[Date][Sess]
        Mask = ((S["Yield"] >= ExpYield - YieldStd) &
                (S["Yield"] <= ExpYield + YieldStd))
        for j in np.where(Mask)[0]:
            GapRecs.append({
                "OrigIdx": S["Idx"][j],
                "Yield":   S["Yield"][j],
                "Flow":    S["Flow"][j],
                "Dur":     S["Dur"][j]
            })

In [66]:
Start   = time.time()
AllGaps = DfGaps[DfGaps["GapDays"] > 30].copy()
AllGaps["AnimalId"] = AllGaps["AnimalId"].astype(str)
ClusterResults = []

TestAnimals = AllGaps["AnimalId"].unique()[:10]

for Aid in TestAnimals:
    if Aid not in AllProfiles:
        continue
    Profile    = AllProfiles[Aid]
    YieldMean  = Profile["YieldMean"]
    YieldStd   = max(Profile["YieldStd"], 0.5)
    AnimalGaps = AllGaps[AllGaps["AnimalId"] == Aid]

    for _, GapRow in AnimalGaps.iterrows():
        GapDates = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")

        # Pre-filter to yield within ±2std — reduces pool dramatically
        GapRecs = []
        for Date in GapDates:
            if Date not in SessionArrays:
                continue
            for Sess in [1,2,3]:
                S = SessionArrays[Date][Sess]
                Mask = ((S["Yield"] >= YieldMean - 2*YieldStd) &
                        (S["Yield"] <= YieldMean + 2*YieldStd))
                for j in np.where(Mask)[0]:
                    GapRecs.append({
                        "OrigIdx": S["Idx"][j],
                        "Yield":   S["Yield"][j],
                        "Flow":    S["Flow"][j],
                        "Dur":     S["Dur"][j]
                    })

        if len(GapRecs) < 20:
            continue

        DfGapRecs  = pd.DataFrame(GapRecs)
        ScalerFit  = StandardScaler().fit(
                     DfGapRecs[["Yield","Flow","Dur"]].fillna(0).values)
        X          = ScalerFit.transform(
                     DfGapRecs[["Yield","Flow","Dur"]].fillna(0).values)
        ProfScaled = ScalerFit.transform(np.array([[
                     YieldMean, Profile["FlowMean"], Profile["DurMean"]]]))

        NClust  = min(30, max(5, len(DfGapRecs) // 100))
        Km      = MiniBatchKMeans(n_clusters=NClust, random_state=42,
                                  n_init=3, max_iter=100, batch_size=10_000)
        Km.fit(X)

        Distances  = np.linalg.norm(Km.cluster_centers_ - ProfScaled, axis=1)
        BestClust  = np.argmin(Distances)
        BestDist   = Distances[BestClust]
        SecondDist = np.sort(Distances)[1] if len(Distances) > 1 else 999
        BestIdx    = DfGapRecs[Km.labels_ == BestClust]["OrigIdx"].values

        ClusterResults.append({
            "AnimalId":    Aid,
            "GapStart":    str(GapRow["GapStart"])[:10],
            "GapEnd":      str(GapRow["GapEnd"])[:10],
            "ClusterIdx":  BestIdx,
            "ClusterDist": BestDist,
            "MarginRatio": SecondDist/BestDist if BestDist > 0 else 999,
            "NRecords":    len(BestIdx),
            "NFiltered":   len(DfGapRecs)
        })

print(f"Results : {len(ClusterResults)}")
print(f"Time    : {time.time()-Start:.1f}s")
for R in ClusterResults[:3]:
    print(f"  NFiltered:{R['NFiltered']:,} NRecords:{R['NRecords']:,} "
          f"Margin:{R['MarginRatio']:.2f}x")

Results : 25
Time    : 57.3s
  NFiltered:114,546 NRecords:5,728 Margin:1.87x
  NFiltered:227,684 NRecords:10,871 Margin:1.16x
  NFiltered:45,627 NRecords:1,947 Margin:1.33x


In [67]:
# Quick check — how many records pass ±1std Wood's curve filter for one animal
Aid      = AllGaps["AnimalId"].unique()[0]
GapRow   = AllGaps[AllGaps["AnimalId"]==Aid].iloc[0]
GapDates = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")
YieldStd = max(AllProfiles[Aid]["YieldStd"], 0.5)

TotalAll     = 0
TotalFiltered = 0
for Date in GapDates:
    if Date not in SessionArrays or (Aid, Date) not in WoodsPredDictAll:
        continue
    ExpYield = WoodsPredDictAll[(Aid, Date)][0]
    for Sess in [1,2,3]:
        S = SessionArrays[Date][Sess]
        TotalAll += len(S["Yield"])
        Mask = ((S["Yield"] >= ExpYield - YieldStd) &
                (S["Yield"] <= ExpYield + YieldStd))
        TotalFiltered += Mask.sum()

print(f"Total records in gap      : {TotalAll:,}")
print(f"After ±1std Wood's filter : {TotalFiltered:,}")
print(f"Reduction                 : {1-TotalFiltered/TotalAll:.1%}")

Total records in gap      : 131,711
After ±1std Wood's filter : 66,701
Reduction                 : 49.4%


In [69]:
# Use densest cluster — not closest to profile
BestClust  = np.bincount(Km.labels_).argmax()  # most records
BestIdx    = TrajRecs[Km.labels_ == BestClust]["OrigIdx"].values

In [70]:
# Cluster only the trajectory-selected records — not all gap records
import time
Start   = time.time()
AllGaps = DfGaps[DfGaps["GapDays"] > 30].copy()
AllGaps["AnimalId"] = AllGaps["AnimalId"].astype(str)

ClusterResults = []
TestAnimals    = AllGaps["AnimalId"].unique()[:10]

for Aid in TestAnimals:
    if Aid not in AllProfiles:
        continue
    Profile    = AllProfiles[Aid]
    AnimalGaps = AllGaps[AllGaps["AnimalId"] == Aid]

    # Get trajectory selected records for this animal
    TrajRecs = DfTrajMerged[DfTrajMerged["AnimalId"].astype(str) == Aid][
        ["OrigIdx","Date","Session","ActYield","ActFlow","ActDur"]
    ].copy()

    if len(TrajRecs) < 10:
        continue

    # Cluster these ~198 records
    ScalerFit  = StandardScaler().fit(
                 TrajRecs[["ActYield","ActFlow","ActDur"]].fillna(0).values)
    X          = ScalerFit.transform(
                 TrajRecs[["ActYield","ActFlow","ActDur"]].fillna(0).values)
    ProfScaled = ScalerFit.transform(np.array([[
                 Profile["YieldMean"],
                 Profile["FlowMean"],
                 Profile["DurMean"]]]))

    NClust = min(20, max(5, len(TrajRecs) // 30))
    Km      = MiniBatchKMeans(n_clusters=NClust, random_state=42,
                              n_init=3, max_iter=100)
    Km.fit(X)

    Distances  = np.linalg.norm(Km.cluster_centers_ - ProfScaled, axis=1)
    BestClust  = np.argmin(Distances)
    BestDist   = Distances[BestClust]
    SecondDist = np.sort(Distances)[1] if len(Distances) > 1 else 999
    BestIdx    = TrajRecs[Km.labels_ == BestClust]["OrigIdx"].values

    ClusterResults.append({
        "AnimalId":    Aid,
        "ClusterIdx":  set(BestIdx),
        "ClusterDist": BestDist,
        "MarginRatio": SecondDist/BestDist if BestDist > 0 else 999,
        "NRecords":    len(BestIdx),
        "NTraj":       len(TrajRecs)
    })

print(f"Results : {len(ClusterResults)}")
print(f"Time    : {time.time()-Start:.1f}s")
for R in ClusterResults[:3]:
    print(f"  NTraj:{R['NTraj']} NClusterRecs:{R['NRecords']} "
          f"Margin:{R['MarginRatio']:.2f}x")

Results : 10
Time    : 5.4s
  NTraj:555 NClusterRecs:13 Margin:1.10x
  NTraj:792 NClusterRecs:22 Margin:1.54x
  NTraj:573 NClusterRecs:6 Margin:1.16x


In [71]:
# Find high-confidence assignments — animals with uniquely tight yield match
# For each contested record, compare yield diffs of all claimants

import time
Start = time.time()

# Get yield diff per record per animal from trajectory
DupCheck = DfTrajMerged.groupby("OrigIdx")["AnimalId"].nunique()
ConflictIdx = DupCheck[DupCheck > 1].index

ConflictRecs = DfTrajMerged[DfTrajMerged["OrigIdx"].isin(ConflictIdx)].copy()

# For each contested record, find best and second best yield diff
BestPerRecord = (ConflictRecs.sort_values("YieldDiff")
                 .groupby("OrigIdx")
                 .agg(
                     BestAnimal   = ("AnimalId", "first"),
                     BestYieldDiff= ("YieldDiff", "first"),
                     SecondDiff   = ("YieldDiff", lambda x: x.nsmallest(2).iloc[-1])
                 ).reset_index())

BestPerRecord["DiffMargin"] = BestPerRecord["SecondDiff"] - BestPerRecord["BestYieldDiff"]
BestPerRecord["Confident"]  = BestPerRecord["DiffMargin"] > 0.5

print(f"Contested records          : {len(BestPerRecord):,}")
print(f"High confidence (>0.5 gap) : {BestPerRecord['Confident'].sum():,}")
print(f"Low confidence             : {(~BestPerRecord['Confident']).sum():,}")
print(f"Mean diff margin           : {BestPerRecord['DiffMargin'].mean():.3f} kg")
print(f"Time                       : {time.time()-Start:.1f}s")
print()

# Check test animal
Aid = '-1007048835775197250'
TestConf = BestPerRecord[BestPerRecord["BestAnimal"].astype(str) == Aid]
print(f"Test animal high-conf records : {TestConf['Confident'].sum():,}")
print(f"Test animal total contested   : {len(TestConf):,}")

Contested records          : 285,675
High confidence (>0.5 gap) : 1,283
Low confidence             : 284,392
Mean diff margin           : 0.165 kg
Time                       : 230.7s

Test animal high-conf records : 0
Test animal total contested   : 73


In [72]:
UniqueA = DfTrajMerged[DfTrajMerged["OrigIdx"].isin(
    DupCheck[DupCheck==1].index)]
print(f"Uncontested records : {len(UniqueA):,}")
print(f"Unique animals      : {UniqueA['AnimalId'].nunique():,}")
print(f"Mean yield diff     : {UniqueA['YieldDiff'].mean():.4f} kg")

Uncontested records : 126,131
Unique animals      : 4,004
Mean yield diff     : 0.1156 kg


In [73]:
# Final export
DfUncontested = DfTrajMerged[DfTrajMerged["OrigIdx"].isin(
    DupCheck[DupCheck==1].index)].copy()
DfUncontested["Confidence"] = "HIGH"

DfContested = DfFinalTraj2[~DfFinalTraj2["OrigIdx"].isin(
    DfUncontested["OrigIdx"])].copy()
DfContested["Confidence"] = "MEDIUM"

DfFinalExport = pd.concat([DfUncontested, DfContested], ignore_index=True)

print(f"HIGH confidence   : {(DfFinalExport['Confidence']=='HIGH').sum():,}")
print(f"MEDIUM confidence : {(DfFinalExport['Confidence']=='MEDIUM').sum():,}")
print(f"Total             : {len(DfFinalExport):,}")
print(f"Unique animals    : {DfFinalExport['AnimalId'].nunique():,}")

DfFinalExport.to_csv(ProjectRoot / "DfFinalExport.csv", index=False)
print("Saved")

HIGH confidence   : 126,131
MEDIUM confidence : 285,675
Total             : 411,806
Unique animals    : 5,330
Saved


In [74]:
# How many gap animals vs available records per date-session?
from collections import defaultdict
import time
Start = time.time()

AllGaps = DfGaps[DfGaps["GapDays"] > 30].copy()
AllGaps["AnimalId"] = AllGaps["AnimalId"].astype(str)

# Build: date -> list of animals missing that date
AnimalsByDate = defaultdict(list)
for _, Row in AllGaps.iterrows():
    Aid = Row["AnimalId"]
    for Date in pd.date_range(Row["GapStart"], Row["GapEnd"], freq="D"):
        AnimalsByDate[Date].append(Aid)

Balance = []
for Date in sorted(AnimalsByDate.keys()):
    if Date not in SessionArrays:
        continue
    NAnimals = len(set(AnimalsByDate[Date]))
    NRecs    = len(SessionArrays[Date][1]["Idx"])
    Balance.append({"Date": Date, "NAnimals": NAnimals, "NRecordsS1": NRecs})

DfBalance = pd.DataFrame(Balance)
print(f"Dates checked          : {len(DfBalance):,}")
print(f"Mean animals missing   : {DfBalance['NAnimals'].mean():.0f}")
print(f"Mean records (session1): {DfBalance['NRecordsS1'].mean():.0f}")
print(f"Mean ratio anim/recs   : {(DfBalance['NAnimals']/DfBalance['NRecordsS1']).mean():.2f}")
print(f"Max animals on a date  : {DfBalance['NAnimals'].max():,}")
print(f"Time                   : {time.time()-Start:.1f}s")

# Save for next step
import pickle
with open(ProjectRoot / "AnimalsByDate.pkl", "wb") as f:
    pickle.dump(dict(AnimalsByDate), f)
print("Saved AnimalsByDate")

Dates checked          : 827
Mean animals missing   : 686
Mean records (session1): 684
Mean ratio anim/recs   : 1.16
Max animals on a date  : 1,023
Time                   : 14.6s
Saved AnimalsByDate


In [75]:
from scipy.optimize import linear_sum_assignment
import time

# Test on one date, one session
TestDate = pd.Timestamp("2020-01-22")
Sess     = 1

# Animals missing on this date
Animals  = [Aid for Aid in AnimalsByDate[TestDate]
            if (Aid, TestDate) in WoodsPredDictAll]
Records  = SessionArrays[TestDate][Sess]

NAnimals = len(Animals)
NRecords = len(Records["Idx"])

print(f"Date     : {TestDate.date()}")
print(f"Session  : {Sess}")
print(f"Animals  : {NAnimals}")
print(f"Records  : {NRecords}")

# Build cost matrix — rows=animals, cols=records
Start     = time.time()
CostMatrix = np.zeros((NAnimals, NRecords))

for i, Aid in enumerate(Animals):
    ExpYield = WoodsPredDictAll[(Aid, TestDate)][0]
    CostMatrix[i, :] = np.abs(Records["Yield"] - ExpYield)

print(f"Cost matrix built : {CostMatrix.shape} in {time.time()-Start:.2f}s")

# Solve — add dummy columns if more animals than records
if NAnimals > NRecords:
    MaxCost   = CostMatrix.max() * 10
    DummyCols = np.full((NAnimals, NAnimals - NRecords), MaxCost)
    CostFull  = np.hstack([CostMatrix, DummyCols])
else:
    CostFull = CostMatrix

RowIdx, ColIdx = linear_sum_assignment(CostFull)
print(f"Assignment solved in {time.time()-Start:.3f}s")

# Results
AssignedYieldDiffs = []
NullAssigned       = 0
for r, c in zip(RowIdx, ColIdx):
    if c >= NRecords:
        NullAssigned += 1
    else:
        AssignedYieldDiffs.append(CostMatrix[r, c])

print(f"Animals assigned to real record : {len(AssignedYieldDiffs)}")
print(f"Animals with no record (null)   : {NullAssigned}")
print(f"Mean yield diff                 : {np.mean(AssignedYieldDiffs):.4f} kg")
print(f"Max yield diff                  : {np.max(AssignedYieldDiffs):.4f} kg")

Date     : 2020-01-22
Session  : 1
Animals  : 735
Records  : 560
Cost matrix built : (735, 560) in 0.13s
Assignment solved in 1.982s
Animals assigned to real record : 560
Animals with no record (null)   : 175
Mean yield diff                 : 3.7304 kg
Max yield diff                  : 15.9225 kg


In [76]:
TestDate = pd.Timestamp("2020-01-22")
Sess     = 1
Records  = SessionArrays[TestDate][Sess]
RecYields = Records["Yield"]
RecFlows  = Records["Flow"]
RecDurs   = Records["Dur"]

All          = 0
AfterGap     = 0
AfterYield   = 0
AfterLac     = 0
AfterDIM     = 0

for Aid in AnimalsByDate.get(TestDate, []):
    All += 1

    # Filter 1: gap pattern — already in AnimalsByDate so pass
    AfterGap += 1

    if (Aid, TestDate) not in WoodsPredDictAll:
        continue
    ExpYield, ExpDIM, ExpLac = WoodsPredDictAll[(Aid, TestDate)]
    YieldStd = max(AllProfiles[Aid]["YieldStd"], 0.5)

    # Filter 2: yield within ±1.5 std
    if not np.any(np.abs(RecYields - ExpYield) <= 1.5 * YieldStd):
        continue
    AfterYield += 1

    # Filter 3: lactation plausible (1-8)
    if ExpLac < 1 or ExpLac > 8:
        continue
    AfterLac += 1

    # Filter 4: DIM must be positive
    if ExpDIM < 1:
        continue
    AfterDIM += 1

print(f"All animals missing on date : {All}")
print(f"After gap pattern filter    : {AfterGap}")
print(f"After yield ±1.5std filter  : {AfterYield}")
print(f"After lactation filter      : {AfterLac}")
print(f"After DIM filter            : {AfterDIM}")
print(f"Records available           : {len(RecYields)}")
print(f"Final ratio                 : {AfterDIM/len(RecYields):.2f}")

All animals missing on date : 735
After gap pattern filter    : 735
After yield ±1.5std filter  : 729
After lactation filter      : 727
After DIM filter            : 727
Records available           : 560
Final ratio                 : 1.30


In [77]:
# How many of the 727 are likely dry/not milking?
DryCount    = 0
MilkingCount= 0

for Aid in AnimalsByDate.get(TestDate, []):
    if (Aid, TestDate) not in WoodsPredDictAll:
        continue
    ExpYield, ExpDIM, ExpLac = WoodsPredDictAll[(Aid, TestDate)]
    YieldStd = max(AllProfiles[Aid]["YieldStd"], 0.5)

    if not np.any(np.abs(RecYields - ExpYield) <= 1.5 * YieldStd):
        continue
    if ExpLac < 1 or ExpLac > 8 or ExpDIM < 1:
        continue

    # Check if likely dry — DIM > 305 or expected yield < 3 kg
    ExpYield2, _, _ = WoodsPredDictAll[(Aid, TestDate)]
    if ExpYield2 < 3.0 or ExpDIM > 305:
        DryCount += 1
    else:
        MilkingCount += 1

print(f"Likely milking (yield>3, DIM<305) : {MilkingCount}")
print(f"Likely dry     (yield<3 or DIM>305): {DryCount}")
print(f"Records available                  : {len(RecYields)}")
print(f"Ratio after dry filter             : {MilkingCount/len(RecYields):.2f}")

Likely milking (yield>3, DIM<305) : 225
Likely dry     (yield<3 or DIM>305): 502
Records available                  : 560
Ratio after dry filter             : 0.40


In [78]:
from scipy.optimize import linear_sum_assignment
import time

TestDate = pd.Timestamp("2020-01-22")
Sess     = 1
Records  = SessionArrays[TestDate][Sess]
RecYields = Records["Yield"]
RecFlows  = Records["Flow"]
RecDurs   = Records["Dur"]
RecIdx    = Records["Idx"]

# Build filtered animal list
FilteredAnimals = []
for Aid in AnimalsByDate.get(TestDate, []):
    if (Aid, TestDate) not in WoodsPredDictAll:
        continue
    ExpYield, ExpDIM, ExpLac = WoodsPredDictAll[(Aid, TestDate)]
    YieldStd = max(AllProfiles[Aid]["YieldStd"], 0.5)
    if ExpLac < 1 or ExpLac > 8 or ExpDIM < 1:
        continue
    if ExpYield < 3.0 or ExpDIM > 305:
        continue
    if not np.any(np.abs(RecYields - ExpYield) <= 1.5 * YieldStd):
        continue
    FilteredAnimals.append((Aid, ExpYield, ExpDIM, ExpLac))

NAnimals = len(FilteredAnimals)
NRecords = len(RecYields)
print(f"Filtered animals : {NAnimals}")
print(f"Records          : {NRecords}")

# Build cost matrix
Start      = time.time()
CostMatrix = np.zeros((NAnimals, NRecords))
for i, (Aid, ExpYield, ExpDIM, ExpLac) in enumerate(FilteredAnimals):
    YieldStd = max(AllProfiles[Aid]["YieldStd"], 0.5)
    FlowMean = AllProfiles[Aid]["FlowMean"]
    FlowStd  = max(AllProfiles[Aid]["FlowStd"], 0.1)
    DurMean  = AllProfiles[Aid]["DurMean"]
    DurStd   = max(AllProfiles[Aid]["DurStd"], 1.0)
    CostMatrix[i,:] = (
        10 * np.abs(RecYields - ExpYield) / YieldStd +
        np.abs(RecFlows - FlowMean) / FlowStd +
        np.abs(RecDurs  - DurMean)  / DurStd
    )

print(f"Cost matrix built : {CostMatrix.shape} in {time.time()-Start:.3f}s")

# Solve
RowIdx, ColIdx = linear_sum_assignment(CostMatrix)
print(f"Solved in {time.time()-Start:.3f}s")

# Results
YieldDiffs = [np.abs(RecYields[c] - FilteredAnimals[r][1])
              for r,c in zip(RowIdx, ColIdx)]
print(f"Mean yield diff  : {np.mean(YieldDiffs):.4f} kg")
print(f"Max yield diff   : {np.max(YieldDiffs):.4f} kg")
print(f"Within ±0.5kg    : {sum(1 for d in YieldDiffs if d<0.5)} / {len(YieldDiffs)}")
print(f"Within ±1.0kg    : {sum(1 for d in YieldDiffs if d<1.0)} / {len(YieldDiffs)}")

Filtered animals : 225
Records          : 560
Cost matrix built : (225, 560) in 0.024s
Solved in 0.041s
Mean yield diff  : 0.4123 kg
Max yield diff   : 6.5853 kg
Within ±0.5kg    : 191 / 225
Within ±1.0kg    : 200 / 225


In [79]:
# Apply yield diff threshold — reject assignments > 2 std
Results = []
Rejected = 0

for r, c in zip(RowIdx, ColIdx):
    Aid, ExpYield, ExpDIM, ExpLac = FilteredAnimals[r]
    YieldStd  = max(AllProfiles[Aid]["YieldStd"], 0.5)
    ActYield  = RecYields[c]
    YieldDiff = np.abs(ActYield - ExpYield)

    if YieldDiff > 2 * YieldStd:
        Rejected += 1
        continue

    Results.append({
        "OrigIdx":   RecIdx[c],
        "AnimalId":  Aid,
        "ExpYield":  ExpYield,
        "ActYield":  ActYield,
        "YieldDiff": YieldDiff,
        "ExpDIM":    ExpDIM,
        "ExpLac":    ExpLac
    })

print(f"Assigned (within 2 std) : {len(Results)}")
print(f"Rejected (too far)      : {Rejected}")
print(f"Mean yield diff         : {np.mean([r['YieldDiff'] for r in Results]):.4f} kg")
print(f"Within ±0.5kg           : {sum(1 for r in Results if r['YieldDiff']<0.5)}")
print(f"Within ±1.0kg           : {sum(1 for r in Results if r['YieldDiff']<1.0)}")

Assigned (within 2 std) : 225
Rejected (too far)      : 0
Mean yield diff         : 0.4123 kg
Within ±0.5kg           : 191
Within ±1.0kg           : 200


In [80]:
import csv, time
Start   = time.time()
OutPath = ProjectRoot / "DfHungarian_test.csv"
Dates   = sorted(AnimalsByDate.keys())[:50]

with open(OutPath, "w", newline="") as f:
    Writer = csv.writer(f)
    Writer.writerow(["OrigIdx","AnimalId","Date","Session",
                     "ExpYield","ActYield","YieldDiff","ExpDIM","ExpLac"])

    TotalAssigned = 0

    for Date in Dates:
        if Date not in SessionArrays:
            continue

        for Sess in [1, 2, 3]:
            Records   = SessionArrays[Date][Sess]
            RecYields = Records["Yield"]
            RecFlows  = Records["Flow"]
            RecDurs   = Records["Dur"]
            RecIdx    = Records["Idx"]

            FilteredAnimals = []
            for Aid in AnimalsByDate.get(Date, []):
                if (Aid, Date) not in WoodsPredDictAll:
                    continue
                ExpYield, ExpDIM, ExpLac = WoodsPredDictAll[(Aid, Date)]
                YieldStd = max(AllProfiles[Aid]["YieldStd"], 0.5)
                if ExpLac < 1 or ExpLac > 8 or ExpDIM < 1:
                    continue
                if ExpYield < 3.0 or ExpDIM > 305:
                    continue
                if not np.any(np.abs(RecYields - ExpYield) <= 1.5 * YieldStd):
                    continue
                FilteredAnimals.append((Aid, ExpYield, ExpDIM, ExpLac))

            if len(FilteredAnimals) == 0:
                continue

            CostMatrix = np.zeros((len(FilteredAnimals), len(RecYields)))
            for i, (Aid, ExpYield, ExpDIM, ExpLac) in enumerate(FilteredAnimals):
                YieldStd = max(AllProfiles[Aid]["YieldStd"], 0.5)
                FlowMean = AllProfiles[Aid]["FlowMean"]
                FlowStd  = max(AllProfiles[Aid]["FlowStd"], 0.1)
                DurMean  = AllProfiles[Aid]["DurMean"]
                DurStd   = max(AllProfiles[Aid]["DurStd"], 1.0)
                CostMatrix[i,:] = (
                    10 * np.abs(RecYields - ExpYield) / YieldStd +
                    np.abs(RecFlows - FlowMean) / FlowStd +
                    np.abs(RecDurs  - DurMean)  / DurStd
                )

            RowIdx, ColIdx = linear_sum_assignment(CostMatrix)

            for r, c in zip(RowIdx, ColIdx):
                Aid, ExpYield, ExpDIM, ExpLac = FilteredAnimals[r]
                YieldStd  = max(AllProfiles[Aid]["YieldStd"], 0.5)
                ActYield  = RecYields[c]
                YieldDiff = np.abs(ActYield - ExpYield)
                if YieldDiff > 2 * YieldStd:
                    continue
                Writer.writerow([RecIdx[c], Aid, str(Date)[:10], Sess,
                                 round(ExpYield,4), round(ActYield,4),
                                 round(YieldDiff,4), ExpDIM, ExpLac])
                TotalAssigned += 1

print(f"Done in {time.time()-Start:.1f}s")
print(f"Assigned on 50 dates : {TotalAssigned:,}")
print(f"Mean per date        : {TotalAssigned/50:.0f}")
print(f"Estimated full run   : {TotalAssigned/50*827:,.0f} records")
print(f"Estimated time       : {(time.time()-Start)/50*827:.0f}s")

Done in 2.1s
Assigned on 50 dates : 11,005
Mean per date        : 220
Estimated full run   : 182,023 records
Estimated time       : 35s


In [86]:
import csv, time
Start   = time.time()
OutPath = ProjectRoot / "DfHungarian.csv"
Dates   = sorted(AnimalsByDate.keys())

with open(OutPath, "w", newline="") as f:
    Writer = csv.writer(f)
    Writer.writerow(["OrigIdx","AnimalId","Date","Session",
                     "ExpYield","ActYield","YieldDiff","ExpDIM","ExpLac"])

    TotalAssigned = 0

    for i, Date in enumerate(Dates):
        if Date not in SessionArrays:
            continue

        for Sess in [1, 2, 3]:
            Records   = SessionArrays[Date][Sess]
            RecYields = Records["Yield"]
            RecFlows  = Records["Flow"]
            RecDurs   = Records["Dur"]
            RecIdx    = Records["Idx"]

            FilteredAnimals = []
            for Aid in AnimalsByDate.get(Date, []):
                if (Aid, Date) not in WoodsPredDictAll:
                    continue
                ExpYield, ExpDIM, ExpLac = WoodsPredDictAll[(Aid, Date)]
                YieldStd = max(AllProfiles[Aid]["YieldStd"], 0.5)
                if ExpLac < 1 or ExpLac > 8 or ExpDIM < 1:
                    continue
                if ExpYield < 3.0:
                  continue
                if not np.any(np.abs(RecYields - ExpYield) <= 1.5 * YieldStd):
                    continue
                FilteredAnimals.append((Aid, ExpYield, ExpDIM, ExpLac))

            if len(FilteredAnimals) == 0:
                continue

            CostMatrix = np.zeros((len(FilteredAnimals), len(RecYields)))
            for j, (Aid, ExpYield, ExpDIM, ExpLac) in enumerate(FilteredAnimals):
                YieldStd = max(AllProfiles[Aid]["YieldStd"], 0.5)
                FlowMean = AllProfiles[Aid]["FlowMean"]
                FlowStd  = max(AllProfiles[Aid]["FlowStd"], 0.1)
                DurMean  = AllProfiles[Aid]["DurMean"]
                DurStd   = max(AllProfiles[Aid]["DurStd"], 1.0)
                CostMatrix[j,:] = (
                    10 * np.abs(RecYields - ExpYield) / YieldStd +
                    np.abs(RecFlows - FlowMean) / FlowStd +
                    np.abs(RecDurs  - DurMean)  / DurStd
                )

            CostMatrix = np.nan_to_num(CostMatrix, nan=999.0, posinf=999.0, neginf=999.0)
            RowIdx, ColIdx = linear_sum_assignment(CostMatrix)

            for r, c in zip(RowIdx, ColIdx):
                Aid, ExpYield, ExpDIM, ExpLac = FilteredAnimals[r]
                YieldStd  = max(AllProfiles[Aid]["YieldStd"], 0.5)
                ActYield  = RecYields[c]
                YieldDiff = np.abs(ActYield - ExpYield)
                if YieldDiff > 2 * YieldStd:
                    continue
                Writer.writerow([RecIdx[c], Aid, str(Date)[:10], Sess,
                                 round(ExpYield,4), round(ActYield,4),
                                 round(YieldDiff,4), ExpDIM, ExpLac])
                TotalAssigned += 1

        if i % 100 == 0:
            print(f"  {i}/827 dates · {TotalAssigned:,} · {time.time()-Start:.0f}s")

print(f"\nDone in {time.time()-Start:.0f}s")
print(f"Total assigned : {TotalAssigned:,}")

  0/827 dates · 3 · 0s
  100/827 dates · 93,654 · 97s
  200/827 dates · 223,728 · 199s
  300/827 dates · 369,436 · 363s
  400/827 dates · 530,584 · 480s
  500/827 dates · 665,366 · 565s
  600/827 dates · 834,541 · 696s
  700/827 dates · 1,014,696 · 823s
  800/827 dates · 1,180,526 · 956s

Done in 981s
Total assigned : 1,236,614


In [87]:
DfHungarian = pd.read_csv(ProjectRoot / "DfHungarian.csv")
print(f"Total assigned     : {len(DfHungarian):,}")
print(f"Unique animals     : {DfHungarian['AnimalId'].nunique():,}")
print(f"Unique records     : {DfHungarian['OrigIdx'].nunique():,}")
print(f"Mean yield diff    : {DfHungarian['YieldDiff'].mean():.4f} kg")
print(f"Within ±0.5kg      : {(DfHungarian['YieldDiff']<0.5).sum():,} ({(DfHungarian['YieldDiff']<0.5).mean():.1%})")
print(f"Within ±1.0kg      : {(DfHungarian['YieldDiff']<1.0).sum():,} ({(DfHungarian['YieldDiff']<1.0).mean():.1%})")
print()
DupCheck = DfHungarian.groupby("OrigIdx")["AnimalId"].nunique()
print(f"Records with 1 animal  : {(DupCheck==1).sum():,}")
print(f"Records with 2 animals : {(DupCheck==2).sum():,}")
print()
PerAnimal = DfHungarian.groupby("AnimalId").size()
print(f"Mean records/animal : {PerAnimal.mean():.0f}")
print(f"Animals 200+ records: {(PerAnimal>=200).sum():,}")
print()
Aid = '-1007048835775197250'
print(f"Test animal records : {(DfHungarian['AnimalId'].astype(str)==Aid).sum()}")
print(f"Coverage (66 days×3): {(DfHungarian['AnimalId'].astype(str)==Aid).sum()/198:.1%}")

Total assigned     : 1,236,614
Unique animals     : 5,601
Unique records     : 1,236,614
Mean yield diff    : 2.1242 kg
Within ±0.5kg      : 586,311 (47.4%)
Within ±1.0kg      : 659,244 (53.3%)

Records with 1 animal  : 1,236,614
Records with 2 animals : 0

Mean records/animal : 221
Animals 200+ records: 2,484

Test animal records : 414
Coverage (66 days×3): 209.1%


In [88]:
Aid = '-1007048835775197250'
AnimalRecs = DfHungarian[DfHungarian["AnimalId"].astype(str) == Aid]
print(f"Records assigned    : {len(AnimalRecs)}")
print(f"Unique dates        : {AnimalRecs['Date'].nunique()}")
print(f"Sessions per date   : {len(AnimalRecs)/AnimalRecs['Date'].nunique():.1f}")
print(f"Mean yield diff     : {AnimalRecs['YieldDiff'].mean():.3f} kg")
print()
print("Date range:")
print(f"  Min: {AnimalRecs['Date'].min()}")
print(f"  Max: {AnimalRecs['Date'].max()}")
print()
print("Gap periods for this animal:")
display(DfGaps[DfGaps["AnimalId"].astype(str)==Aid][["GapStart","GapEnd","GapDays","PreLac","PostLac"]])

Records assigned    : 414
Unique dates        : 149
Sessions per date   : 2.8
Mean yield diff     : 2.156 kg

Date range:
  Min: 2020-01-22
  Max: 2021-07-08

Gap periods for this animal:


,GapStart,GapEnd,GapDays,PreLac,PostLac
0,2020-01-22,2020-03-27,66,1.0,2.0
1,2021-03-12,2021-07-08,119,2.0,3.0


In [89]:
# Smart dry filter — use Wood's curve yield not DIM threshold
# If Wood's predicts > 3kg → milking
# If Wood's predicts < 3kg → dry, exclude

# This is exactly Filter 3 (ExpYield > 3.0) which we already have
# The DIM 305 filter was redundant — but removing it added too many
# animals whose Wood's curve DOES predict > 3kg but whose DIM is 306+

# Check: how many animals at DIM > 305 have ExpYield > 3 on a sample date?
Date     = pd.Timestamp("2020-02-07")
Sess     = 1
Records  = SessionArrays[Date][Sess]
RecYields = Records["Yield"]

HighDIM  = []
for Aid in AnimalsByDate.get(Date, []):
    if (Aid, Date) not in WoodsPredDictAll:
        continue
    ExpYield, ExpDIM, ExpLac = WoodsPredDictAll[(Aid, Date)]
    if ExpDIM > 305 and ExpYield > 3.0:
        HighDIM.append((Aid, ExpYield, ExpDIM))

print(f"Animals with DIM>305 and ExpYield>3 on {Date.date()}: {len(HighDIM)}")
print(f"Records available                                    : {len(RecYields)}")
print(f"Sample high-DIM animals:")
for Aid, Y, D in HighDIM[:5]:
    print(f"  DIM={D} ExpYield={Y:.2f} kg")

Animals with DIM>305 and ExpYield>3 on 2020-02-07: 470
Records available                                    : 700
Sample high-DIM animals:
  DIM=306 ExpYield=11.85 kg
  DIM=369 ExpYield=3.84 kg
  DIM=402 ExpYield=6.33 kg
  DIM=339 ExpYield=8.89 kg
  DIM=388 ExpYield=6.20 kg


In [90]:
# Test with max cost threshold on one date
Date     = pd.Timestamp("2020-02-07")
Sess     = 1
Records  = SessionArrays[Date][Sess]
RecYields = Records["Yield"]
RecFlows  = Records["Flow"]
RecDurs   = Records["Dur"]
RecIdx    = Records["Idx"]

FilteredAnimals = []
for Aid in AnimalsByDate.get(Date, []):
    if (Aid, Date) not in WoodsPredDictAll:
        continue
    ExpYield, ExpDIM, ExpLac = WoodsPredDictAll[(Aid, Date)]
    YieldStd = max(AllProfiles[Aid]["YieldStd"], 0.5)
    if ExpLac < 1 or ExpLac > 8 or ExpDIM < 1 or ExpYield < 3.0:
        continue
    if not np.any(np.abs(RecYields - ExpYield) <= 1.5 * YieldStd):
        continue
    FilteredAnimals.append((Aid, ExpYield, ExpDIM, ExpLac))

print(f"Filtered animals : {len(FilteredAnimals)}")
print(f"Records          : {len(RecYields)}")

# Build cost matrix with high penalty for bad matches
CostMatrix = np.full((len(FilteredAnimals), len(RecYields)), 999.0)
for i, (Aid, ExpYield, ExpDIM, ExpLac) in enumerate(FilteredAnimals):
    YieldStd = max(AllProfiles[Aid]["YieldStd"], 0.5)
    FlowMean = AllProfiles[Aid]["FlowMean"]
    FlowStd  = max(AllProfiles[Aid]["FlowStd"], 0.1)
    DurMean  = AllProfiles[Aid]["DurMean"]
    DurStd   = max(AllProfiles[Aid]["DurStd"], 1.0)

    YieldDiffs = np.abs(RecYields - ExpYield)
    Mask       = YieldDiffs <= 1.5 * YieldStd  # only fill plausible pairs

    CostMatrix[i, Mask] = (
        10 * YieldDiffs[Mask] / YieldStd +
        np.abs(RecFlows[Mask] - FlowMean) / FlowStd +
        np.abs(RecDurs[Mask]  - DurMean)  / DurStd
    )

CostMatrix = np.nan_to_num(CostMatrix, nan=999.0)
RowIdx, ColIdx = linear_sum_assignment(CostMatrix)

Results = [(r,c) for r,c in zip(RowIdx,ColIdx) if CostMatrix[r,c] < 999.0]
YieldDiffs = [np.abs(RecYields[c] - FilteredAnimals[r][1]) for r,c in Results]

print(f"Assigned (cost < 999) : {len(Results)}")
print(f"Unassigned (no match) : {len(FilteredAnimals)-len(Results)}")
print(f"Mean yield diff       : {np.mean(YieldDiffs):.4f} kg")
print(f"Within ±0.5kg         : {sum(1 for d in YieldDiffs if d<0.5)} / {len(Results)}")
print(f"Within ±1.0kg         : {sum(1 for d in YieldDiffs if d<1.0)} / {len(Results)}")

Filtered animals : 695
Records          : 700
Assigned (cost < 999) : 625
Unassigned (no match) : 70
Mean yield diff       : 3.9419 kg
Within ±0.5kg         : 96 / 625
Within ±1.0kg         : 114 / 625


In [91]:
Date      = pd.Timestamp("2020-02-07")
Sess      = 1
Records   = SessionArrays[Date][Sess]
RecYields = Records["Yield"]

# Build yield range per animal
AnimalRanges = []
for Aid in AnimalsByDate.get(Date, []):
    if (Aid, Date) not in WoodsPredDictAll:
        continue
    ExpYield, ExpDIM, ExpLac = WoodsPredDictAll[(Aid, Date)]
    YieldStd = max(AllProfiles[Aid]["YieldStd"], 0.5)
    if ExpLac < 1 or ExpLac > 8 or ExpDIM < 1 or ExpYield < 3.0:
        continue
    Lo = ExpYield - 1.5 * YieldStd
    Hi = ExpYield + 1.5 * YieldStd
    AnimalRanges.append((Aid, ExpYield, Lo, Hi))

# For each animal count competitors whose range overlaps
NoCompetitors = 0
FewCompetitors = 0
for Aid, ExpY, Lo, Hi in AnimalRanges:
    Competitors = sum(
        1 for _, EY, L, H in AnimalRanges
        if L <= ExpY <= H or Lo <= EY <= Hi
    )
    if Competitors == 1:
        NoCompetitors += 1
    elif Competitors <= 3:
        FewCompetitors += 1

print(f"Animals on this date          : {len(AnimalRanges)}")
print(f"No yield range competitors    : {NoCompetitors}")
print(f"1-3 competitors               : {FewCompetitors}")
print(f"Many competitors              : {len(AnimalRanges)-NoCompetitors-FewCompetitors}")

Animals on this date          : 697
No yield range competitors    : 3
1-3 competitors               : 0
Many competitors              : 694


In [92]:
# Test iterative enrichment on 10 animals
HungTest = pd.read_csv(ProjectRoot / "DfHungarian.csv")
HungTest  = HungTest[HungTest["YieldDiff"] < 0.5]  # HIGH confidence only

print(f"HIGH confidence records : {len(HungTest):,}")
print(f"Animals                 : {HungTest['AnimalId'].nunique():,}")

# For one animal — compare Wood's curve before and after adding recovered records
Aid     = '-1007048835775197250'
GapRow  = DfGaps[DfGaps["AnimalId"].astype(str)==Aid].sort_values("GapStart").iloc[0]
PreLac  = int(GapRow["PreLac"])

# Original Wood's curve
A0, B0, C0 = AllWoodsParams[Aid][PreLac]

# Add HIGH confidence recovered records
RecoveredRecs = HungTest[HungTest["AnimalId"].astype(str)==Aid].copy()
RecoveredRecs["DaysInMilk"] = RecoveredRecs["ExpDIM"]
RecoveredRecs["TotalMilkYieldSessionKg"] = RecoveredRecs["ActYield"]
RecoveredRecs["LactationNumber"] = RecoveredRecs["ExpLac"]

# Combine with known records
KnownRecs = DfKnown[
    (DfKnown["AnimalId"]==Aid) &
    (DfKnown["LactationNumber"]==PreLac)
][["DaysInMilk","TotalMilkYieldSessionKg"]].copy()

EnrichedRecs = pd.concat([
    KnownRecs,
    RecoveredRecs[["DaysInMilk","TotalMilkYieldSessionKg"]]
], ignore_index=True).sort_values("DaysInMilk")

print(f"\nAnimal {Aid[:15]}...")
print(f"Known records (Lac {PreLac})     : {len(KnownRecs)}")
print(f"Recovered HIGH conf records : {len(RecoveredRecs)}")
print(f"Enriched total              : {len(EnrichedRecs)}")

# Refit Wood's curve
from scipy.optimize import curve_fit
try:
    T = EnrichedRecs["DaysInMilk"].values.astype(float)
    Y = EnrichedRecs["TotalMilkYieldSessionKg"].values.astype(float)
    Params, _ = curve_fit(WoodsCurve, T, Y,
                          p0=[10,0.1,0.003], maxfev=5000,
                          bounds=([0,0,0],[100,2,0.1]))
    A1, B1, C1 = Params

    # Compare at gap start DIM
    TestDIM = int(GapRow["PreDIM"]) + 10
    print(f"\nAt DIM {TestDIM}:")
    print(f"  Original Wood's : {WoodsCurve(TestDIM,A0,B0,C0):.3f} kg")
    print(f"  Enriched Wood's : {WoodsCurve(TestDIM,A1,B1,C1):.3f} kg")
    print(f"  Known PreYield  : {GapRow['PreYield']:.3f} kg")
except Exception as e:
    print(f"Fit failed: {e}")

HIGH confidence records : 586,311
Animals                 : 4,628

Animal -10070488357751...
Known records (Lac 1)     : 396
Recovered HIGH conf records : 109
Enriched total              : 505

At DIM 299:
  Original Wood's : 11.998 kg
  Enriched Wood's : 11.912 kg
  Known PreYield  : 12.383 kg


In [93]:
# How many animals get 50+ HIGH confidence records added?
RecordsAdded = HungTest.groupby("AnimalId").size()
print(f"Animals with 1-50 HIGH conf records added   : {(RecordsAdded<50).sum():,}")
print(f"Animals with 50-100 records added           : {((RecordsAdded>=50)&(RecordsAdded<100)).sum():,}")
print(f"Animals with 100-200 records added          : {((RecordsAdded>=100)&(RecordsAdded<200)).sum():,}")
print(f"Animals with 200+ records added             : {(RecordsAdded>=200).sum():,}")
print()
print(f"Mean records added per animal : {RecordsAdded.mean():.0f}")
print(f"Median                        : {RecordsAdded.median():.0f}")

Animals with 1-50 HIGH conf records added   : 1,762
Animals with 50-100 records added           : 562
Animals with 100-200 records added          : 1,487
Animals with 200+ records added             : 817

Mean records added per animal : 127
Median                        : 98


In [95]:
import time, pickle
Start = time.time()

# Pre-group everything outside the loop
print("Pre-grouping...")
HungByAnimalLac = HungTest.groupby(["AnimalId","ExpLac"])
KnownByAnimalLac = DfKnown.groupby(["AnimalId","LactationNumber"])

print(f"Done in {time.time()-Start:.0f}s")
print("Refitting...")

EnrichedWoodsParams = {}
Count = 0

for Aid in HungTest["AnimalId"].astype(str).unique():
    if Aid not in AllWoodsParams:
        continue

    LacParams = dict(AllWoodsParams[Aid])  # start with existing params

    for Lac in HungTest[HungTest["AnimalId"].astype(str)==Aid]["ExpLac"].unique():
        Lac = int(Lac)

        try:
            RecLac = HungByAnimalLac.get_group((Aid, Lac))[
                ["ExpDIM","ActYield"]].values
        except:
            continue

        try:
            KnownLac = KnownByAnimalLac.get_group((Aid, Lac))[
                ["DaysInMilk","TotalMilkYieldSessionKg"]].values
        except:
            KnownLac = np.empty((0,2))

        Combined = np.vstack([KnownLac, RecLac]) if len(KnownLac) > 0 else RecLac
        Combined = Combined[Combined[:,0] > 0]

        if len(Combined) < 10:
            continue

        T = Combined[:,0].astype(float)
        Y = Combined[:,1].astype(float)
        try:
            Params, _ = curve_fit(WoodsCurve, T, Y,
                                  p0=[10,0.1,0.003], maxfev=3000,
                                  bounds=([0,0,0],[100,2,0.1]))
            LacParams[Lac] = Params
        except:
            pass

    EnrichedWoodsParams[Aid] = LacParams
    Count += 1
    if Count % 500 == 0:
        print(f"  {Count}/{HungTest['AnimalId'].nunique()} · {time.time()-Start:.0f}s")

print(f"Done : {len(EnrichedWoodsParams):,} animals in {time.time()-Start:.0f}s")
with open(ProjectRoot / "EnrichedWoodsParams.pkl", "wb") as f:
    pickle.dump(EnrichedWoodsParams, f)
print("Saved")

Pre-grouping...
Done in 0s
Refitting...
  500/4628 · 229s
  1000/4628 · 381s
  1500/4628 · 534s
  2000/4628 · 685s
  2500/4628 · 835s
  3000/4628 · 986s
  3500/4628 · 1137s
  4000/4628 · 1289s
  4500/4628 · 1439s
Done : 4,628 animals in 1478s
Saved


In [96]:
# Rebuild WoodsPredDictAll with enriched params
import time
Start = time.time()

# Merge enriched with original — enriched takes priority
MergedWoodsParams = {}
for Aid in AllWoodsParams:
    if Aid in EnrichedWoodsParams:
        MergedWoodsParams[Aid] = EnrichedWoodsParams[Aid]
    else:
        MergedWoodsParams[Aid] = AllWoodsParams[Aid]

print(f"Merged params : {len(MergedWoodsParams):,} animals")

# Rebuild predictions
WoodsPredDictEnriched = {}
AllGaps = DfGaps[DfGaps["GapDays"] > 30].copy()
AllGaps["AnimalId"] = AllGaps["AnimalId"].astype(str)

for _, Row in AllGaps.iterrows():
    Aid      = str(Row["AnimalId"])
    if Aid not in MergedWoodsParams:
        continue
    GapDates = pd.date_range(Row["GapStart"], Row["GapEnd"], freq="D")
    PreLac   = int(Row["PreLac"])
    PreDIM   = int(Row["PreDIM"])
    PreDate  = Row["PreDate"]
    CalvDate = Row["CalvingDate"]
    PostLac  = int(Row["PostLac"])

    for Date in GapDates:
        if pd.isna(CalvDate) or Date < pd.Timestamp(CalvDate):
            ExpLac = PreLac
            ExpDIM = PreDIM + (Date - pd.Timestamp(PreDate)).days
        else:
            ExpLac = PostLac
            ExpDIM = (Date - pd.Timestamp(CalvDate)).days + 1

        if ExpLac in MergedWoodsParams[Aid]:
            A, B, C  = MergedWoodsParams[Aid][ExpLac]
            ExpYield = WoodsCurve(max(ExpDIM, 1), A, B, C)
        else:
            ExpYield = AllProfiles[Aid]["YieldMean"]

        WoodsPredDictEnriched[(Aid, Date)] = (ExpYield, ExpDIM, ExpLac)

print(f"Enriched predictions : {len(WoodsPredDictEnriched):,} in {time.time()-Start:.0f}s")

Merged params : 5,610 animals
Enriched predictions : 597,822 in 29s


In [97]:
import csv, time
from scipy.optimize import linear_sum_assignment
Start   = time.time()
OutPath = ProjectRoot / "DfHungarianEnriched.csv"
Dates   = sorted(AnimalsByDate.keys())

with open(OutPath, "w", newline="") as f:
    Writer = csv.writer(f)
    Writer.writerow(["OrigIdx","AnimalId","Date","Session",
                     "ExpYield","ActYield","YieldDiff","ExpDIM","ExpLac"])

    TotalAssigned = 0

    for i, Date in enumerate(Dates):
        if Date not in SessionArrays:
            continue

        for Sess in [1, 2, 3]:
            Records   = SessionArrays[Date][Sess]
            RecYields = Records["Yield"]
            RecFlows  = Records["Flow"]
            RecDurs   = Records["Dur"]
            RecIdx    = Records["Idx"]

            FilteredAnimals = []
            for Aid in AnimalsByDate.get(Date, []):
                if (Aid, Date) not in WoodsPredDictEnriched:
                    continue
                ExpYield, ExpDIM, ExpLac = WoodsPredDictEnriched[(Aid, Date)]
                YieldStd = max(AllProfiles[Aid]["YieldStd"], 0.5)
                if ExpLac < 1 or ExpLac > 8 or ExpDIM < 1 or ExpYield < 3.0:
                    continue
                if not np.any(np.abs(RecYields - ExpYield) <= 1.5 * YieldStd):
                    continue
                FilteredAnimals.append((Aid, ExpYield, ExpDIM, ExpLac))

            if len(FilteredAnimals) == 0:
                continue

            CostMatrix = np.zeros((len(FilteredAnimals), len(RecYields)))
            for j, (Aid, ExpYield, ExpDIM, ExpLac) in enumerate(FilteredAnimals):
                YieldStd = max(AllProfiles[Aid]["YieldStd"], 0.5)
                FlowMean = AllProfiles[Aid]["FlowMean"]
                FlowStd  = max(AllProfiles[Aid]["FlowStd"], 0.1)
                DurMean  = AllProfiles[Aid]["DurMean"]
                DurStd   = max(AllProfiles[Aid]["DurStd"], 1.0)
                CostMatrix[j,:] = (
                    10 * np.abs(RecYields - ExpYield) / YieldStd +
                    np.abs(RecFlows - FlowMean) / FlowStd +
                    np.abs(RecDurs  - DurMean)  / DurStd
                )

            CostMatrix = np.nan_to_num(CostMatrix, nan=999.0)
            RowIdx, ColIdx = linear_sum_assignment(CostMatrix)

            for r, c in zip(RowIdx, ColIdx):
                Aid, ExpYield, ExpDIM, ExpLac = FilteredAnimals[r]
                YieldStd  = max(AllProfiles[Aid]["YieldStd"], 0.5)
                ActYield  = RecYields[c]
                YieldDiff = np.abs(ActYield - ExpYield)
                if YieldDiff > 2 * YieldStd:
                    continue
                Writer.writerow([RecIdx[c], Aid, str(Date)[:10], Sess,
                                 round(ExpYield,4), round(ActYield,4),
                                 round(YieldDiff,4), ExpDIM, ExpLac])
                TotalAssigned += 1

        if i % 100 == 0:
            print(f"  {i}/827 · {TotalAssigned:,} · {time.time()-Start:.0f}s")

print(f"Done in {time.time()-Start:.0f}s")
print(f"Total : {TotalAssigned:,}")

  0/827 · 3 · 0s
  100/827 · 93,654 · 130s
  200/827 · 223,728 · 245s
  300/827 · 369,436 · 351s
  400/827 · 530,584 · 469s
  500/827 · 665,366 · 555s
  600/827 · 834,541 · 686s
  700/827 · 1,014,696 · 815s
  800/827 · 1,180,526 · 953s
Done in 977s
Total : 1,236,614


In [98]:
# Take Hungarian HIGH confidence (within ±0.5kg) as Tier 1
# Take cluster matching for remaining as Tier 2

HungHigh = DfHungarian[DfHungarian["YieldDiff"] < 0.5]
print(f"Tier 1 (Hungarian HIGH) : {len(HungHigh):,}")
print(f"Tier 2 (remaining)      : to be filled by cluster matching")

Tier 1 (Hungarian HIGH) : 586,311
Tier 2 (remaining)      : to be filled by cluster matching


In [99]:
# Final combined export
DfHungarian = pd.read_csv(ProjectRoot / "DfHungarian.csv")

# Tier 1 — Hungarian HIGH confidence (within ±0.5 kg)
Tier1 = DfHungarian[DfHungarian["YieldDiff"] < 0.5].copy()
Tier1["Confidence"] = "HIGH"

# Tier 2 — Hungarian MEDIUM (0.5–2std)
Tier2 = DfHungarian[DfHungarian["YieldDiff"] >= 0.5].copy()
Tier2["Confidence"] = "MEDIUM"

DfFinalAll = pd.concat([Tier1, Tier2], ignore_index=True)

print(f"Tier 1 HIGH   : {len(Tier1):,} records, {Tier1['AnimalId'].nunique():,} animals")
print(f"Tier 2 MEDIUM : {len(Tier2):,} records, {Tier2['AnimalId'].nunique():,} animals")
print(f"Total         : {len(DfFinalAll):,} records")
print(f"Unique animals: {DfFinalAll['AnimalId'].nunique():,}")
print()

# Apply to DfMissing
DfMissing["AnimalIdRecovered"] = pd.NA
DfMissing["Confidence"]        = pd.NA
DfMissing["ExpDIM"]            = pd.NA
DfMissing["ExpLac"]            = pd.NA

AssignMap = DfFinalAll.set_index("OrigIdx")[
    ["AnimalId","Confidence","ExpDIM","ExpLac"]
].to_dict("index")

for Idx, Row in AssignMap.items():
    DfMissing.loc[Idx, "AnimalIdRecovered"] = Row["AnimalId"]
    DfMissing.loc[Idx, "Confidence"]        = Row["Confidence"]
    DfMissing.loc[Idx, "ExpDIM"]            = Row["ExpDIM"]
    DfMissing.loc[Idx, "ExpLac"]            = Row["ExpLac"]

Recovered = DfMissing["AnimalIdRecovered"].notna().sum()
print(f"Records recovered : {Recovered:,} / {len(DfMissing):,} ({Recovered/len(DfMissing)*100:.1f}%)")
print(f"HIGH confidence   : {(DfMissing['Confidence']=='HIGH').sum():,}")
print(f"MEDIUM confidence : {(DfMissing['Confidence']=='MEDIUM').sum():,}")
print(f"Unrecovered       : {DfMissing['AnimalIdRecovered'].isna().sum():,}")

# Export
DfFinalOutput = pd.concat([
    DfKnown.assign(RecoveryMethod="Original", Confidence="CONFIRMED"),
    DfMissing.assign(RecoveryMethod=np.where(
        DfMissing["AnimalIdRecovered"].notna(),
        "Recovered", "Unrecovered"))
], ignore_index=True)

DfFinalOutput.to_csv(ProjectRoot / "AnimalIDRecovered_Final.csv", index=False)
print(f"Saved")

Tier 1 HIGH   : 586,311 records, 4,628 animals
Tier 2 MEDIUM : 650,303 records, 5,231 animals
Total         : 1,236,614 records
Unique animals: 5,601

Records recovered : 1,236,614 / 1,701,003 (72.7%)
HIGH confidence   : 586,311
MEDIUM confidence : 650,303
Unrecovered       : 464,389
Saved


In [100]:
# Use the cleaner DIM<305 Hungarian result
# DfHungarian already has the DIM<305 version (521,414 records)

Tier1 = DfHungarian[DfHungarian["YieldDiff"] < 0.5].copy()
Tier1["Confidence"] = "HIGH"

Tier2 = DfHungarian[DfHungarian["YieldDiff"] >= 0.5].copy()
Tier2["Confidence"] = "MEDIUM"

DfFinalAll = pd.concat([Tier1, Tier2], ignore_index=True)

print(f"Tier 1 HIGH   : {len(Tier1):,} ({Tier1['YieldDiff'].mean():.3f} kg mean diff)")
print(f"Tier 2 MEDIUM : {len(Tier2):,} ({Tier2['YieldDiff'].mean():.3f} kg mean diff)")
print(f"Total         : {len(DfFinalAll):,}")
print(f"Unique animals: {DfFinalAll['AnimalId'].nunique():,}")
print()

# Apply to DfMissing
DfMissing["AnimalIdRecovered"] = pd.NA
DfMissing["Confidence"]        = pd.NA
DfMissing["ExpDIM"]            = pd.NA
DfMissing["ExpLac"]            = pd.NA

AssignMap = DfFinalAll.set_index("OrigIdx")[
    ["AnimalId","Confidence","ExpDIM","ExpLac"]
].to_dict("index")

for Idx, Row in AssignMap.items():
    DfMissing.loc[Idx, "AnimalIdRecovered"] = Row["AnimalId"]
    DfMissing.loc[Idx, "Confidence"]        = Row["Confidence"]
    DfMissing.loc[Idx, "ExpDIM"]            = Row["ExpDIM"]
    DfMissing.loc[Idx, "ExpLac"]            = Row["ExpLac"]

Recovered = DfMissing["AnimalIdRecovered"].notna().sum()
print(f"Records recovered : {Recovered:,} / {len(DfMissing):,} ({Recovered/len(DfMissing)*100:.1f}%)")
print(f"HIGH confidence   : {(DfMissing['Confidence']=='HIGH').sum():,}")
print(f"MEDIUM confidence : {(DfMissing['Confidence']=='MEDIUM').sum():,}")
print(f"Unrecovered       : {DfMissing['AnimalIdRecovered'].isna().sum():,}")

DfFinalOutput = pd.concat([
    DfKnown.assign(RecoveryMethod="Original", Confidence="CONFIRMED"),
    DfMissing.assign(RecoveryMethod=np.where(
        DfMissing["AnimalIdRecovered"].notna(),
        "Recovered", "Unrecovered"))
], ignore_index=True)

DfFinalOutput.to_csv(ProjectRoot / "AnimalIDRecovered_Final.csv", index=False)
print("Saved")

Tier 1 HIGH   : 586,311 (0.124 kg mean diff)
Tier 2 MEDIUM : 650,303 (3.928 kg mean diff)
Total         : 1,236,614
Unique animals: 5,601

Records recovered : 1,236,614 / 1,701,003 (72.7%)
HIGH confidence   : 586,311
MEDIUM confidence : 650,303
Unrecovered       : 464,389
Saved


In [101]:
# Keep only HIGH confidence (within 0.5 kg)
print("=== Final Recovery Summary ===")
print()
print(f"Total missing records    : 1,701,003")
print(f"HIGH confidence (Tier 1) : {(DfMissing['Confidence']=='HIGH').sum():,} ({(DfMissing['Confidence']=='HIGH').sum()/1701003*100:.1f}%)")
print(f"MEDIUM confidence        : {(DfMissing['Confidence']=='MEDIUM').sum():,} — excluded (3.9 kg mean diff)")
print(f"Unrecovered              : {DfMissing['AnimalIdRecovered'].isna().sum():,}")
print()
print(f"Animals identified       : {DfMissing[DfMissing['Confidence']=='HIGH']['AnimalIdRecovered'].nunique():,}")
print()
print("Confidence breakdown:")
print(f"  HIGH  : yield diff < 0.5 kg — within natural daily variation")
print(f"  MEDIUM: yield diff 0.5-2 std — biologically plausible but uncertain")
print(f"  Unrecovered: no match found within biological tolerance")

=== Final Recovery Summary ===

Total missing records    : 1,701,003
HIGH confidence (Tier 1) : 586,311 (34.5%)
MEDIUM confidence        : 650,303 — excluded (3.9 kg mean diff)
Unrecovered              : 464,389

Animals identified       : 4,628

Confidence breakdown:
  HIGH  : yield diff < 0.5 kg — within natural daily variation
  MEDIUM: yield diff 0.5-2 std — biologically plausible but uncertain
  Unrecovered: no match found within biological tolerance


In [102]:
# Final clean export — HIGH confidence only
DfMissing["AnimalIdRecovered"] = np.where(
    DfMissing["Confidence"] == "HIGH",
    DfMissing["AnimalIdRecovered"],
    pd.NA
)

DfFinalOutput = pd.concat([
    DfKnown.assign(RecoveryMethod="Original"),
    DfMissing.assign(RecoveryMethod=np.where(
        DfMissing["AnimalIdRecovered"].notna(),
        "Recovered", "Unrecovered"))
], ignore_index=True)

print(f"Original   : {(DfFinalOutput['RecoveryMethod']=='Original').sum():,}")
print(f"Recovered  : {(DfFinalOutput['RecoveryMethod']=='Recovered').sum():,}")
print(f"Unrecovered: {(DfFinalOutput['RecoveryMethod']=='Unrecovered').sum():,}")
print(f"Total      : {len(DfFinalOutput):,}")

DfFinalOutput.to_csv(ProjectRoot / "AnimalIDRecovered_Final.csv", index=False)
print("Saved")

Original   : 6,737,364
Recovered  : 586,311
Unrecovered: 1,114,692
Total      : 8,438,367
Saved


In [83]:
DfHungarian = pd.read_csv(ProjectRoot / "DfHungarian.csv")
print(f"Total assigned     : {len(DfHungarian):,}")
print(f"Unique animals     : {DfHungarian['AnimalId'].nunique():,}")
print(f"Unique records     : {DfHungarian['OrigIdx'].nunique():,}")
print(f"Mean yield diff    : {DfHungarian['YieldDiff'].mean():.4f} kg")
print(f"Within ±0.5kg      : {(DfHungarian['YieldDiff']<0.5).sum():,} ({(DfHungarian['YieldDiff']<0.5).mean():.1%})")
print(f"Within ±1.0kg      : {(DfHungarian['YieldDiff']<1.0).sum():,} ({(DfHungarian['YieldDiff']<1.0).mean():.1%})")
print()

# Conflict check — should be minimal with Hungarian
DupCheck = DfHungarian.groupby("OrigIdx")["AnimalId"].nunique()
print(f"Records with 1 animal  : {(DupCheck==1).sum():,}")
print(f"Records with 2 animals : {(DupCheck==2).sum():,}")
print(f"Records with 3+animals : {(DupCheck>=3).sum():,}")
print()

# Coverage per animal
PerAnimal = DfHungarian.groupby("AnimalId").size()
print(f"Mean records/animal : {PerAnimal.mean():.0f}")
print(f"Animals 200+ records: {(PerAnimal>=200).sum():,}")
print(f"Animals 0 records   : {DfHungarian['AnimalId'].nunique() - len(PerAnimal):,}")
print()

# Test animal
Aid = '-1007048835775197250'
print(f"Test animal records : {(DfHungarian['AnimalId'].astype(str)==Aid).sum()}")

Total assigned     : 521,414
Unique animals     : 5,579
Unique records     : 521,414
Mean yield diff    : 0.4286 kg
Within ±0.5kg      : 438,697 (84.1%)
Within ±1.0kg      : 459,855 (88.2%)

Records with 1 animal  : 521,414
Records with 2 animals : 0
Records with 3+animals : 0

Mean records/animal : 93
Animals 200+ records: 521
Animals 0 records   : 0

Test animal records : 57


In [84]:
Aid      = '-1007048835775197250'
GapRow   = DfGaps[DfGaps["AnimalId"].astype(str)==Aid].iloc[0]
GapDates = pd.date_range(GapRow["GapStart"], GapRow["GapEnd"], freq="D")

InHungarian = set(DfHungarian[DfHungarian["AnimalId"].astype(str)==Aid]["Date"].values)
Missing     = [str(D)[:10] for D in GapDates if str(D)[:10] not in InHungarian]

print(f"Gap days total     : {len(GapDates)}")
print(f"In Hungarian       : {len(InHungarian)}")
print(f"Missing from output: {len(Missing)}")
print()
print(f"First 5 missing dates:")
for D in Missing[:5]:
    Date = pd.Timestamp(D)
    InABD = Date in AnimalsByDate and Aid in AnimalsByDate[Date]
    InWPD = (Aid, Date) in WoodsPredDictAll
    print(f"  {D} InAnimalsByDate:{InABD} InWoodsPred:{InWPD}")

Gap days total     : 66
In Hungarian       : 19
Missing from output: 48

First 5 missing dates:
  2020-02-07 InAnimalsByDate:True InWoodsPred:True
  2020-02-08 InAnimalsByDate:True InWoodsPred:True
  2020-02-09 InAnimalsByDate:True InWoodsPred:True
  2020-02-10 InAnimalsByDate:True InWoodsPred:True
  2020-02-11 InAnimalsByDate:True InWoodsPred:True


In [85]:
Aid      = '-1007048835775197250'
Date     = pd.Timestamp("2020-02-07")
Sess     = 1
Records  = SessionArrays[Date][Sess]
RecYields = Records["Yield"]

ExpYield, ExpDIM, ExpLac = WoodsPredDictAll[(Aid, Date)]
YieldStd = max(AllProfiles[Aid]["YieldStd"], 0.5)

print(f"ExpYield : {ExpYield:.3f} kg")
print(f"ExpDIM   : {ExpDIM}")
print(f"ExpLac   : {ExpLac}")
print(f"YieldStd : {YieldStd:.3f} kg")
print()
print(f"Filter 1 — ExpLac 1-8   : {1 <= ExpLac <= 8}")
print(f"Filter 2 — ExpDIM > 0   : {ExpDIM > 0}")
print(f"Filter 3 — ExpYield > 3 : {ExpYield > 3.0}")
print(f"Filter 4 — ExpDIM < 305 : {ExpDIM < 305}")
print(f"Filter 5 — any record within ±1.5std: {np.any(np.abs(RecYields - ExpYield) <= 1.5 * YieldStd)}")

ExpYield : 11.854 kg
ExpDIM   : 306
ExpLac   : 1
YieldStd : 2.908 kg

Filter 1 — ExpLac 1-8   : True
Filter 2 — ExpDIM > 0   : True
Filter 3 — ExpYield > 3 : True
Filter 4 — ExpDIM < 305 : False
Filter 5 — any record within ±1.5std: True
